# AutoRaceAI Ver17.1 race_key修正版

**実行するセルは①と②の2つだけです。**

修正内容：

- `_race_key()` を `開催日・開催場・レース番号` の3引数で呼び出すよう修正
- 結果保存前に `race_date / venue / race_no` を必ず正規化
- 複数行の着順ブロック解析を維持
- 着順結果を解析後、そのままDBへ保存


In [ ]:
# ① 初期化：このセルを実行
!pip -q install openpyxl ipywidgets numpy pandas

import contextlib
import io as _quiet_io
from pathlib import Path
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

DB_DIR = Path("/content/drive/MyDrive/AutoRaceAI")
DB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DB_DIR / "autorace_players.sqlite3"

def _hidden_display(*args, **kwargs):
    return None

def _hidden_clear_output(*args, **kwargs):
    return None

_ENGINE_SOURCE = '\nimport io\nimport math\nimport re\nfrom collections import Counter\nfrom datetime import datetime, date\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport ipywidgets as widgets\nfrom IPython.display import display, clear_output\nfrom google.colab import files\nfrom openpyxl import load_workbook\nfrom openpyxl.styles import Font, PatternFill, Alignment\nfrom openpyxl.cell.cell import MergedCell\nfrom openpyxl.utils import get_column_letter\n\nprint("準備完了")\n\n\nfrom google.colab import drive\nfrom pathlib import Path\n\nif not Path("/content/drive/MyDrive").exists():\n    drive.mount("/content/drive")\n\nDB_DIR = Path("/content/drive/MyDrive/AutoRaceAI")\nDB_DIR.mkdir(parents=True, exist_ok=True)\nDB_PATH = DB_DIR / "autorace_players.sqlite3"\n\n# ============================================================\n# Ver13.0 選手マスタDB管理（Google Drive + SQLite）\n# ============================================================\nimport io, re, sqlite3, hashlib\nfrom datetime import datetime\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport ipywidgets as widgets\nfrom IPython.display import display, clear_output\nfrom openpyxl import load_workbook\nfrom openpyxl.styles import Font, PatternFill, Alignment\nfrom google.colab import drive, files\n\nDB_DIR = Path(\'/content/drive/MyDrive/AutoRaceAI\')\nDB_PATH = DB_DIR / \'autorace_players.sqlite3\'\nHISTORY_COLS_DB = [\'開催日\',\'開催場\',\'レース\',\'着順\',\'出走\',\'走路\',\'ハンデ\',\'試走T\',\'競走T\',\'ST\']\n\ndef mount_and_init_db():\n    if not Path(\'/content/drive/MyDrive\').exists(): drive.mount(\'/content/drive\')\n    DB_DIR.mkdir(parents=True, exist_ok=True)\n    with sqlite3.connect(DB_PATH) as con:\n        con.executescript("""\n        CREATE TABLE IF NOT EXISTS players (\n            player_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            player_name TEXT NOT NULL UNIQUE,\n            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n            updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP\n        );\n        CREATE TABLE IF NOT EXISTS race_history (\n            history_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            player_id INTEGER NOT NULL,\n            race_date TEXT, venue TEXT, race_no TEXT, finish REAL, starters REAL,\n            surface TEXT, handicap TEXT, trial_time REAL, race_time REAL, start_time REAL,\n            result_status TEXT NOT NULL DEFAULT \'通常\',\n            use_for_model INTEGER NOT NULL DEFAULT 1,\n            source TEXT, record_key TEXT NOT NULL UNIQUE,\n            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n            FOREIGN KEY(player_id) REFERENCES players(player_id)\n        );\n        CREATE INDEX IF NOT EXISTS idx_history_player_date ON race_history(player_id, race_date DESC);\n        """)\n    return DB_PATH\n\ndef _s(v):\n    if v is None or (isinstance(v,float) and np.isnan(v)): return \'\'\n    return str(v).strip()\n\ndef _num(v):\n    if v in (None,\'\'): return None\n    if isinstance(v,(int,float)) and not isinstance(v,bool): return float(v) if np.isfinite(v) else None\n    m=re.search(r\'-?\\d+(?:\\.\\d+)?\',str(v).replace(\',\',\'\'))\n    return float(m.group()) if m else None\n\ndef _date_text(v):\n    if v in (None,\'\'): return \'\'\n    if isinstance(v,datetime): return v.strftime(\'%Y-%m-%d\')\n    if hasattr(v,\'strftime\'):\n        try: return v.strftime(\'%Y-%m-%d\')\n        except Exception: pass\n    d=pd.to_datetime(v,errors=\'coerce\')\n    return d.strftime(\'%Y-%m-%d\') if not pd.isna(d) else _s(v)\n\ndef _find_header(ws):\n    aliases={\'開催日\':[\'開催日\'],\'開催場\':[\'開催場\'],\'レース\':[\'レース\'],\'着順\':[\'着順\'],\'出走\':[\'出走\'],\n             \'走路\':[\'走路\'],\'ハンデ\':[\'ハンデ\'],\'試走T\':[\'試走T\',\'試走\'],\'競走T\':[\'競走T\',\'競走\'],\'ST\':[\'ST\',\'ＳＴ\']}\n    for r in range(1,min(ws.max_row,40)+1):\n        vals={_s(ws.cell(r,c).value):c for c in range(1,ws.max_column+1)}; cmap={}\n        for key,names in aliases.items():\n            for name in names:\n                if name in vals: cmap[key]=vals[name]; break\n        if all(k in cmap for k in [\'開催日\',\'開催場\',\'着順\',\'試走T\',\'競走T\',\'ST\']):\n            for k in HISTORY_COLS_DB: cmap.setdefault(k,HISTORY_COLS_DB.index(k)+1)\n            return r,cmap\n    raise ValueError(f\'{ws.title}: 履歴見出しが見つかりません\')\n\ndef _player_name(ws):\n    return _s(ws[\'B2\'].value) or ws.title\n\ndef _record_key(player,row):\n    raw=\'|\'.join([player,row[\'開催日\'],row[\'開催場\'],row[\'レース\'],_s(row[\'着順\']),_s(row[\'ハンデ\']),\n                  _s(row[\'試走T\']),_s(row[\'競走T\']),_s(row[\'ST\']),row.get(\'結果区分\',\'通常\')])\n    return hashlib.sha256(raw.encode()).hexdigest()\n\ndef _ensure_player(con,name):\n    con.execute(\'INSERT OR IGNORE INTO players(player_name) VALUES (?)\',(name,))\n    con.execute(\'UPDATE players SET updated_at=CURRENT_TIMESTAMP WHERE player_name=?\',(name,))\n    return con.execute(\'SELECT player_id FROM players WHERE player_name=?\',(name,)).fetchone()[0]\n\ndef add_history_rows(player,rows,source=\'Excel取込\'):\n    added=skipped=0\n    with sqlite3.connect(DB_PATH) as con:\n        pid=_ensure_player(con,player)\n        for row in rows:\n            status=_s(row.get(\'結果区分\',\'通常\')) or \'通常\'\n            use=0 if status in [\'反妨\',\'落車\',\'故障\',\'他落\',\'反則\',\'不成立\'] else 1\n            try:\n                con.execute("""INSERT INTO race_history(\n                    player_id,race_date,venue,race_no,finish,starters,surface,handicap,\n                    trial_time,race_time,start_time,result_status,use_for_model,source,record_key)\n                    VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",\n                    (pid,row[\'開催日\'],row[\'開催場\'],row[\'レース\'],row[\'着順\'],row[\'出走\'],row[\'走路\'],row[\'ハンデ\'],\n                     row[\'試走T\'],row[\'競走T\'],row[\'ST\'],status,use,source,_record_key(player,row)))\n                added+=1\n            except sqlite3.IntegrityError: skipped+=1\n    return added,skipped\n\ndef _sheet_rows(ws):\n    hr,cmap=_find_header(ws); rows=[]\n    for r in range(hr+1,ws.max_row+1):\n        vals={k:ws.cell(r,cmap[k]).value for k in HISTORY_COLS_DB}\n        if all(v in (None,\'\') for v in vals.values()): continue\n        rows.append({\'開催日\':_date_text(vals[\'開催日\']),\'開催場\':_s(vals[\'開催場\']),\'レース\':_s(vals[\'レース\']),\n                     \'着順\':_num(vals[\'着順\']),\'出走\':_num(vals[\'出走\']),\'走路\':_s(vals[\'走路\']),\'ハンデ\':_s(vals[\'ハンデ\']),\n                     \'試走T\':_num(vals[\'試走T\']),\'競走T\':_num(vals[\'競走T\']),\'ST\':_num(vals[\'ST\']),\'結果区分\':\'通常\'})\n    return hr,cmap,rows\n\ndef import_workbook_to_db(content,filename=\'uploaded.xlsx\'):\n    wb=load_workbook(io.BytesIO(content),data_only=True); report=[]\n    for ws in wb.worksheets:\n        if not re.fullmatch(r\'選手\\d+\',ws.title): continue\n        name=_player_name(ws)\n        try: _,_,rows=_sheet_rows(ws); a,s=add_history_rows(name,rows,filename); report.append((name,a,s,\'OK\'))\n        except Exception as e: report.append((name,0,0,str(e)))\n    return pd.DataFrame(report,columns=[\'選手名\',\'追加\',\'重複スキップ\',\'状態\'])\n\ndef get_player_history(name,model_only=False):\n    extra=\'AND h.use_for_model=1\' if model_only else \'\'\n    with sqlite3.connect(DB_PATH) as con:\n        return pd.read_sql_query(f"""SELECT h.race_date AS 開催日,h.venue AS 開催場,h.race_no AS レース,\n        h.finish AS 着順,h.starters AS 出走,h.surface AS 走路,h.handicap AS ハンデ,h.trial_time AS 試走T,\n        h.race_time AS 競走T,h.start_time AS ST,h.result_status AS 結果区分\n        FROM race_history h JOIN players p ON p.player_id=h.player_id\n        WHERE p.player_name=? {extra} ORDER BY h.race_date DESC,h.history_id DESC""",con,params=(name,))\n\ndef enrich_workbook_from_db(content):\n    wb=load_workbook(io.BytesIO(content)); summary=[]\n    for ws in wb.worksheets:\n        if not re.fullmatch(r\'選手\\d+\',ws.title): continue\n        name=_player_name(ws)\n        try: hr,cmap,local=_sheet_rows(ws)\n        except Exception: continue\n        a,_=add_history_rows(name,local,\'予測Excel自動追加\')\n        dbdf=get_player_history(name)\n        for r in range(hr+1,ws.max_row+1):\n            for col in set(cmap.values()): ws.cell(r,col).value=None\n        for i,row in dbdf.iterrows():\n            rr=hr+1+i\n            for col in HISTORY_COLS_DB:\n                v=row[col]; ws.cell(rr,cmap[col]).value=None if pd.isna(v) else v\n        summary.append((name,len(dbdf),a))\n    bio=io.BytesIO(); wb.save(bio)\n    return bio.getvalue(),pd.DataFrame(summary,columns=[\'選手名\',\'DB履歴件数\',\'今回新規追加\'])\n\ndef make_result_entry_workbook(content):\n    wb=load_workbook(io.BytesIO(content))\n    if \'結果入力\' in wb.sheetnames: del wb[\'結果入力\']\n    ws=wb.create_sheet(\'結果入力\',0)\n    headers=[\'選手名\',\'着順\',\'競走T\',\'ST\',\'結果区分\',\'開催日\',\'開催場\',\'レース\',\'出走\',\'走路\',\'ハンデ\',\'試走T\']\n    ws.append(headers)\n    for c in ws[1]: c.font=Font(bold=True,color=\'FFFFFF\'); c.fill=PatternFill(\'solid\',fgColor=\'17365D\'); c.alignment=Alignment(horizontal=\'center\')\n    race_ws=wb[\'レース予測\'] if \'レース予測\' in wb.sheetnames else None; labels={}\n    if race_ws:\n        for r in range(1,min(30,race_ws.max_row)+1): labels[_s(race_ws.cell(r,1).value)]=race_ws.cell(r,2).value\n    players=[w for w in wb.worksheets if re.fullmatch(r\'選手\\d+\',w.title)]\n    for pws in players:\n        ws.append([_player_name(pws),\'\',\'\',\'\',\'通常\',_date_text(labels.get(\'レース開催日\')),_s(labels.get(\'今回の開催場\')),\n                   \'\',len(players),_s(labels.get(\'今回の走路\')),_s(pws[\'G2\'].value),_num(pws[\'E2\'].value)])\n    ws.freeze_panes=\'A2\'; ws.auto_filter.ref=f\'A1:L{ws.max_row}\'\n    bio=io.BytesIO(); wb.save(bio); return bio.getvalue()\n\ndef register_results_workbook(content,filename=\'result.xlsx\'):\n    wb=load_workbook(io.BytesIO(content),data_only=True)\n    if \'結果入力\' not in wb.sheetnames: raise ValueError(\'「結果入力」シートがありません\')\n    ws=wb[\'結果入力\']; h={_s(ws.cell(1,c).value):c for c in range(1,ws.max_column+1)}; report=[]\n    required=[\'選手名\',\'着順\',\'競走T\',\'ST\',\'結果区分\',\'開催日\',\'開催場\',\'レース\',\'出走\',\'走路\',\'ハンデ\',\'試走T\']\n    if any(x not in h for x in required): raise ValueError(\'結果入力シートの列が不足しています\')\n    for r in range(2,ws.max_row+1):\n        name=_s(ws.cell(r,h[\'選手名\']).value)\n        if not name: continue\n        status=_s(ws.cell(r,h[\'結果区分\']).value) or \'通常\'\n        row={\'開催日\':_date_text(ws.cell(r,h[\'開催日\']).value),\'開催場\':_s(ws.cell(r,h[\'開催場\']).value),\'レース\':_s(ws.cell(r,h[\'レース\']).value),\n             \'着順\':_num(ws.cell(r,h[\'着順\']).value),\'出走\':_num(ws.cell(r,h[\'出走\']).value),\'走路\':_s(ws.cell(r,h[\'走路\']).value),\n             \'ハンデ\':_s(ws.cell(r,h[\'ハンデ\']).value),\'試走T\':_num(ws.cell(r,h[\'試走T\']).value),\'競走T\':_num(ws.cell(r,h[\'競走T\']).value),\n             \'ST\':_num(ws.cell(r,h[\'ST\']).value),\'結果区分\':status}\n        if status==\'通常\' and row[\'着順\'] is None: report.append((name,0,\'着順未入力\')); continue\n        a,_=add_history_rows(name,[row],filename); report.append((name,a,\'追加\' if a else \'重複\'))\n    return pd.DataFrame(report,columns=[\'選手名\',\'追加件数\',\'状態\'])\n\nmount_and_init_db(); print(f\'選手DB: {DB_PATH}\')\nupload_db=widgets.FileUpload(accept=\'.xlsx\',multiple=False,description=\'履歴をDB登録\')\nupload_enrich=widgets.FileUpload(accept=\'.xlsx\',multiple=False,description=\'DB履歴を補充\')\nupload_template=widgets.FileUpload(accept=\'.xlsx\',multiple=False,description=\'結果入力票を作成\')\nupload_result=widgets.FileUpload(accept=\'.xlsx\',multiple=False,description=\'結果をDB追加\')\nout_db=widgets.Output()\n\ndef file_bytes(u):\n    v=u.value\n    if not v:return None,None\n    if isinstance(v,dict):\n        name=next(iter(v)); return name,bytes(v[name][\'content\'])\n    item=v[0]; return item[\'name\'],bytes(item[\'content\'])\n\ndef on_import(change):\n    name,data=file_bytes(upload_db)\n    if data:\n        with out_db:\n            _hidden_clear_output()\n            try: _hidden_display(import_workbook_to_db(data,name))\n            except Exception as e: print(\'エラー:\',e)\n\ndef on_enrich(change):\n    name,data=file_bytes(upload_enrich)\n    if data:\n        with out_db:\n            _hidden_clear_output()\n            try:\n                result,summary=enrich_workbook_from_db(data); outname=\'DB補充済み_\'+name\n                Path(\'/content/\'+outname).write_bytes(result); _hidden_display(summary); files.download(\'/content/\'+outname)\n            except Exception as e: print(\'エラー:\',e)\n\ndef on_template(change):\n    name,data=file_bytes(upload_template)\n    if data:\n        with out_db:\n            _hidden_clear_output()\n            try:\n                outname=\'結果入力用_\'+name; Path(\'/content/\'+outname).write_bytes(make_result_entry_workbook(data))\n                print(\'着順・競走T・ST・結果区分を入力してください。\'); files.download(\'/content/\'+outname)\n            except Exception as e: print(\'エラー:\',e)\n\ndef on_result(change):\n    name,data=file_bytes(upload_result)\n    if data:\n        with out_db:\n            _hidden_clear_output()\n            try: _hidden_display(register_results_workbook(data,name))\n            except Exception as e: print(\'エラー:\',e)\n\nupload_db.observe(on_import,names=\'value\'); upload_enrich.observe(on_enrich,names=\'value\')\nupload_template.observe(on_template,names=\'value\'); upload_result.observe(on_result,names=\'value\')\n_hidden_display(widgets.HTML(\'<h3>① 初回・未登録履歴をDBへ追加</h3>\'),upload_db)\n_hidden_display(widgets.HTML(\'<h3>② 今回のExcelへ登録済み履歴を自動補充</h3>\'),upload_enrich)\n_hidden_display(widgets.HTML(\'<h3>③ レース前に結果入力票を作成</h3>\'),upload_template)\n_hidden_display(widgets.HTML(\'<h3>④ レース後、入力済み結果をDBへ追加</h3>\'),upload_result,out_db)\n\n\nHISTORY_COLUMNS = [\n    "開催日", "開催場", "レース", "着順", "出走",\n    "走路", "ハンデ", "試走T", "競走T", "ST"\n]\n\n\n\ndef same_or_stronger_repass_penalty(chaser_strength, leader_strength):\n    """\n    一度前へ出た相手が同レベル以上なら、抜き返しに必要な差を大きくする。\n    leader_strength >= chaser_strength のとき強い追加障壁を与える。\n    """\n    chaser = float(chaser_strength)\n    leader = float(leader_strength)\n    gap = leader - chaser\n\n    if gap >= 5.0:\n        return 2.4\n    if gap >= 2.0:\n        return 1.8\n    if gap >= 0.0:\n        return 1.35\n    if gap >= -2.0:\n        return 0.75\n    return 0.25\n\n\ndef uploaded_file(value):\n    if not value:\n        return None, None\n    if isinstance(value, dict):\n        name = next(iter(value))\n        return name, bytes(value[name]["content"])\n    item = value[0]\n    return item["name"], bytes(item["content"])\n\n\ndef text(v):\n    return "" if v is None else str(v).strip()\n\n\ndef number(v, default=np.nan):\n    if v is None or v == "":\n        return default\n    if isinstance(v, (int, float)) and not isinstance(v, bool):\n        return float(v)\n    try:\n        return float(str(v).replace(",", "").strip())\n    except Exception:\n        return default\n\n\ndef handicap_number(v, default=np.nan):\n    if v is None or v == "":\n        return default\n    if isinstance(v, (int, float)):\n        return float(v)\n    m = re.search(r"-?\\d+(?:\\.\\d+)?", str(v))\n    return float(m.group()) if m else default\n\n\ndef excel_serial(v):\n    if v is None or v == "":\n        return np.nan\n    if isinstance(v, datetime):\n        base = datetime(1899, 12, 30)\n        return float((v - base).days)\n    if isinstance(v, date):\n        base = date(1899, 12, 30)\n        return float((v - base).days)\n    return number(v)\n\n\ndef mean_or_blank(values):\n    vals = [float(v) for v in values if not pd.isna(v)]\n    return float(np.mean(vals)) if vals else np.nan\n\n\ndef sample_std(values):\n    vals = [float(v) for v in values if not pd.isna(v)]\n    return float(np.std(vals, ddof=1)) if len(vals) >= 2 else np.nan\n\n\ndef find_header(ws):\n    for r in range(1, min(ws.max_row, 20) + 1):\n        vals = [text(ws.cell(r, c).value) for c in range(1, 20)]\n        if all(col in vals for col in HISTORY_COLUMNS):\n            return r, {col: vals.index(col) + 1 for col in HISTORY_COLUMNS}\n    raise ValueError(f"{ws.title}: 履歴10列の見出しが見つかりません")\n\n\ndef read_history(ws):\n    header_row, col_map = find_header(ws)\n    rows = []\n\n    for r in range(header_row + 1, ws.max_row + 1):\n        raw = {col: ws.cell(r, col_map[col]).value for col in HISTORY_COLUMNS}\n        if all(v in (None, "") for v in raw.values()):\n            continue\n\n        trial = number(raw["試走T"])\n        race_t = number(raw["競走T"])\n\n        rows.append({\n            "開催日": excel_serial(raw["開催日"]),\n            "開催場": text(raw["開催場"]),\n            "レース": text(raw["レース"]),\n            "着順": number(raw["着順"]),\n            "出走": number(raw["出走"]),\n            "走路": text(raw["走路"]),\n            "ハンデ": text(raw["ハンデ"]),\n            "ハンデ数値": handicap_number(raw["ハンデ"]),\n            "試走T": trial,\n            "競走T": race_t,\n            "ST": number(raw["ST"]),\n            "タイム差": (\n                race_t - trial\n                if not pd.isna(trial)\n                and not pd.isna(race_t)\n                and (race_t - trial) > 0\n                else np.nan\n            ),\n            "有効": (\n                not pd.isna(trial)\n                and not pd.isna(race_t)\n                and (race_t - trial) > 0\n            ),\n        })\n\n    return pd.DataFrame(rows)\n\n\ndef read_race(ws):\n    labels = {}\n    for r in range(1, min(ws.max_row, 30) + 1):\n        key = text(ws.cell(r, 1).value)\n        if key:\n            labels[key] = ws.cell(r, 2).value\n\n    return {\n        "開催日": excel_serial(labels.get("レース開催日")),\n        "開催場": text(labels.get("今回の開催場")),\n        "走路": text(labels.get("今回の走路")),\n    }\n\n\ndef read_settings(ws):\n    # 元Excelでは設定値の位置が固定\n    def b(row, default):\n        return number(ws.cell(row, 2).value, default)\n\n    return {\n        "最近重視日数": b(4, 30),\n        "同一開催場倍率": b(5, 1.35),\n        "別開催場倍率": b(6, 0.85),\n        "同一走路倍率": b(7, 1.5),\n        "別走路倍率": b(8, 0.7),\n        "同一ハンデ倍率": b(9, 1.4),\n        "近接ハンデ倍率": b(10, 1.1),\n        "遠隔ハンデ倍率": b(11, 0.75),\n        "ST補正係数": b(12, 0.04),\n        "10m換算秒": b(22, 0.018),\n\n        "試走点": b(29, 24),\n        "最近5走点": 10.0,\n        "ST点": b(31, 8),\n        "ハンデ適性点": b(32, 8),\n        "開催場適性点": b(33, 5),\n        "走路適性点": b(34, 5),\n        "位置取り点": b(35, 6),\n        "着順指数点": b(36, 8),\n        "勝率点": b(37, 6),\n        "連対率点": b(38, 4),\n        "上昇度点": b(39, 2),\n        "再現性点": b(40, 1),\n        "今回ハンデ勝率点": 4.0,\n\n        "ハンデ改善点": b(43, 6),\n        "満点改善幅": b(44, 20),\n        "前走比率": b(45, 0.6),\n        "3着以内率点": b(49, 6),\n        "平均着順点": b(50, 6),\n        "着順安定度点": b(51, 4),\n        "審査Pランク点": b(52, 6),\n    }\n\n\ndef current_player(ws):\n    return {\n        "選手名": text(ws["B2"].value) or ws.title,\n        "今回試走T": number(ws["E2"].value),\n        "今回ハンデ": handicap_number(ws["G2"].value),\n        "今回走路": text(ws["E3"].value),\n        "審査P": number(ws["I2"].value),\n        "現ランク": text(ws["I3"].value).upper(),\n    }\n\n\ndef normalize_surface(value):\n    """走路表記を 良・斑・湿 にそろえる。"""\n    s = text(value)\n    if "良" in s:\n        return "良"\n    if "斑" in s:\n        return "斑"\n    if "湿" in s:\n        return "湿"\n    return s\n\n\ndef surface_compatibility_weight(history_surface, current_surface):\n    """\n    今回走路に対する過去走路の採用率。\n    同一条件を中心にし、異なる条件はほぼ度外視する。\n    """\n    past = normalize_surface(history_surface)\n    now = normalize_surface(current_surface)\n\n    if not past or not now:\n        return 0.05\n    if past == now:\n        return 1.0\n\n    weights = {\n        # 今回が良走路\n        ("斑", "良"): 0.30,\n        ("湿", "良"): 0.02,\n\n        # 今回が湿走路\n        ("斑", "湿"): 0.20,\n        ("良", "湿"): 0.02,\n\n        # 今回が斑走路（良と湿の中間として扱う）\n        ("良", "斑"): 0.15,\n        ("湿", "斑"): 0.20,\n    }\n    return weights.get((past, now), 0.05)\n\n\ndef weighted_mean_series(values, weights):\n    v = pd.to_numeric(values, errors="coerce")\n    w = pd.to_numeric(weights, errors="coerce")\n    mask = v.notna() & w.notna() & (w > 0)\n    if not mask.any() or float(w[mask].sum()) == 0:\n        return np.nan\n    return float((v[mask] * w[mask]).sum() / w[mask].sum())\n\n\ndef weighted_rate(condition, weights):\n    c = pd.Series(condition, dtype=float)\n    w = pd.to_numeric(weights, errors="coerce")\n    mask = c.notna() & w.notna() & (w > 0)\n    if not mask.any() or float(w[mask].sum()) == 0:\n        return np.nan\n    return float((c[mask] * w[mask]).sum() / w[mask].sum())\n\n\ndef weighted_std_series(values, weights):\n    v = pd.to_numeric(values, errors="coerce")\n    w = pd.to_numeric(weights, errors="coerce")\n    mask = v.notna() & w.notna() & (w > 0)\n    if mask.sum() < 2 or float(w[mask].sum()) == 0:\n        return np.nan\n\n    vv = v[mask].astype(float)\n    ww = w[mask].astype(float)\n    avg = float((vv * ww).sum() / ww.sum())\n    variance = float((ww * (vv - avg) ** 2).sum() / ww.sum())\n    return float(np.sqrt(max(variance, 0.0)))\n\n\ndef prepare_history(df, current, race, settings):\n    x = df.copy()\n\n    if x.empty:\n        return x\n\n    # Ver10.6: 公式プロフィール取得時に、予測対象レースの確定結果が\n    # 履歴先頭へ混入している場合がある。同日・同開催場の行は予測から除外する。\n    same_day_result = (\n        pd.to_numeric(x["開催日"], errors="coerce") == float(race["開催日"])\n    ) & (x["開催場"].astype(str) == str(race["開催場"]))\n    x = x.loc[~same_day_result].copy()\n    if x.empty:\n        return x\n\n    x["経過日数"] = np.maximum(0, race["開催日"] - x["開催日"])\n    x["最近重み"] = np.exp(-x["経過日数"] / settings["最近重視日数"])\n\n    x["場重み"] = np.where(\n        x["開催場"] == race["開催場"],\n        settings["同一開催場倍率"],\n        settings["別開催場倍率"],\n    )\n\n    # 走路が異なる履歴は一律倍率ではなく、今回走路との近さで採用率を変える\n    x["走路適合重み"] = x["走路"].apply(\n        lambda surface: surface_compatibility_weight(surface, race["走路"])\n    )\n\n    # 従来の設定倍率は使用せず、1.0 / 0.20 / 0.15 / 0.10 / 0.02 を直接使用\n    x["走路重み"] = x["走路適合重み"]\n\n    diff = np.abs(x["ハンデ数値"] - current["今回ハンデ"])\n    x["ハンデ重み"] = np.where(\n        diff == 0,\n        settings["同一ハンデ倍率"],\n        np.where(\n            diff <= 10,\n            settings["近接ハンデ倍率"],\n            settings["遠隔ハンデ倍率"],\n        ),\n    )\n\n    x["総合重み"] = (\n        x["最近重み"]\n        * x["場重み"]\n        * x["走路重み"]\n        * x["ハンデ重み"]\n    )\n\n    x.loc[~x["有効"], "総合重み"] = np.nan\n    return x\n\n\ndef smooth_points(values, maximum, lower_is_better=True, zero_is_zero=False):\n    """\n    順位だけではなく、平均との差とばらつきを使って点数化する。\n    平均値付近は最大点の50%、おおむね±2標準偏差で0～最大点に収まる。\n    """\n    arr = np.array(values, dtype=float)\n    valid = arr[~np.isnan(arr)]\n\n    if len(valid) == 0:\n        return [np.nan] * len(arr)\n\n    mean = float(np.mean(valid))\n    std = float(np.std(valid, ddof=1)) if len(valid) >= 2 else 0.0\n\n    # 全員ほぼ同じ場合は中間点\n    if std < 1e-9:\n        result = []\n        for v in arr:\n            if np.isnan(v):\n                result.append(np.nan)\n            elif zero_is_zero and v == 0:\n                result.append(0.0)\n            else:\n                result.append(maximum * 0.5)\n        return result\n\n    result = []\n    for v in arr:\n        if np.isnan(v):\n            result.append(np.nan)\n            continue\n\n        if zero_is_zero and v == 0:\n            result.append(0.0)\n            continue\n\n        z = (v - mean) / std\n        if lower_is_better:\n            z = -z\n\n        # z=-2で0点、z=0で50%、z=+2で満点\n        normalized = np.clip(0.5 + z / 4.0, 0.0, 1.0)\n        result.append(float(normalized * maximum))\n\n    return result\n\n\ndef player_metrics(car, current, hist, race, settings):\n    x = prepare_history(hist, current, race, settings)\n\n    valid = x[x["有効"]].copy()\n    first5 = valid.head(5)\n    first10 = valid.head(10)\n    first30 = valid.head(30)\n\n    converted_trial = (\n        current["今回試走T"]\n        + current["今回ハンデ"] / 10 * settings["10m換算秒"]\n    )\n\n    # タイム差も今回走路との適合度で加重する\n    recent5_diff = weighted_mean_series(\n        first5["タイム差"],\n        first5["走路適合重み"],\n    )\n\n    # 最近の試走水準と今回試走の悪化幅。\n    # 過去ハンデも今回と同じ10m換算秒で補正して比較する。\n    recent_trial_converted = (\n        pd.to_numeric(first5["試走T"], errors="coerce")\n        + pd.to_numeric(first5["ハンデ数値"], errors="coerce").fillna(0.0)\n        / 10 * settings["10m換算秒"]\n    )\n    recent_trial_baseline = weighted_mean_series(\n        recent_trial_converted,\n        first5["走路適合重み"],\n    )\n    current_trial_deterioration = (\n        max(0.0, converted_trial - recent_trial_baseline)\n        if not pd.isna(recent_trial_baseline) else 0.0\n    )\n    # Ver12.2: 選手自身の平常試走からの良化・悪化を測る。\n    # 絶対タイムだけではなく、普段より何秒速いかを当日の仕上がりとして使う。\n    recent_trial_sd = weighted_std_series(\n        recent_trial_converted, first5["走路適合重み"]\n    )\n    current_trial_change = (\n        float(recent_trial_baseline - converted_trial)\n        if not pd.isna(recent_trial_baseline) else 0.0\n    )\n\n    weighted_rows = x[\n        x["ST"].notna() & x["総合重み"].notna()\n    ]\n    weighted_st = (\n        float((weighted_rows["ST"] * weighted_rows["総合重み"]).sum()\n              / weighted_rows["総合重み"].sum())\n        if not weighted_rows.empty and weighted_rows["総合重み"].sum() != 0\n        else np.nan\n    )\n\n    # STの安定性: 平均が速くてもばらつきが大きい選手は少し評価を下げる\n    st_std = weighted_std_series(\n        weighted_rows["ST"], weighted_rows["総合重み"]\n    ) if not weighted_rows.empty else np.nan\n    st_stability = (\n        1 / (1 + st_std * 25)\n        if not pd.isna(st_std) else np.nan\n    )\n\n    # 試走再現率: 本走タイムと試走タイムの差が小さく、安定する選手を評価\n    trial_rows = first10[\n        first10["試走T"].notna() & first10["競走T"].notna()\n    ].copy()\n    if not trial_rows.empty:\n        trial_rows["試走本走差"] = (\n            pd.to_numeric(trial_rows["競走T"], errors="coerce")\n            - pd.to_numeric(trial_rows["試走T"], errors="coerce")\n        ).abs()\n        trial_gap_mean = weighted_mean_series(\n            trial_rows["試走本走差"], trial_rows["走路適合重み"]\n        )\n        trial_gap_std = weighted_std_series(\n            trial_rows["試走本走差"], trial_rows["走路適合重み"]\n        )\n        trial_reproduction = (\n            1 / (1 + trial_gap_mean * 12 + trial_gap_std * 10)\n            if not pd.isna(trial_gap_mean) else np.nan\n        )\n    else:\n        trial_gap_mean = np.nan\n        trial_gap_std = np.nan\n        trial_reproduction = np.nan\n\n    # Ver6.9: 試走信頼度とレース巧者指数\n    craft_rows = first30[\n        first30["試走T"].notna() & first30["着順"].notna()\n    ].copy()\n    if len(craft_rows) >= 5:\n        craft_rows["試走換算履歴"] = (\n            pd.to_numeric(craft_rows["試走T"], errors="coerce")\n            + pd.to_numeric(craft_rows["ハンデ数値"], errors="coerce").fillna(0.0)\n            / 10 * settings["10m換算秒"]\n        )\n        craft_rows["試走百分位"] = craft_rows["試走換算履歴"].rank(\n            pct=True, method="average", ascending=True\n        )\n        craft_rows["着順百分位"] = pd.to_numeric(\n            craft_rows["着順"], errors="coerce"\n        ).rank(pct=True, method="average", ascending=True)\n        trial_finish_corr = craft_rows["試走百分位"].corr(\n            craft_rows["着順百分位"], method="spearman"\n        )\n        if pd.isna(trial_finish_corr):\n            trial_finish_corr = 0.0\n        trial_trust = float(np.clip(\n            0.50 + 0.50 * trial_finish_corr, 0.20, 1.00\n        ))\n        craft_rows["巧者差"] = (\n            craft_rows["試走百分位"] - craft_rows["着順百分位"]\n        )\n        race_craft_index = weighted_mean_series(\n            craft_rows["巧者差"], craft_rows["走路適合重み"]\n        )\n    else:\n        trial_finish_corr = np.nan\n        trial_trust = 0.50\n        race_craft_index = 0.0\n\n    same_h = valid[\n        valid["ハンデ"] == f"{int(current[\'今回ハンデ\'])}m"\n    ]\n    same_h_diff = (\n        mean_or_blank(same_h["タイム差"].tolist())\n        if not same_h.empty else recent5_diff\n    )\n\n    same_venue = valid[valid["開催場"] == race["開催場"]]\n    venue_diff = mean_or_blank(same_venue["タイム差"].tolist())\n\n    same_surface = valid[valid["走路"] == race["走路"]]\n    surface_diff = mean_or_blank(same_surface["タイム差"].tolist())\n\n    # 着順系も今回走路との適合度で加重する\n    finish_scores = {1: 10, 2: 8, 3: 6, 4: 5, 5: 4, 6: 3, 7: 2, 8: 1}\n\n    scored_finish10 = first10["着順"].map(\n        lambda v: finish_scores.get(int(v), np.nan) if not pd.isna(v) else np.nan\n    )\n    finish_index = weighted_mean_series(\n        scored_finish10,\n        first10["走路適合重み"],\n    )\n\n    finish30 = pd.to_numeric(first30["着順"], errors="coerce")\n    road_weight30 = first30["走路適合重み"]\n\n    win_rate = weighted_rate(\n        (finish30 == 1).where(finish30.notna(), np.nan),\n        road_weight30,\n    )\n    quinella_rate = weighted_rate(\n        finish30.isin([1, 2]).where(finish30.notna(), np.nan),\n        road_weight30,\n    )\n    show_rate = weighted_rate(\n        finish30.isin([1, 2, 3]).where(finish30.notna(), np.nan),\n        road_weight30,\n    )\n\n    average_finish = weighted_mean_series(\n        first10["着順"],\n        first10["走路適合重み"],\n    )\n    finish_std10 = weighted_std_series(\n        first10["着順"],\n        first10["走路適合重み"],\n    )\n    finish_stability = (\n        1 / (1 + finish_std10)\n        if not pd.isna(finish_std10) else np.nan\n    )\n\n    newest5_df = valid.head(5)\n    previous5_df = valid.iloc[5:10]\n    newest5_avg = weighted_mean_series(\n        newest5_df["着順"], newest5_df["走路適合重み"]\n    )\n    previous5_avg = weighted_mean_series(\n        previous5_df["着順"], previous5_df["走路適合重み"]\n    )\n    improvement = (\n        float(previous5_avg - newest5_avg)\n        if not pd.isna(newest5_avg) and not pd.isna(previous5_avg)\n        else np.nan\n    )\n\n    diff_std = weighted_std_series(\n        first10["タイム差"],\n        first10["走路適合重み"],\n    )\n    reproducibility = (\n        1 / (1 + diff_std * 100)\n        if not pd.isna(diff_std) else np.nan\n    )\n\n    same_h_30 = first30[\n        first30["ハンデ"] == f"{int(current[\'今回ハンデ\'])}m"\n    ]\n    current_h_win_rate = (\n        float((same_h_30["着順"] == 1).sum() / len(same_h_30))\n        if len(same_h_30) else 0.0\n    )\n\n    prev_h = (\n        float(valid.iloc[0]["ハンデ数値"])\n        if len(valid) and not pd.isna(valid.iloc[0]["ハンデ数値"])\n        else np.nan\n    )\n    avg5_h = mean_or_blank(valid.head(5)["ハンデ数値"].tolist())\n\n    handicap_improvement = (\n        max(0, prev_h - current["今回ハンデ"]) * settings["前走比率"]\n        + max(0, avg5_h - current["今回ハンデ"]) * (1 - settings["前走比率"])\n        if not pd.isna(prev_h) and not pd.isna(avg5_h)\n        else np.nan\n    )\n\n    handicap_change_score = (\n        min(handicap_improvement / settings["満点改善幅"], 1)\n        * settings["ハンデ改善点"]\n        if not pd.isna(handicap_improvement) and settings["満点改善幅"] != 0\n        else 0.0\n    )\n\n    # 直近3走の好調補正\n    # 今回走路との適合度を使うため、湿走路の好走だけで大きく上がりにくい。\n    recent3 = valid.head(3).copy()\n    if len(recent3):\n        recent3_top3_rate = weighted_rate(\n            (pd.to_numeric(recent3["着順"], errors="coerce") <= 3)\n            .where(pd.to_numeric(recent3["着順"], errors="coerce").notna(), np.nan),\n            recent3["走路適合重み"],\n        )\n    else:\n        recent3_top3_rate = 0.0\n\n    if len(recent3) >= 3 and recent3_top3_rate >= (2 / 3):\n        recent3_form_bonus = 3.0\n    elif len(recent3) >= 3 and recent3_top3_rate >= (1 / 3):\n        recent3_form_bonus = 1.2\n    else:\n        recent3_form_bonus = 0.0\n\n    # 直近5走補正: 一時的な1走だけでなく、5走全体の好調・不調を反映\n    recent5_finish = pd.to_numeric(first5["着順"], errors="coerce")\n    recent5_top3_rate = weighted_rate(\n        (recent5_finish <= 3).where(recent5_finish.notna(), np.nan),\n        first5["走路適合重み"],\n    ) if len(first5) else np.nan\n    recent5_avg_finish = weighted_mean_series(\n        recent5_finish, first5["走路適合重み"]\n    ) if len(first5) else np.nan\n\n    if len(first5) >= 5 and not pd.isna(recent5_avg_finish):\n        if recent5_avg_finish <= 3.0 and recent5_top3_rate >= 0.60:\n            recent5_form_bonus = 3.0\n        elif recent5_avg_finish <= 4.0 and recent5_top3_rate >= 0.40:\n            recent5_form_bonus = 1.5\n        elif recent5_avg_finish >= 6.0:\n            recent5_form_bonus = -2.0\n        elif recent5_avg_finish >= 5.0:\n            recent5_form_bonus = -0.8\n        else:\n            recent5_form_bonus = 0.0\n    else:\n        recent5_form_bonus = 0.0\n\n    # Ver6.8: 今回走路だけに絞り過ぎず、直近の着順そのものから\n    # 「今回の試走不振が単発の外れだった可能性」を推定する。\n    # 湿走路の能力評価を上げる処理ではなく、試走1回の信頼度だけを緩める。\n    raw_recent3_finish = pd.to_numeric(valid.head(3)["着順"], errors="coerce")\n    raw_recent5_finish = pd.to_numeric(valid.head(5)["着順"], errors="coerce")\n    raw_recent3_top3 = (\n        float((raw_recent3_finish <= 3).mean())\n        if raw_recent3_finish.notna().any() else 0.0\n    )\n    raw_recent3_top2 = (\n        float((raw_recent3_finish <= 2).mean())\n        if raw_recent3_finish.notna().any() else 0.0\n    )\n    raw_recent5_top3 = (\n        float((raw_recent5_finish <= 3).mean())\n        if raw_recent5_finish.notna().any() else 0.0\n    )\n    raw_recent5_poor = (\n        float((raw_recent5_finish >= 6).mean())\n        if raw_recent5_finish.notna().any() else 0.0\n    )\n    raw_recent4_top3 = (\n        float((raw_recent5_finish.head(4) <= 3).mean())\n        if raw_recent5_finish.head(4).notna().any() else 0.0\n    )\n    raw_recent2_poor = (\n        float((raw_recent5_finish.head(2) >= 6).mean())\n        if raw_recent5_finish.head(2).notna().any() else 0.0\n    )\n\n    # Ver10.6: 上昇カーブ指数。validは新しい履歴から並ぶ前提で、\n    # 古い3走平均から新しい3走平均への改善幅を、回帰傾向と合わせて連続値化する。\n    curve_finish = pd.to_numeric(valid.head(8)["着順"], errors="coerce").dropna().to_numpy(float)\n    if len(curve_finish) >= 4:\n        recent_n = min(3, len(curve_finish) // 2)\n        old_part = curve_finish[-recent_n:]\n        new_part = curve_finish[:recent_n]\n        level_gain = float(np.mean(old_part) - np.mean(new_part))\n        # 時系列を古い→新しいへ反転し、負の傾きほど着順改善\n        chrono = curve_finish[::-1]\n        x_curve = np.arange(len(chrono), dtype=float)\n        slope = float(np.polyfit(x_curve, chrono, 1)[0]) if len(chrono) >= 3 else 0.0\n        upward_curve_index = float(np.clip(level_gain * 0.18 - slope * 0.55, -1.0, 1.0))\n    else:\n        upward_curve_index = 0.0\n\n    # Ver10.6: 終盤指数。対戦相手ごとの試走順位は入力に無いため、\n    # 選手自身の履歴内で「試走水準より着順が良かったか」を標準化して推定する。\n    late_rows = valid.head(20).copy()\n    late_rows = late_rows[late_rows["試走T"].notna() & late_rows["着順"].notna()]\n    if len(late_rows) >= 5:\n        late_trial = (\n            pd.to_numeric(late_rows["試走T"], errors="coerce")\n            + pd.to_numeric(late_rows["ハンデ数値"], errors="coerce").fillna(0.0)\n            / 10 * settings["10m換算秒"]\n        )\n        late_finish = pd.to_numeric(late_rows["着順"], errors="coerce")\n        trial_sd = float(late_trial.std(ddof=0))\n        finish_sd_local = float(late_finish.std(ddof=0))\n        trial_z = (late_trial - late_trial.mean()) / (trial_sd if trial_sd > 1e-9 else 1.0)\n        finish_z = (late_finish - late_finish.mean()) / (finish_sd_local if finish_sd_local > 1e-9 else 1.0)\n        # 試走が悪い側（正）でも着順が良い側（負）なら正の終盤力\n        conversion_advantage = trial_z - finish_z\n        recency_w = np.exp(-np.arange(len(late_rows), dtype=float) / 7.0)\n        final_kick_index = float(np.clip(np.average(conversion_advantage, weights=recency_w) / 1.8, -1.0, 1.0))\n    else:\n        final_kick_index = 0.0\n\n    # 直近3走の複数好走を重視。1走だけの好走では救済を弱くする。\n    trial_outlier_confidence = np.clip(\n        (raw_recent3_top3 - 1 / 3) * 1.05\n        + raw_recent3_top2 * 0.45\n        + max(0.0, raw_recent5_top3 - 0.40) * 0.55,\n        0.0,\n        1.0,\n    )\n\n    # Ver12.2: 過去レースから「好位置を取れた時の粘り」と「出遅れ時の追上げ」を推定する。\n    # 1周目順位は履歴にないため、ST・枠位置・ハンデ帯から序盤位置を確率的に推定する。\n    pos_rows = first30[\n        first30["ST"].notna() & first30["着順"].notna()\n    ].copy()\n    position_sample = int(len(pos_rows))\n    if position_sample >= 6:\n        st_num = pd.to_numeric(pos_rows["ST"], errors="coerce")\n        finish_num = pd.to_numeric(pos_rows["着順"], errors="coerce")\n        lane_num = pd.to_numeric(pos_rows["出走"], errors="coerce").fillna(4.5)\n        hand_num = pd.to_numeric(pos_rows["ハンデ数値"], errors="coerce").fillna(0.0)\n        st_center = float(st_num.median())\n        st_scale = float(st_num.std(ddof=1)) if len(st_num) >= 2 else 0.04\n        st_scale = max(0.025, min(0.10, st_scale if np.isfinite(st_scale) else 0.04))\n        # 小さいほど序盤で前にいる可能性が高い。内枠効果は控えめにする。\n        early_score = (st_num - st_center) / st_scale + (lane_num - 4.5) * 0.075 + hand_num * 0.010\n        good_cut = float(early_score.quantile(0.38))\n        poor_cut = float(early_score.quantile(0.68))\n        good_mask = early_score <= good_cut\n        poor_mask = early_score >= poor_cut\n        good_n = int(good_mask.sum()); poor_n = int(poor_mask.sum())\n        good_top3 = float((finish_num[good_mask] <= 3).mean()) if good_n >= 2 else np.nan\n        good_top2 = float((finish_num[good_mask] <= 2).mean()) if good_n >= 2 else np.nan\n        good_avg = float(finish_num[good_mask].mean()) if good_n >= 2 else np.nan\n        poor_top3 = float((finish_num[poor_mask] <= 3).mean()) if poor_n >= 2 else np.nan\n        poor_avg = float(finish_num[poor_mask].mean()) if poor_n >= 2 else np.nan\n        # 少数標本は全体平均へ縮約し、極端な判定を避ける。\n        overall_top3 = float((finish_num <= 3).mean())\n        shrink_good = min(1.0, good_n / 8.0)\n        shrink_poor = min(1.0, poor_n / 8.0)\n        good_top3_s = overall_top3 + (good_top3 - overall_top3) * shrink_good if np.isfinite(good_top3) else overall_top3\n        good_top2_s = float((finish_num <= 2).mean()) + (good_top2 - float((finish_num <= 2).mean())) * shrink_good if np.isfinite(good_top2) else float((finish_num <= 2).mean())\n        poor_top3_s = overall_top3 + (poor_top3 - overall_top3) * shrink_poor if np.isfinite(poor_top3) else overall_top3\n        position_hold_index = float(np.clip(good_top3_s * 0.72 + good_top2_s * 0.28, 0.0, 1.0))\n        rear_chase_index = float(np.clip(poor_top3_s, 0.0, 1.0))\n        position_dependency = float(np.clip((good_top3_s - poor_top3_s + 0.45) / 0.90, 0.0, 1.0))\n        bipolar_index = float(np.clip(abs(good_top3_s - poor_top3_s) * 0.80 + min(1.0, float(finish_num.std(ddof=1)) / 2.6) * 0.20, 0.0, 1.0))\n    else:\n        good_n = poor_n = 0\n        good_avg = poor_avg = np.nan\n        position_hold_index = 0.50\n        rear_chase_index = 0.35\n        position_dependency = 0.50\n        bipolar_index = 0.30\n\n    return {\n        "車": car,\n        "選手名": current["選手名"],\n        "ハンデ": current["今回ハンデ"],\n        "試走換算": converted_trial,\n        "直近5差": recent5_diff,\n        "平均ST": weighted_st,\n        "ST標準偏差": st_std,\n        "ST安定性": st_stability,\n        "試走本走差": trial_gap_mean,\n        "試走本走差標準偏差": trial_gap_std,\n        "試走再現率": trial_reproduction,\n        "試走着順相関": trial_finish_corr,\n        "試走信頼度": trial_trust,\n        "レース巧者指数": race_craft_index,\n        "同ハンデ差": same_h_diff,\n        "開催場差": venue_diff,\n        "走路差": surface_diff,\n        "着順指数": finish_index,\n        "勝率": win_rate,\n        "連対率": quinella_rate,\n        "上昇度": improvement,\n        "再現性": reproducibility,\n        "今回ハンデ勝率": current_h_win_rate,\n        "3着以内率": show_rate,\n        "平均着順": average_finish,\n        "着順安定度": finish_stability,\n        "審査P": current["審査P"],\n        "現ランク": current["現ランク"],\n        "前走ハンデ": prev_h,\n        "直近5走平均ハンデ": avg5_h,\n        "ハンデ改善量": handicap_improvement,\n        "ハンデ変化点": handicap_change_score,\n        "着順標準偏差": sample_std(first30["着順"].tolist()),\n        "直近3走好調補正": recent3_form_bonus,\n        "直近5走補正": recent5_form_bonus,\n        "直近5走平均着順": recent5_avg_finish,\n        "直近5走3着内率": recent5_top3_rate,\n        "直近3走生3着内率": raw_recent3_top3,\n        "直近3走生2着内率": raw_recent3_top2,\n        "直近5走生3着内率": raw_recent5_top3,\n        "直近5走生凡走率": raw_recent5_poor,\n        "直近4走生3着内率": raw_recent4_top3,\n        "直近2走生凡走率": raw_recent2_poor,\n        "上昇カーブ指数": upward_curve_index,\n        "終盤指数": final_kick_index,\n        "試走単発外れ信頼度": float(trial_outlier_confidence),\n        "直近試走基準": recent_trial_baseline,\n        "直近試走標準偏差": recent_trial_sd,\n        "自己試走変化秒": current_trial_change,\n        "今回試走悪化幅": float(current_trial_deterioration),\n        "位置分析走数": position_sample,\n        "好位置推定走数": good_n,\n        "後方推定走数": poor_n,\n        "好位置時平均着順": good_avg,\n        "後方時平均着順": poor_avg,\n        "好位置維持指数": position_hold_index,\n        "後方追上げ指数": rear_chase_index,\n        "位置依存指数": position_dependency,\n        "二極化指数": bipolar_index,\n    }\n\n\ndef outer_lane_penalty(df, max_penalty=4.5):\n    """\n    10m以上の同ハンデ帯で横並びが3人以上の場合、外側ほど減点する。\n\n    - 0m線は対象外\n    - 同ハンデ2人以下は対象外\n    - 人数が多いほど補正を強くする\n    - 最外枠でも最大4.5点まで\n    """\n    penalty = pd.Series(0.0, index=df.index)\n    same_line_count = pd.Series(1, index=df.index, dtype=int)\n    outer_order = pd.Series(1, index=df.index, dtype=int)\n\n    for handicap, group in df.groupby("ハンデ", dropna=False):\n        if pd.isna(handicap):\n            continue\n\n        ordered = group.sort_values("車")\n        count = len(ordered)\n\n        for order, idx in enumerate(ordered.index, 1):\n            same_line_count.loc[idx] = count\n            outer_order.loc[idx] = order\n\n        if float(handicap) < 10 or count < 3:\n            continue\n\n        # 3人で1/3、4人で2/3、5人以上で最大強度\n        crowd_factor = min(1.0, max(0.0, (count - 2) / 3.0))\n\n        for order, idx in enumerate(ordered.index, 1):\n            # 内から外への位置を0～1で表す\n            outer_ratio = (order - 1) / (count - 1)\n\n            # 内側半分は減点せず、外側半分から滑らかに減点\n            outer_exposure = max(0.0, (outer_ratio - 0.5) / 0.5)\n            penalty.loc[idx] = -max_penalty * crowd_factor * outer_exposure\n\n    return penalty, same_line_count, outer_order\n\n\ndef add_race_type_features(df):\n    """逃げ成功率・内枠残存率・レースタイプを当該レースの相対比較から作る。"""\n    df = df.copy()\n    handicap = pd.to_numeric(df["ハンデ"], errors="coerce").fillna(0.0)\n    front_h = float(handicap.min())\n    front_mask = handicap == front_h\n\n    escape_prob = np.zeros(len(df), dtype=float)\n    inner_hold_prob = np.zeros(len(df), dtype=float)\n    inner_ratio = np.zeros(len(df), dtype=float)\n    st_speed_score = np.zeros(len(df), dtype=float)\n\n    front = df.loc[front_mask].sort_values("車")\n    count = len(front)\n    if count:\n        st_vals = pd.to_numeric(front["平均ST"], errors="coerce").fillna(0.20)\n        st_rank = st_vals.rank(method="average", ascending=True)\n        if count > 1:\n            speed = 1.0 - (st_rank - 1.0) / (count - 1.0)\n        else:\n            speed = pd.Series(0.60, index=front.index)\n\n        st_std = pd.to_numeric(front["ST標準偏差"], errors="coerce").fillna(0.055)\n        stability = 1.0 - np.clip((st_std - 0.020) / 0.075, 0.0, 1.0)\n\n        if count > 1:\n            lane = pd.Series(\n                [1.0 - i / (count - 1.0) for i in range(count)],\n                index=front.index,\n                dtype=float,\n            )\n        else:\n            lane = pd.Series(0.70, index=front.index, dtype=float)\n\n        quinella = pd.to_numeric(front["連対率"], errors="coerce").fillna(0.15).clip(0, 1)\n        show = pd.to_numeric(front["3着以内率"], errors="coerce").fillna(0.25).clip(0, 1)\n        recent_avg = pd.to_numeric(front["直近5走平均着順"], errors="coerce").fillna(5.0)\n        recent_hold = np.clip((6.5 - recent_avg) / 5.5, 0.0, 1.0)\n        sustain = np.clip(quinella * 0.45 + show * 0.25 + recent_hold * 0.30, 0.0, 1.0)\n\n        escape_raw = (\n            0.06\n            + speed.to_numpy(float) * 0.46\n            + stability.to_numpy(float) * 0.22\n            + lane.to_numpy(float) * 0.12\n            + sustain.to_numpy(float) * 0.20\n        )\n        escape_raw = np.clip(escape_raw, 0.08, 0.88)\n\n        # 逃げ切れなくても、最内寄り・ST安定・近況の粘りがあれば\n        # 2～4番手でコースを守る確率を別に持たせる。\n        hold_raw = (\n            0.10\n            + lane.to_numpy(float) * 0.38\n            + stability.to_numpy(float) * 0.18\n            + show.to_numpy(float) * 0.17\n            + quinella.to_numpy(float) * 0.08\n            + recent_hold.to_numpy(float) * 0.11\n            + escape_raw * 0.05\n            - (1.0 - speed.to_numpy(float)) * 0.04\n        )\n        hold_raw = np.clip(hold_raw, 0.10, 0.86)\n\n        for idx, escape_value, hold_value in zip(front.index, escape_raw, hold_raw):\n            pos = df.index.get_loc(idx)\n            escape_prob[pos] = float(escape_value)\n            inner_hold_prob[pos] = float(hold_value)\n            inner_ratio[pos] = float(lane.loc[idx])\n            st_speed_score[pos] = float(speed.loc[idx])\n\n    df["前線ハンデ"] = front_h\n    df["前線内側度"] = inner_ratio\n    df["同ハンデST優位度"] = st_speed_score\n    df["逃げ成功率"] = escape_prob\n    df["内枠残存率"] = inner_hold_prob\n\n    race_types = []\n    for _, row in df.iterrows():\n        is_front = float(row["ハンデ"]) == front_h\n        escape = float(row["逃げ成功率"])\n        hold = float(row["内枠残存率"])\n        craft = float(row.get("レース巧者指数", 0.0) or 0.0)\n        trust = float(row.get("試走信頼度", 0.5) or 0.5)\n        finish_sd = float(row.get("着順標準偏差", 1.5) or 1.5)\n        st_sd = float(row.get("ST標準偏差", 0.05) or 0.05)\n\n        if is_front and escape >= 0.58:\n            race_types.append("逃げ型")\n        elif is_front and hold >= 0.58:\n            race_types.append("内枠粘り型")\n        elif craft >= 0.08:\n            race_types.append("追込型")\n        elif trust >= 0.62 and craft < -0.03:\n            race_types.append("試走型")\n        elif finish_sd >= 2.35 or st_sd >= 0.070:\n            race_types.append("ムラ型")\n        else:\n            race_types.append("バランス型")\n    df["レースタイプ"] = race_types\n    return df\n\ndef calculate_excel_model(metrics, settings):\n    df = pd.DataFrame(metrics)\n    df = add_race_type_features(df)\n\n    df["試走点"] = smooth_points(\n        df["試走換算"].tolist(),\n        settings["試走点"],\n        lower_is_better=True,\n    )\n    # 信頼度が低い選手は、良い試走も悪い試走も中間点へ圧縮する。\n    trial_midpoint = settings["試走点"] * 0.50\n    trial_trust = pd.to_numeric(\n        df["試走信頼度"], errors="coerce"\n    ).fillna(0.50).clip(0.20, 1.00)\n    df["試走点"] = (\n        trial_midpoint\n        + (df["試走点"] - trial_midpoint) * trial_trust\n    ) * 0.85\n\n    # Ver6.8: 直近に複数回の好走がある選手は、今回試走が悪くても\n    # その1回だけが外れ値だった可能性を残す。全員一律ではなく、\n    # 試走点が集団中央値を下回った分だけ最大55%戻す。\n    trial_median = float(pd.to_numeric(df["試走点"], errors="coerce").median())\n    relief_conf = pd.to_numeric(\n        df["試走単発外れ信頼度"], errors="coerce"\n    ).fillna(0.0).clip(0.0, 1.0)\n    trial_shortfall = np.maximum(0.0, trial_median - df["試走点"])\n    trial_deterioration = pd.to_numeric(\n        df["今回試走悪化幅"], errors="coerce"\n    ).fillna(0.0).clip(lower=0.0)\n\n    # 集団平均との差だけでなく、その選手自身の最近の試走水準から\n    # 0.05秒以上悪化していれば最大効果として扱う。\n    personal_outlier_scale = np.clip(trial_deterioration / 0.05, 0.0, 1.0)\n    df["試走単発救済点"] = (\n        trial_shortfall * relief_conf * 0.35\n        + personal_outlier_scale * relief_conf * 4.0\n    )\n\n    # Ver8.1: 直近好調で今回試走だけ悪い選手を「近況主役候補」として点数側でも評価。\n    # 車番ではなく、直近3走・5走と試走単発外れ信頼度の組合せで決める。\n    recent3_lead = np.clip(\n        (pd.to_numeric(df["直近3走好調補正"], errors="coerce").fillna(0.0) + 1.0) / 4.5,\n        0.0, 1.0,\n    )\n    recent5_lead = np.clip(\n        (pd.to_numeric(df["直近5走補正"], errors="coerce").fillna(0.0) + 1.0) / 4.5,\n        0.0, 1.0,\n    )\n    df["近況主役点"] = (\n        relief_conf\n        * (0.58 * recent3_lead + 0.42 * recent5_lead)\n        * (2.8 + personal_outlier_scale * 2.7)\n    )\n\n    df["最近点"] = smooth_points(\n        df["直近5差"].tolist(),\n        settings["最近5走点"],\n        lower_is_better=True,\n    )\n    df["ST点"] = smooth_points(\n        df["平均ST"].tolist(),\n        settings["ST点"],\n        lower_is_better=True,\n    )\n    df["ST安定点"] = smooth_points(\n        df["ST安定性"].tolist(),\n        2.5,\n        lower_is_better=False,\n    )\n    df["試走再現点"] = smooth_points(\n        df["試走再現率"].tolist(),\n        3.5,\n        lower_is_better=False,\n    )\n    df["レース巧者点"] = smooth_points(\n        df["レース巧者指数"].tolist(),\n        4.0,\n        lower_is_better=False,\n    )\n    df["ハンデ点"] = smooth_points(\n        df["同ハンデ差"].tolist(),\n        settings["ハンデ適性点"],\n        lower_is_better=True,\n    )\n    df["開催場点"] = smooth_points(\n        df["開催場差"].tolist(),\n        settings["開催場適性点"],\n        lower_is_better=True,\n    )\n    df["走路点"] = smooth_points(\n        df["走路差"].tolist(),\n        settings["走路適性点"],\n        lower_is_better=True,\n    )\n\n    # Ver7.4: 位置の利を「逃げ切り」だけでなく「インで残る力」でも評価。\n    # 逃げ成功率が低めでも、最内でコースを守れる選手の位置点を失わせない。\n    front_h = float(pd.to_numeric(df["ハンデ"], errors="coerce").min())\n    position_use = (\n        pd.to_numeric(df["逃げ成功率"], errors="coerce").fillna(0.0) * 0.48\n        + pd.to_numeric(df["内枠残存率"], errors="coerce").fillna(0.0) * 0.52\n    ).clip(0.0, 1.0)\n    df["位置活用率"] = position_use\n    df["位置点"] = [\n        settings["位置取り点"] * (0.30 + 0.70 * float(use))\n        if float(h) == front_h\n        else settings["位置取り点"] * 0.2\n        if float(h) == front_h + 10\n        else 0.0\n        for h, use in zip(df["ハンデ"], position_use)\n    ]\n\n    # Ver7.5: 最前線の最内は、逃げ切れなくても距離ロスが少なく、\n    # コースを守って後続の壁になりやすい。その戦略価値を点数側にも反映する。\n    front_mask = pd.to_numeric(df["ハンデ"], errors="coerce").fillna(0.0) == front_h\n    lane_value = pd.to_numeric(df["前線内側度"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    hold_value = pd.to_numeric(df["内枠残存率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    recent_value = np.clip(\n        (6.5 - pd.to_numeric(df["直近5走平均着順"], errors="coerce").fillna(5.0)) / 5.5,\n        0.0, 1.0\n    )\n    # Ver8.3: 内枠は総合能力そのものではなく補助的な戦略価値として扱う。\n    # Ver8.2より加算幅を約55%へ縮小し、元から強い最内車の過剰評価を防ぐ。\n    df["内枠展開点"] = np.where(\n        front_mask,\n        settings["位置取り点"] * 0.55 * (\n            lane_value * 0.46 + hold_value * 0.34 + recent_value * 0.20\n        ),\n        0.0,\n    )\n\n    # Ver8.3: 勝ち切り力は表示用の参考指標。総合点には直接加算しない。\n    # 最内だけでなく、逃げ、連対力、近況の裏付けがある場合のみ高くなる。\n    escape_value = pd.to_numeric(df["逃げ成功率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    quinella_value = pd.to_numeric(df["連対率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    st_value = pd.to_numeric(df["同ハンデST優位度"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    craft_value = np.clip(\n        0.50 + pd.to_numeric(df["レース巧者指数"], errors="coerce").fillna(0.0) * 2.0,\n        0.0, 1.0,\n    )\n    df["最内勝ち切り点"] = np.where(\n        front_mask,\n        settings["位置取り点"] * 0.45 * (\n            lane_value * 0.18\n            + escape_value * 0.25\n            + quinella_value * 0.20\n            + recent_value * 0.15\n            + st_value * 0.12\n            + craft_value * 0.10\n        ),\n        0.0,\n    )\n\n    df["着順点"] = smooth_points(\n        df["着順指数"].tolist(),\n        settings["着順指数点"],\n        lower_is_better=False,\n    )\n    df["勝率点"] = smooth_points(\n        df["勝率"].tolist(),\n        settings["勝率点"],\n        lower_is_better=False,\n    )\n    df["連対点"] = smooth_points(\n        df["連対率"].tolist(),\n        settings["連対率点"],\n        lower_is_better=False,\n    )\n    df["上昇点"] = smooth_points(\n        df["上昇度"].tolist(),\n        settings["上昇度点"],\n        lower_is_better=False,\n    )\n    df["再現性点"] = smooth_points(\n        df["再現性"].tolist(),\n        settings["再現性点"],\n        lower_is_better=False,\n    )\n    df["ハンデ勝率点"] = smooth_points(\n        df["今回ハンデ勝率"].tolist(),\n        settings["今回ハンデ勝率点"],\n        lower_is_better=False,\n        zero_is_zero=True,\n    )\n\n    df["3着以内率点"] = smooth_points(\n        df["3着以内率"].tolist(),\n        settings["3着以内率点"],\n        lower_is_better=False,\n    )\n    df["平均着順点"] = smooth_points(\n        df["平均着順"].tolist(),\n        settings["平均着順点"],\n        lower_is_better=True,\n    )\n    df["着順安定度点"] = smooth_points(\n        df["着順安定度"].tolist(),\n        settings["着順安定度点"],\n        lower_is_better=False,\n    )\n\n    def parse_rank_strength(rank):\n        """\n        S1 / S-1 / A103 / B-204 などを数値化する。\n        S級 > A級 > B級を維持し、同じ級では数字が小さいほど高評価。\n        """\n        value = str(rank).strip().upper().replace("－", "-")\n        match = re.fullmatch(r"([SAB])\\s*-?\\s*(\\d{1,3})", value)\n        if not match:\n            return np.nan\n\n        grade = match.group(1)\n        rank_no = int(match.group(2))\n        if rank_no < 1:\n            return np.nan\n\n        # 1位を1.0、200位をほぼ0.0として級内評価。\n        # 201以上も解析し、0未満にはしない。\n        within_grade = max(0.0, 1.0 - (rank_no - 1) / 200.0)\n        grade_base = {"S": 2.0, "A": 1.0, "B": 0.0}[grade]\n        return grade_base + within_grade\n\n    rank_values = [\n        parse_rank_strength(rank)\n        for rank in df["現ランク"]\n    ]\n\n    judge_points = smooth_points(\n        df["審査P"].tolist(),\n        settings["審査Pランク点"] * 0.50,\n        lower_is_better=False,\n    )\n    rank_points = smooth_points(\n        rank_values,\n        settings["審査Pランク点"] * 0.50,\n        lower_is_better=False,\n    )\n\n    df["審査Pランク点"] = [\n        (\n            jp if not pd.isna(jp)\n            else settings["審査Pランク点"] * 0.50 * 0.5\n        )\n        + (\n            rp if not pd.isna(rp)\n            else settings["審査Pランク点"] * 0.50 * 0.5\n        )\n        for jp, rp in zip(judge_points, rank_points)\n    ]\n\n\n    # Ver9.0: 「走れる能力」と「1着まで取り切る力」を分離する。\n    # 3着以内が多くても1着が少ない選手は、総合能力を大きく落とさず、\n    # シミュレーション上の1着昇格だけを抑える。\n    win_rate_v = pd.to_numeric(df["勝率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    quinella_v = pd.to_numeric(df["連対率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    top3_v = pd.to_numeric(df["3着以内率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    recent_avg_v = pd.to_numeric(df["直近5走平均着順"], errors="coerce").fillna(5.0)\n    craft_v = np.clip(\n        0.50 + pd.to_numeric(df["レース巧者指数"], errors="coerce").fillna(0.0) * 2.0,\n        0.0, 1.0,\n    )\n    win_given_top3 = (win_rate_v / (top3_v + 0.08)).clip(0.0, 1.0)\n    win_given_top2 = (win_rate_v / (quinella_v + 0.06)).clip(0.0, 1.0)\n    recent_win_shape = np.clip((4.8 - recent_avg_v) / 3.8, 0.0, 1.0)\n    df["勝ち切り指数"] = np.clip(\n        win_given_top3 * 0.42\n        + win_given_top2 * 0.26\n        + win_rate_v * 0.18\n        + recent_win_shape * 0.08\n        + craft_v * 0.06,\n        0.0, 1.0,\n    )\n    # 連下力は、能力は高いが勝ち切れないタイプを2～3着候補として残す指標。\n    df["連下安定指数"] = np.clip(\n        top3_v * 0.52 + quinella_v * 0.28 + craft_v * 0.20,\n        0.0, 1.0,\n    )\n\n    # Ver12.2: 勝ち切り率とは別に、上位へ安定して残る再現性を評価する。\n    # 車番や今回の結果は使わず、長期・直近成績、凡走の少なさ、着順安定度で構成する。\n    finish_stability_v = pd.to_numeric(\n        df.get("着順安定度", pd.Series(0.45, index=df.index)), errors="coerce"\n    ).fillna(0.45).clip(0.0, 1.0)\n    recent_top3_stable = pd.to_numeric(\n        df.get("直近5走生3着内率", pd.Series(0.0, index=df.index)), errors="coerce"\n    ).fillna(0.0).clip(0.0, 1.0)\n    recent_poor_stable = pd.to_numeric(\n        df.get("直近5走生凡走率", pd.Series(0.0, index=df.index)), errors="coerce"\n    ).fillna(0.0).clip(0.0, 1.0)\n    df["安定上位指数"] = np.clip(\n        top3_v * 0.30\n        + quinella_v * 0.16\n        + recent_top3_stable * 0.23\n        + (1.0 - recent_poor_stable) * 0.13\n        + finish_stability_v * 0.12\n        + craft_v * 0.06,\n        0.0, 1.0,\n    )\n\n    # Ver10.0: 能力評価を4層へ再設計する。\n    # 1) 基礎スピード: 今回試走・ST・格・適性\n    # 2) 実戦能力: 近況の着順構成・安定性・速度を結果へ変える力\n    # 3) 勝負強さ: 上位進出時に1着まで取り切る力\n    # 4) 展開適性: 逃げ・差し・混戦・位置維持への対応力\n    # 車番自体は能力点へ使わず、最内の逃げ残りはシミュレーション側で扱う。\n\n    # ---------- 直近内容 ----------\n    recent_top3 = pd.to_numeric(df["直近5走生3着内率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    recent_poor = pd.to_numeric(df["直近5走生凡走率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    recent4_top3 = pd.to_numeric(df["直近4走生3着内率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    recent2_poor = pd.to_numeric(df["直近2走生凡走率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    recent3_top3 = pd.to_numeric(df["直近3走生3着内率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n\n    craft_reliability = np.clip(\n        0.50 + pd.to_numeric(df["レース巧者指数"], errors="coerce").fillna(0.0) * 2.0,\n        0.0, 1.0,\n    )\n    df["近況信頼度"] = np.clip(\n        pd.to_numeric(df["試走信頼度"], errors="coerce").fillna(0.50) * 0.58\n        + craft_reliability * 0.42,\n        0.0, 1.0,\n    )\n\n    raw_recent_content = (\n        (recent_top3 - 0.35) * 13.0\n        - recent_poor * 10.0\n        + (recent4_top3 - 0.50) * 6.5\n        - recent2_poor * 8.0\n        + np.maximum(0.0, recent3_top3 - 1.0 / 3.0) * 3.5\n    )\n    positive_recent = np.maximum(raw_recent_content, 0.0)\n    negative_recent = np.minimum(raw_recent_content, 0.0)\n    reliability_gate = np.clip(0.72 + 0.38 * df["近況信頼度"], 0.72, 1.0)\n    low_reliability_penalty = np.maximum(0.0, 0.48 - df["近況信頼度"]) * 7.0\n    sustainable_top3 = 0.40 + 0.55 * df["近況信頼度"]\n    df["短期上振れ抑制"] = np.clip(\n        np.maximum(0.0, recent_top3 - sustainable_top3) * 4.0,\n        0.0, 2.0,\n    )\n    df["近況信頼度補正"] = (\n        positive_recent * (reliability_gate - 1.0)\n        - low_reliability_penalty\n        - df["短期上振れ抑制"]\n    )\n    df["直近内容補正"] = np.clip(\n        negative_recent\n        + positive_recent * reliability_gate\n        - low_reliability_penalty\n        - df["短期上振れ抑制"],\n        -10.0, 8.0,\n    )\n\n    # ---------- 1. 基礎スピード層 ----------\n    # 試走だけで順位を作らず、ST・格・ハンデ/場/走路適性と組み合わせる。\n    df["基礎スピード点"] = (\n        pd.to_numeric(df["試走点"], errors="coerce").fillna(0.0) * 0.88\n        + pd.to_numeric(df["試走単発救済点"], errors="coerce").fillna(0.0) * 0.55\n        + pd.to_numeric(df["試走再現点"], errors="coerce").fillna(0.0) * 0.65\n        + pd.to_numeric(df["ST点"], errors="coerce").fillna(0.0) * 0.78\n        + pd.to_numeric(df["ST安定点"], errors="coerce").fillna(0.0) * 0.62\n        + pd.to_numeric(df["審査Pランク点"], errors="coerce").fillna(0.0) * 0.72\n        + pd.to_numeric(df["ハンデ点"], errors="coerce").fillna(0.0) * 0.65\n        + pd.to_numeric(df["開催場点"], errors="coerce").fillna(0.0) * 0.55\n        + pd.to_numeric(df["走路点"], errors="coerce").fillna(0.0) * 0.55\n    )\n\n    # ---------- 2. 実戦能力層 ----------\n    # 相関の強い着順系を丸ごと足さず、長期実績・直近内容・安定性へ整理する。\n    long_result = (\n        pd.to_numeric(df["着順点"], errors="coerce").fillna(0.0) * 0.72\n        + pd.to_numeric(df["3着以内率点"], errors="coerce").fillna(0.0) * 0.92\n        + pd.to_numeric(df["平均着順点"], errors="coerce").fillna(0.0) * 0.82\n        + pd.to_numeric(df["着順安定度点"], errors="coerce").fillna(0.0) * 0.95\n        + pd.to_numeric(df["レース巧者点"], errors="coerce").fillna(0.0) * 0.85\n        + pd.to_numeric(df["再現性点"], errors="coerce").fillna(0.0) * 0.65\n    )\n    speed_core = (\n        pd.to_numeric(df["試走点"], errors="coerce").fillna(0.0)\n        + pd.to_numeric(df["ST点"], errors="coerce").fillna(0.0)\n    )\n    conversion = np.clip(\n        recent_top3 * 0.58\n        + (1.0 - recent_poor) * 0.27\n        + craft_reliability * 0.15,\n        0.0, 1.0,\n    )\n    df["速度実戦変換補正"] = np.clip(\n        speed_core * (conversion - 0.52) * 0.68,\n        -4.5, 4.0,\n    )\n    df["実戦能力点"] = (\n        long_result\n        + pd.to_numeric(df["最近点"], errors="coerce").fillna(0.0) * 0.52\n        + pd.to_numeric(df["近況主役点"], errors="coerce").fillna(0.0) * 0.42\n        + df["直近内容補正"]\n        + df["速度実戦変換補正"]\n    )\n\n    # ---------- 3. 勝負強さ層 ----------\n    # ここは能力順位への寄与を限定し、主にシミュレーションの1着昇格へ使う。\n    df["勝負強さ点"] = (\n        pd.to_numeric(df["勝率点"], errors="coerce").fillna(0.0) * 0.82\n        + pd.to_numeric(df["連対点"], errors="coerce").fillna(0.0) * 0.48\n        + pd.to_numeric(df["上昇点"], errors="coerce").fillna(0.0) * 0.60\n        + pd.to_numeric(df["勝ち切り指数"], errors="coerce").fillna(0.35) * 5.0\n    )\n\n    # ---------- 4. 展開適性層 ----------\n    # 車番位置の直接点は含めない。位置維持・混戦対応・ハンデ実績のみを評価。\n    df["展開適性点"] = (\n        craft_reliability * 4.0\n        + pd.to_numeric(df["ST安定性"], errors="coerce").fillna(0.50).clip(0.0, 1.0) * 2.0\n        + pd.to_numeric(df["ハンデ勝率点"], errors="coerce").fillna(0.0) * 0.35\n        + pd.to_numeric(df["連下安定指数"], errors="coerce").fillna(0.35) * 2.5\n    )\n\n    # 選手タイプを履歴から自動分類。表示とシミュレーション補助に使用する。\n    front_trait = np.clip(\n        recent_top3 * 0.35\n        + pd.to_numeric(df["ST安定性"], errors="coerce").fillna(0.5) * 0.30\n        + pd.to_numeric(df["連下安定指数"], errors="coerce").fillna(0.35) * 0.35,\n        0.0, 1.0,\n    )\n    chase_trait = np.clip(\n        pd.to_numeric(df["勝ち切り指数"], errors="coerce").fillna(0.35) * 0.42\n        + craft_reliability * 0.33\n        + conversion * 0.25,\n        0.0, 1.0,\n    )\n    volatile_trait = np.clip(recent_poor * 0.60 + recent2_poor * 0.40, 0.0, 1.0)\n    df["選手タイプ"] = np.select(\n        [\n            volatile_trait >= 0.55,\n            (front_trait >= 0.67) & (pd.to_numeric(df["勝ち切り指数"], errors="coerce").fillna(0.35) < 0.48),\n            chase_trait >= 0.62,\n            front_trait >= 0.58,\n        ],\n        ["ムラ型", "安定連下型", "勝負型", "安定型"],\n        default="標準型",\n    )\n\n    # 4層の最終能力点。実戦能力を最大、基礎スピードを次点とする。\n    # 勝負強さは能力順位を独占しないよう15%、展開適性は10%に制限する。\n    df["改良総合点"] = (\n        df["基礎スピード点"] * 0.35\n        + df["実戦能力点"] * 0.45\n        + df["勝負強さ点"] * 0.10\n        + df["展開適性点"] * 0.10\n        + pd.to_numeric(df["ハンデ変化点"], errors="coerce").fillna(0.0) * 0.30\n    )\n\n    # 10m以上の同ハンデ帯で、人数が多い外枠を補助的に減点\n    (\n        df["同ハンデ人数"],\n        df["同ハンデ外順"],\n    ) = (np.nan, np.nan)\n    outer_penalty, line_count, outer_order = outer_lane_penalty(\n        df, max_penalty=4.5\n    )\n    df["同ハンデ人数"] = line_count\n    df["同ハンデ外順"] = outer_order\n    df["外枠不利補正"] = outer_penalty\n\n    df["改善後総合点"] = (\n        df["改良総合点"]\n        + df["ハンデ変化点"] * 0.70\n        + df["外枠不利補正"]\n    )\n\n    # Ver10.2: 実戦評価の裏付けを確認する。\n    # 過去の実戦成績が、今回の試走・ST・格などの基礎速度を大きく上回る場合は、\n    # 一時的な上振れとして能力点だけを穏やかに割り引く。展開確率は維持する。\n    practical_overhang = np.maximum(\n        0.0,\n        pd.to_numeric(df["実戦能力点"], errors="coerce").fillna(0.0)\n        - pd.to_numeric(df["基礎スピード点"], errors="coerce").fillna(0.0) * 1.05,\n    )\n    df["実戦裏付け不足補正"] = -np.clip(practical_overhang, 0.0, 5.0)\n\n    # 0m内枠は車番そのものではなく、位置活用率・試走信頼度・長期連対率が\n    # そろったときだけ「今回も実戦力を再現しやすい」と評価する。\n    handicap_v = pd.to_numeric(df["ハンデ"], errors="coerce").fillna(0.0)\n    position_v = pd.to_numeric(df["位置活用率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    trust_v = pd.to_numeric(df["試走信頼度"], errors="coerce").fillna(0.45).clip(0.0, 1.0)\n    quinella_support = pd.to_numeric(df["連対率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    df["内枠実戦再現補正"] = np.where(\n        handicap_v <= 0.0,\n        position_v * (0.45 + trust_v) * (0.70 + quinella_support) * 6.0,\n        0.0,\n    )\n\n    # 内枠でも試走信頼度が低い場合は、前残り期待を能力順位へ過剰転写しない。\n    df["内枠信頼不足補正"] = np.where(\n        handicap_v <= 0.0,\n        -np.maximum(0.0, 0.50 - trust_v) * 17.0,\n        0.0,\n    )\n\n    # 直近5走全体より直近4走が改善し、直近3走も上位を保つ選手を小幅加点。\n    # 単なる高水準ではなく「新しい方へ向けて良化」した場合のみ働く。\n    recent5_v = pd.to_numeric(df["直近5走生3着内率"], errors="coerce").fillna(0.0)\n    recent4_v = pd.to_numeric(df["直近4走生3着内率"], errors="coerce").fillna(0.0)\n    recent3_v = pd.to_numeric(df["直近3走生3着内率"], errors="coerce").fillna(0.0)\n    df["継続上昇補正"] = np.clip(\n        np.maximum(0.0, recent4_v - recent5_v) * 7.0\n        + np.maximum(0.0, recent3_v - 0.50) * 1.2,\n        0.0, 1.5,\n    ) * np.clip(0.70 + trust_v * 0.45, 0.70, 1.15)\n\n    # 最内車は、試走信頼度が十分高い場合に限り、前で自分のペースを作る再現性を加点。\n    # 信頼度が低い最内車には働かないため、単純な1番車優遇にはしない。\n    car_v = pd.to_numeric(df["車"], errors="coerce").fillna(99)\n    df["最内高信頼再現補正"] = np.where(\n        (car_v == 1) & (handicap_v <= 0.0),\n        np.clip(np.maximum(0.0, trust_v - 0.50) * 50.0, 0.0, 2.6),\n        0.0,\n    )\n\n    df["評価整合補正"] = (\n        df["実戦裏付け不足補正"]\n        + df["内枠実戦再現補正"]\n        + df["内枠信頼不足補正"]\n        + df["継続上昇補正"]\n        + df["最内高信頼再現補正"]\n    )\n    df["改善後総合点"] = df["改善後総合点"] + df["評価整合補正"]\n\n    # Ver10.2: 選手タイプを3方向で補足する。車番固定ではなく、\n    # レース内の相対値から「巻き返し余地」「再現力」「近況単独の過大評価」を判定する。\n    def _race_z(series):\n        v = pd.to_numeric(series, errors="coerce")\n        v = v.fillna(v.mean())\n        sd = float(v.std(ddof=0))\n        if not np.isfinite(sd) or sd < 1e-9:\n            return pd.Series(0.0, index=v.index)\n        return ((v - float(v.mean())) / sd).clip(-2.0, 2.0)\n\n    base_z = _race_z(df["基礎スピード点"])\n    avg_finish_z = _race_z(df["平均着順"])\n    trust_z = _race_z(df["試走信頼度"])\n    craft_z = _race_z(df["レース巧者指数"])\n    long_top3_z = _race_z(df["3着以内率"])\n    recent_top3_z = _race_z(df["直近5走生3着内率"])\n    practical_z = _race_z(df["実戦能力点"])\n    win_z = _race_z(df["勝ち切り指数"])\n\n    df["巻き返し余地補正"] = np.clip(\n        np.maximum(0.0, base_z) * np.maximum(0.0, avg_finish_z) * 2.0\n        + np.maximum(0.0, base_z) * 0.55,\n        0.0, 3.5,\n    )\n    df["実戦再現力補正"] = np.clip(\n        np.maximum(0.0, trust_z) * 0.75\n        + np.maximum(0.0, craft_z) * 1.05\n        + np.maximum(0.0, long_top3_z) * 0.45,\n        0.0, 2.4,\n    )\n    recent_top3_raw = pd.to_numeric(df["直近5走生3着内率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    df["勝負余力補正"] = np.clip(\n        np.maximum(0.0, win_z) * np.maximum(0.0, 0.75 - recent_top3_raw) * 2.8,\n        0.0, 1.8,\n    )\n    df["内側高信頼補正"] = np.where(\n        handicap_v <= 0.0,\n        np.clip(np.maximum(0.0, trust_v - 0.50) * position_v * 10.0, 0.0, 1.2),\n        0.0,\n    )\n    unsupported = (\n        np.maximum(0.0, practical_z) * np.maximum(0.0, -base_z) * 1.50\n        + np.maximum(0.0, recent_top3_z) * np.maximum(0.0, -craft_z) * 1.20\n        + np.maximum(0.0, recent_top3_z) * np.maximum(0.0, -trust_z) * 0.70\n    )\n    perfect_surge = np.maximum(0.0, recent_top3_z - 1.0) * (1.10 + np.maximum(0.0, -craft_z))\n    df["近況単独過大補正"] = -np.clip(unsupported + perfect_surge, 0.0, 4.5)\n    df["選手タイプ総合補正"] = (\n        df["巻き返し余地補正"]\n        + df["実戦再現力補正"]\n        + df["勝負余力補正"]\n        + df["内側高信頼補正"]\n        + df["近況単独過大補正"]\n    )\n    df["改善後総合点"] = df["改善後総合点"] + df["選手タイプ総合補正"]\n\n    # Ver10.5: 3着以内と凡走の間にある4～5着の内容を評価する。\n    # 3着以内率だけの段差で、競走内容をまとめている選手が沈みすぎるのを防ぐ。\n    # 車番は使用せず、直近5走の「6着以下ではないが3着以内でもない」割合から算出。\n    recent_mid_hold = np.clip(\n        1.0\n        - pd.to_numeric(df["直近5走生3着内率"], errors="coerce").fillna(0.0)\n        - pd.to_numeric(df["直近5走生凡走率"], errors="coerce").fillna(0.0),\n        0.0, 1.0,\n    )\n    trust_for_hold = pd.to_numeric(\n        df["試走信頼度"], errors="coerce"\n    ).fillna(0.45).clip(0.0, 1.0)\n    craft_for_hold = np.clip(\n        0.50 + pd.to_numeric(df["レース巧者指数"], errors="coerce").fillna(0.0) * 2.0,\n        0.0, 1.0,\n    )\n    df["中位粘り率"] = recent_mid_hold\n    df["中位粘り補正"] = np.clip(\n        recent_mid_hold\n        * (0.75 + trust_for_hold * 0.45 + craft_for_hold * 0.35)\n        * 2.6,\n        0.0, 2.8,\n    )\n\n    # 重いハンデから軽くなっただけで無条件加点せず、\n    # 試走信頼度と中位粘りが伴う場合に限り条件改善を追加評価する。\n    handicap_change_v = pd.to_numeric(\n        df["ハンデ変化点"], errors="coerce"\n    ).fillna(0.0).clip(lower=0.0)\n    df["条件改善再現補正"] = np.clip(\n        handicap_change_v\n        * (0.30 + trust_for_hold * 0.30 + recent_mid_hold * 0.35),\n        0.0, 1.6,\n    )\n\n    df["中位内容総合補正"] = (\n        df["中位粘り補正"] + df["条件改善再現補正"]\n    )\n    df["改善後総合点"] = df["改善後総合点"] + df["中位内容総合補正"]\n\n    # Ver10.6: 新しい3指標を能力評価へ追加。\n    # 全て車番非依存で、レース内相対値と履歴の流れから算出する。\n    curve_v = pd.to_numeric(df.get("上昇カーブ指数", 0.0), errors="coerce").fillna(0.0).clip(-1.0, 1.0)\n    kick_v = pd.to_numeric(df.get("終盤指数", 0.0), errors="coerce").fillna(0.0).clip(-1.0, 1.0)\n\n    df["上昇カーブ補正"] = np.clip(curve_v * 3.2, -2.0, 3.2)\n    df["終盤力補正"] = np.clip(kick_v * 3.0, -1.8, 3.0)\n\n    # 履歴に対戦相手ランクが無いため、現在の格に対して長期実戦成績が上回る選手を\n    # 「格以上に走れる＝相手耐性あり」と推定する。単純な低ランク救済にはしない。\n    rank_base_z = _race_z(df["審査Pランク点"])\n    durable_z = _race_z(\n        pd.to_numeric(df["3着以内率"], errors="coerce").fillna(0.0) * 0.55\n        + pd.to_numeric(df["着順安定度"], errors="coerce").fillna(0.0) * 0.25\n        + pd.to_numeric(df["レース巧者指数"], errors="coerce").fillna(0.0) * 0.20\n    )\n    recent_poor_gate = 1.0 - pd.to_numeric(\n        df["直近5走生凡走率"], errors="coerce"\n    ).fillna(0.0).clip(0.0, 1.0)\n    under_ranked = np.maximum(0.0, -rank_base_z)\n    df["相手レベル耐性補正"] = np.clip(\n        np.maximum(0.0, durable_z)\n        * (0.35 + under_ranked * 0.85)\n        * recent_poor_gate\n        * 1.45\n        + np.maximum(0.0, kick_v) * 0.45,\n        0.0, 2.2,\n    )\n\n    df["Ver10_6総合補正"] = (\n        df["上昇カーブ補正"]\n        + df["終盤力補正"]\n        + df["相手レベル耐性補正"]\n    )\n    df["改善後総合点"] = df["改善後総合点"] + df["Ver10_6総合補正"]\n\n\n    # Ver10.7: 能力順位とは別に「1着まで届く上振れ余地」を評価する。\n    # ① 前線最内で逃げの形を作れるタイプ\n    # ② 格・ST・勝ち切り力があり、近況不振でも一発の天井が残るタイプ\n    # の2経路を車番非依存で算出する。\n    handicap_now = pd.to_numeric(df["ハンデ"], errors="coerce").fillna(0.0)\n    front_h = float(handicap_now.min())\n    front_gate = (handicap_now == front_h).astype(float)\n    inner_degree = pd.to_numeric(df.get("前線内側度", 0.0), errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    escape_v = pd.to_numeric(df.get("逃げ成功率", 0.0), errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    hold_v = pd.to_numeric(df.get("内枠残存率", 0.0), errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    close_v = pd.to_numeric(df.get("勝ち切り指数", 0.0), errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    st_point_z = _race_z(pd.to_numeric(df["ST点"], errors="coerce").fillna(0.0))\n    rank_point_z = _race_z(pd.to_numeric(df["審査Pランク点"], errors="coerce").fillna(0.0))\n    trial_point_z = _race_z(pd.to_numeric(df["試走点"], errors="coerce").fillna(0.0))\n    recent_poor_v = pd.to_numeric(df["直近5走生凡走率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n\n    df["逃げ切り上振れ指数"] = np.clip(\n        front_gate\n        * inner_degree\n        * (0.34 * escape_v + 0.28 * hold_v + 0.18 * close_v\n           + 0.10 * np.maximum(0.0, trial_point_z)\n           + 0.10 * np.maximum(0.0, -st_point_z)),\n        0.0, 1.0,\n    )\n\n    # 近況不振は通常評価では減点するが、格・ST・勝率が揃う場合の\n    # 「一発の天井」までゼロにはしない。連下安定とは分離する。\n    raw_chase_ceiling = (\n        0.30 * np.maximum(0.0, rank_point_z)\n        + 0.24 * np.maximum(0.0, st_point_z)\n        + 0.30 * close_v\n        + 0.16 * np.maximum(0.0, trial_point_z)\n    )\n    # 格やSTだけでなく、最低限の勝ち切り実績を必要条件にする。\n    # これにより「速いが勝ち筋が薄い」選手の過大評価を防ぐ。\n    df["追上げ勝負余地指数"] = np.clip(\n        raw_chase_ceiling\n        * (0.48 + close_v * 1.55)\n        * (0.82 + recent_poor_v * 0.18),\n        0.0, 1.0,\n    )\n\n    df["勝ち上振れ余地指数"] = np.maximum(\n        df["逃げ切り上振れ指数"],\n        df["追上げ勝負余地指数"],\n    )\n    # 能力点への反映は小さく留め、主効果はシミュレーションの1着分岐に置く。\n    df["勝ち上振れ能力補正"] = np.clip(\n        df["勝ち上振れ余地指数"] * 1.55,\n        0.0, 1.55,\n    )\n    df["改善後総合点"] = df["改善後総合点"] + df["勝ち上振れ能力補正"]\n\n    # 格・ST・勝ち切りの3要素が偏らず揃う選手を「一発勝負型」として小幅評価。\n    # どれか1項目だけ突出した選手は上がりすぎないよう、3要素の最小値寄りで判定する。\n    rank_pct = pd.to_numeric(df["審査Pランク点"], errors="coerce").rank(pct=True)\n    st_pct = pd.to_numeric(df["ST点"], errors="coerce").rank(pct=True)\n    close_pct = close_v.rank(pct=True)\n    balanced_ceiling = (\n        np.minimum(np.minimum(rank_pct, st_pct), close_pct) * 0.65\n        + (rank_pct * st_pct * close_pct) ** (1.0 / 3.0) * 0.35\n    )\n    df["一発勝負バランス補正"] = np.clip(\n        np.maximum(0.0, balanced_ceiling - 0.48) * 2.2,\n        0.0, 1.15,\n    )\n    df["改善後総合点"] = df["改善後総合点"] + df["一発勝負バランス補正"]\n\n\n    # Ver11.3: 当日状態指数。今回試走の相対順位を中心に、\n    # 直近状態・上昇カーブ・ST・試走信頼度が伴う場合だけ強く評価する。\n    trial_raw = pd.to_numeric(df["試走換算"], errors="coerce")\n    trial_pct = trial_raw.rank(pct=True, ascending=False, method="average").fillna(0.50)\n    st_pct_today = pd.to_numeric(df["平均ST"], errors="coerce").rank(\n        pct=True, ascending=False, method="average"\n    ).fillna(0.50)\n    trust_today = pd.to_numeric(df["試走信頼度"], errors="coerce").fillna(0.50).clip(0.20, 1.00)\n    recent3_today = pd.to_numeric(df["直近3走生3着内率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    recent5_today = pd.to_numeric(df["直近5走生3着内率"], errors="coerce").fillna(0.0).clip(0.0, 1.0)\n    curve_today = pd.to_numeric(df.get("上昇カーブ指数", 0.0), errors="coerce").fillna(0.0).clip(-1.0, 1.0)\n    craft_today = pd.to_numeric(df.get("レース巧者指数", 0.0), errors="coerce").fillna(0.0)\n    craft_today = np.clip(0.50 + craft_today * 1.5, 0.0, 1.0)\n    recent_today = np.clip(recent3_today * 0.58 + recent5_today * 0.42, 0.0, 1.0)\n    trend_today = np.clip(0.50 + curve_today * 0.50, 0.0, 1.0)\n    support_gate = np.clip(\n        0.30 + trust_today * 0.28 + recent_today * 0.20\n        + trend_today * 0.12 + craft_today * 0.10, 0.35, 1.00\n    )\n    own_trial_change = pd.to_numeric(\n        df.get("自己試走変化秒", pd.Series(0.0, index=df.index)), errors="coerce"\n    ).fillna(0.0)\n    own_trial_sd = pd.to_numeric(\n        df.get("直近試走標準偏差", pd.Series(0.035, index=df.index)), errors="coerce"\n    ).fillna(0.035).clip(0.018, 0.080)\n    # 良化0.04秒前後で強評価、悪化0.04秒前後で弱評価。\n    # 標準偏差が大きい選手は、単発の変化を少し割り引く。\n    own_scale = np.maximum(0.040, own_trial_sd * 1.35)\n    df["自己比試走指数"] = np.clip(\n        0.50 + own_trial_change / (own_scale * 2.0), 0.0, 1.0\n    )\n    df["当日状態指数"] = np.clip(\n        (trial_pct * 0.39 + df["自己比試走指数"] * 0.22\n         + st_pct_today * 0.12 + recent_today * 0.15\n         + trend_today * 0.08 + craft_today * 0.04) * support_gate, 0.0, 1.0\n    )\n    state_center = float(df["当日状態指数"].median())\n    df["当日状態補正"] = np.clip(\n        np.maximum(0.0, df["当日状態指数"] - state_center) * 8.5, 0.0, 3.4\n    )\n    df["改善後総合点"] = df["改善後総合点"] + df["当日状態補正"]\n\n    # Ver12.2: 爆発指数。\n    # 単なる試走1位ではなく、2位との差が大きい「突出試走」を今日だけの勝ち切り上振れとして扱う。\n    trial_values = trial_raw.to_numpy(float)\n    valid_trial = trial_values[np.isfinite(trial_values)]\n    if len(valid_trial) >= 2:\n        sorted_trial = np.sort(valid_trial)\n        best_trial = float(sorted_trial[0])\n        second_trial = float(sorted_trial[1])\n    elif len(valid_trial) == 1:\n        best_trial = second_trial = float(valid_trial[0])\n    else:\n        best_trial = second_trial = np.nan\n\n    # 0.05秒差でほぼ最大。トップ以外は突出度を持たない。\n    best_mask = np.isfinite(trial_values) & np.isclose(trial_values, best_trial, atol=1e-9)\n    top_margin = max(0.0, second_trial - best_trial) if np.isfinite(best_trial) else 0.0\n    margin_power = float(np.clip(top_margin / 0.05, 0.0, 1.0))\n    trial_dominance = np.zeros(len(df), dtype=float)\n    trial_dominance[best_mask] = margin_power\n    df["試走突出度"] = trial_dominance\n\n    # 今日の気配を主役にしつつ、信頼度・上昇傾向を補助条件にする。\n    # 履歴が弱くても突出試走そのものは残すが、無条件で能力トップにはしない。\n    explosion_support = np.clip(\n        0.58 + trust_today.to_numpy(float) * 0.20\n        + trend_today.to_numpy(float) * 0.12\n        + recent_today.to_numpy(float) * 0.10,\n        0.58, 1.00,\n    )\n    df["爆発指数"] = np.clip(\n        (trial_pct.to_numpy(float) * 0.24\n         + trial_dominance * 0.34\n         + df["自己比試走指数"].to_numpy(float) * 0.25\n         + df["当日状態指数"].to_numpy(float) * 0.17)\n        * explosion_support,\n        0.0, 1.0,\n    )\n    explosion_center = float(df["爆発指数"].median())\n    # 能力順位への加点は控えめ。主効果はシミュレーションの上振れ分岐。\n    df["爆発能力補正"] = np.clip(\n        np.maximum(0.0, df["爆発指数"] - explosion_center) * 2.2,\n        0.0, 1.10,\n    )\n    df["改善後総合点"] = df["改善後総合点"] + df["爆発能力補正"]\n\n    # Ver12.2: 当日レース評価。\n    # 過去能力の絶対点差が大きすぎると、突出試走を何点足しても届かないため、\n    # レース内百分位で「基礎力・当日試走・格・ST・ハンデ位置・上昇度」を再合成する。\n    # これにより過去能力を捨てず、その日の番組内での相対的な主役候補を評価する。\n    base_today_pct = pd.to_numeric(df["改善後総合点"], errors="coerce").rank(\n        pct=True, ascending=True, method="average"\n    ).fillna(0.50)\n    rank_today_pct = pd.to_numeric(df["審査Pランク点"], errors="coerce").rank(\n        pct=True, ascending=True, method="average"\n    ).fillna(0.50)\n    handicap_today_pct = pd.to_numeric(df["ハンデ"], errors="coerce").rank(\n        pct=True, ascending=True, method="average"\n    ).fillna(0.50)\n    trend_rank_pct = pd.to_numeric(df.get("上昇カーブ指数", 0.0), errors="coerce").rank(\n        pct=True, ascending=True, method="average"\n    ).fillna(0.50)\n\n    # Ver12.2: 前団主導指数。\n    # 0m逃げを無条件に上げず、最前線から10m後ろの選手が、今回試走・当日状態・ST・\n    # 試走信頼度を伴っている場合に「前団を先に攻略できる主役候補」として評価する。\n    handicap_num = pd.to_numeric(df["ハンデ"], errors="coerce").fillna(0.0)\n    min_handicap_value = float(handicap_num.min())\n    handicap_gap = handicap_num - min_handicap_value\n    front_band_gate = ((handicap_gap > 0.0) & (handicap_gap <= 10.0)).astype(float)\n    df["前団主導指数"] = np.clip(\n        (trial_pct * 0.36\n         + df["自己比試走指数"] * 0.18\n         + df["当日状態指数"] * 0.22\n         + st_pct_today * 0.12\n         + trust_today * 0.08\n         + trend_rank_pct * 0.04) * front_band_gate,\n        0.0, 1.0,\n    )\n\n    # 後方追走不成立リスク。\n    # 20m以上後方で、前団の最速試走より0.02秒を超えて劣る場合だけ発生。\n    # ランクそのものは下げず、今回の追走が成立しにくい条件として扱う。\n    front_trial_mask = handicap_gap <= 10.0\n    front_trials = trial_raw[front_trial_mask & trial_raw.notna()]\n    best_front_trial = float(front_trials.min()) if len(front_trials) else float(trial_raw.min())\n    trial_deficit = np.clip((trial_raw - best_front_trial - 0.02) / 0.04, 0.0, 1.0).fillna(0.0)\n    backline_gate = (handicap_gap >= 20.0).astype(float)\n    df["後方追走不成立リスク"] = np.clip(\n        trial_deficit\n        * backline_gate\n        * (0.78 - df["当日状態指数"] * 0.28)\n        * (1.00 - df["爆発指数"] * 0.45),\n        0.0, 0.85,\n    )\n\n    df["当日レース指数"] = np.clip(\n        trial_pct * 0.25\n        + df["自己比試走指数"] * 0.11\n        + rank_today_pct * 0.17\n        + base_today_pct * 0.11\n        + st_pct_today * 0.05\n        + handicap_today_pct * 0.15\n        + trend_rank_pct * 0.06\n        + df["前団主導指数"] * 0.14\n        - df["後方追走不成立リスク"] * 0.08,\n        0.0, 1.0,\n    )\n    # 爆発指数が高い選手は僅差の順位比較で優先する。過大な固定加点にはしない。\n    df["当日レース指数"] = np.clip(\n        df["当日レース指数"] + df["試走突出度"] * 0.08 + df["爆発指数"] * 0.04,\n        0.0, 1.0,\n    )\n    df["当日勝ち切り指数"] = np.clip(\n        df["自己比試走指数"] * 0.36\n        + df["前団主導指数"] * 0.30\n        + df["爆発指数"] * 0.18\n        + pd.to_numeric(df.get("勝ち切り指数", 0.0), errors="coerce").fillna(0.0).rank(pct=True) * 0.16,\n        0.0, 1.0,\n    )\n\n    # Ver12.2: 当日試走が横並びのレースでは、格・突破力だけの順位独占を抑え、\n    # 安定して2～4着へ残れる選手を評価へ戻す。\n    stable_top_pct = pd.to_numeric(df["安定上位指数"], errors="coerce").fillna(0.5).rank(pct=True)\n    recent_poor_gate_v = 1.0 - pd.to_numeric(\n        df.get("直近5走生凡走率", 0.0), errors="coerce"\n    ).fillna(0.0).clip(0.0, 1.0)\n    trial_spread = float(pd.to_numeric(df["試走換算"], errors="coerce").max() - pd.to_numeric(df["試走換算"], errors="coerce").min())\n    trial_tie_gate = float(np.clip((0.08 - trial_spread) / 0.06, 0.0, 1.0))\n    df["安定上位評価補正"] = np.clip(\n        (stable_top_pct - 0.50) * (0.10 + 0.10 * trial_tie_gate)\n        + (recent_poor_gate_v - 0.55) * 0.035,\n        -0.055, 0.125,\n    )\n    df["当日レース指数"] = np.clip(\n        df["当日レース指数"] + df["安定上位評価補正"], 0.0, 1.0\n    )\n\n    df["基礎履歴総合点"] = df["改善後総合点"]\n    df["改善後総合点"] = df["当日レース指数"] * 30.0\n\n    # 選手タイプVer2。既存タイプを残しつつ、強い特徴がある場合だけ上書きする。\n    df["選手タイプVer2"] = df["選手タイプ"].astype(str)\n    df.loc[(curve_v >= 0.35) & (kick_v >= 0.05), "選手タイプVer2"] = "上昇型"\n    df.loc[(kick_v >= 0.35) & (curve_v < 0.35), "選手タイプVer2"] = "終盤型"\n    df.loc[(kick_v >= 0.22) & (pd.to_numeric(df["連下安定指数"], errors="coerce").fillna(0.0) >= 0.52), "選手タイプVer2"] = "追走終盤型"\n    df.loc[(pd.to_numeric(df["内枠残存率"], errors="coerce").fillna(0.0) >= 0.58) & (kick_v < 0.20), "選手タイプVer2"] = "前残り型"\n\n    # Excel COUNTIF(">") + 1 と同じ競技順位\n    df["改良順位"] = [\n        1 + sum(other > value for other in df["改良総合点"])\n        for value in df["改良総合点"]\n    ]\n    df["改善後順位"] = [\n        1 + sum(other > value for other in df["改善後総合点"])\n        for value in df["改善後総合点"]\n    ]\n\n    return df\n\n\n\n\ndef prepare_simulation_arrays(df):\n    """Ver7.2: 近況好調を実戦展開へ強く反映する配列。"""\n    cars = df["車"].astype(int).to_numpy()\n    handicap = pd.to_numeric(df["ハンデ"], errors="coerce").fillna(0.0).to_numpy(float)\n    st = pd.to_numeric(df["平均ST"], errors="coerce").fillna(0.20).clip(0.06, 0.40).to_numpy(float)\n    st_std = pd.to_numeric(df.get("ST標準偏差", pd.Series(0.045, index=df.index)), errors="coerce").fillna(0.045).clip(0.015, 0.12).to_numpy(float)\n    outer_penalty = pd.to_numeric(df["外枠不利補正"], errors="coerce").fillna(0.0).abs().to_numpy(float)\n    line_count = pd.to_numeric(df["同ハンデ人数"], errors="coerce").fillna(1.0).to_numpy(float)\n    scores = df["改善後総合点"].to_numpy(float)\n    finish_sd = pd.to_numeric(df["着順標準偏差"], errors="coerce").fillna(1.5).clip(0.8, 3.0).to_numpy(float)\n\n    # Ver11.1: 走路状況が変わっても崩れにくい度合い。\n    # 着順の散らばり、ST安定性、試走再現性を組み合わせる。\n    finish_consistency = np.clip(1.0 - (finish_sd - 0.8) / 2.2, 0.0, 1.0)\n\n    trial_outlier_confidence = pd.to_numeric(df.get("試走単発外れ信頼度", pd.Series(0.0, index=df.index)), errors="coerce").fillna(0.0).clip(0.0, 1.0).to_numpy(float)\n    trial_relief_points = pd.to_numeric(df.get("試走単発救済点", pd.Series(0.0, index=df.index)), errors="coerce").fillna(0.0).to_numpy(float)\n    # Ver7.2: 試走が単発で悪かった好調選手は、本走で戻す分岐を強める。\n    form_upside_prob = np.clip(\n        trial_outlier_confidence\n        * np.clip(trial_relief_points / 2.6, 0.0, 1.0)\n        * 1.02,\n        0.0, 0.58,\n    )\n    form_upside_power = 0.34 + trial_outlier_confidence * 0.76\n\n    # 直近の好調度を、能力点とは別の「展開発現力」として作る。\n    recent3_bonus = pd.to_numeric(df.get("直近3走好調補正", pd.Series(0.0, index=df.index)), errors="coerce").fillna(0.0).clip(-3.0, 4.0).to_numpy(float)\n    recent5_bonus = pd.to_numeric(df.get("直近5走補正", pd.Series(0.0, index=df.index)), errors="coerce").fillna(0.0).clip(-3.0, 4.0).to_numpy(float)\n    top3_rate_flow = pd.to_numeric(df.get("3着以内率", pd.Series(0.35, index=df.index)), errors="coerce").fillna(0.35).clip(0.0, 1.0).to_numpy(float)\n    quinella_rate_flow = pd.to_numeric(df.get("連対率", pd.Series(0.20, index=df.index)), errors="coerce").fillna(0.20).clip(0.0, 1.0).to_numpy(float)\n    craft_flow = pd.to_numeric(df.get("レース巧者指数", pd.Series(0.0, index=df.index)), errors="coerce").fillna(0.0).to_numpy(float)\n    craft_scale = np.tanh(craft_flow / 0.30)\n\n    # Ver7.4: 試走だけ速く、実戦で着順へ変換しにくい選手を抑える。\n    # 車番固定ではなく、試走信頼度・レース巧者・ST安定性・外位置で判定する。\n    trial_trust_flow = pd.to_numeric(\n        df.get("試走信頼度", pd.Series(0.50, index=df.index)),\n        errors="coerce",\n    ).fillna(0.50).clip(0.20, 1.00).to_numpy(float)\n    st_stability_flow = pd.to_numeric(\n        df.get("ST安定性", pd.Series(0.50, index=df.index)),\n        errors="coerce",\n    ).fillna(0.50).clip(0.0, 1.0).to_numpy(float)\n    execution_quality = np.clip(\n        0.34 * trial_trust_flow\n        + 0.27 * ((craft_scale + 1.0) / 2.0)\n        + 0.22 * st_stability_flow\n        + 0.17 * top3_rate_flow,\n        0.0, 1.0,\n    )\n\n    road_stability = np.clip(\n        0.42 * finish_consistency\n        + 0.31 * st_stability_flow\n        + 0.17 * trial_trust_flow\n        + 0.10 * execution_quality,\n        0.0, 1.0,\n    )\n    volatility_profile = np.clip(1.0 - road_stability, 0.0, 1.0)\n\n    recent_form_strength = np.clip(\n        0.34 * top3_rate_flow\n        + 0.24 * quinella_rate_flow\n        + 0.18 * np.clip((recent3_bonus + 1.5) / 5.5, 0.0, 1.0)\n        + 0.14 * np.clip((recent5_bonus + 1.5) / 5.5, 0.0, 1.0)\n        + 0.10 * ((craft_scale + 1.0) / 2.0),\n        0.0, 1.0,\n    )\n\n    current_day_strength = pd.to_numeric(\n        df.get("当日状態指数", pd.Series(0.50, index=df.index)), errors="coerce"\n    ).fillna(0.50).clip(0.0, 1.0).to_numpy(float)\n    explosion_strength = pd.to_numeric(\n        df.get("爆発指数", pd.Series(0.0, index=df.index)), errors="coerce"\n    ).fillna(0.0).clip(0.0, 1.0).to_numpy(float)\n    trial_dominance_flow = pd.to_numeric(\n        df.get("試走突出度", pd.Series(0.0, index=df.index)), errors="coerce"\n    ).fillna(0.0).clip(0.0, 1.0).to_numpy(float)\n    front_command_flow = pd.to_numeric(\n        df.get("前団主導指数", pd.Series(0.0, index=df.index)), errors="coerce"\n    ).fillna(0.0).clip(0.0, 1.0).to_numpy(float)\n    pursuit_failure_risk = pd.to_numeric(\n        df.get("後方追走不成立リスク", pd.Series(0.0, index=df.index)), errors="coerce"\n    ).fillna(0.0).clip(0.0, 0.85).to_numpy(float)\n    own_trial_form = pd.to_numeric(\n        df.get("自己比試走指数", pd.Series(0.5, index=df.index)), errors="coerce"\n    ).fillna(0.5).clip(0.0, 1.0).to_numpy(float)\n    today_win_conversion = pd.to_numeric(\n        df.get("当日勝ち切り指数", pd.Series(0.5, index=df.index)), errors="coerce"\n    ).fillna(0.5).clip(0.0, 1.0).to_numpy(float)\n\n    score_std = np.std(scores, ddof=1)\n    ability_z = (scores - np.mean(scores)) / (score_std if score_std > 0 else 1.0)\n    form_std = np.std(recent_form_strength, ddof=1)\n    form_z = (recent_form_strength - np.mean(recent_form_strength)) / (form_std if form_std > 0 else 1.0)\n    # Ver7.4: 能力75%・近況展開25%。\n    # 通常スタート時は基礎能力をしっかり出し、展開失敗時だけ大きく落ちる二山型。\n    state_std = np.std(current_day_strength, ddof=1)\n    state_z = (current_day_strength - np.mean(current_day_strength)) / (state_std if state_std > 0 else 1.0)\n    z_scores = ability_z * 0.68 + form_z * 0.20 + state_z * 0.12\n\n    same_line_outer_ratio = np.zeros(len(df), dtype=float)\n    same_line_inner_ratio = np.zeros(len(df), dtype=float)\n    for _, group in df.groupby("ハンデ", dropna=False):\n        ordered = group.sort_values("車")\n        count = len(ordered)\n        if count <= 1:\n            continue\n        for order, idx in enumerate(ordered.index, 1):\n            pos = df.index.get_loc(idx)\n            ratio = (order - 1) / (count - 1)\n            same_line_outer_ratio[pos] = ratio\n            same_line_inner_ratio[pos] = 1.0 - ratio\n\n    # 外側で実戦変換力が低いほど、混戦で位置を失いやすい。\n    traffic_conversion = np.clip(\n        execution_quality\n        - same_line_outer_ratio * 0.20\n        + np.clip(craft_scale, -1.0, 1.0) * 0.10,\n        0.05, 0.95,\n    )\n\n\n    # Ver9.0: 能力とは別の勝ち切り・連下指標。\n    closing_base = pd.to_numeric(\n        df.get("勝ち切り指数", pd.Series(0.35, index=df.index)),\n        errors="coerce",\n    ).fillna(0.35).clip(0.0, 1.0)\n    final_kick_flow = pd.to_numeric(\n        df.get("終盤指数", pd.Series(0.0, index=df.index)),\n        errors="coerce",\n    ).fillna(0.0).clip(-1.0, 1.0)\n    win_upside_flow = pd.to_numeric(\n        df.get("勝ち上振れ余地指数", pd.Series(0.0, index=df.index)),\n        errors="coerce",\n    ).fillna(0.0).clip(0.0, 1.0).to_numpy(float)\n    closing_strength = np.clip(\n        closing_base.to_numpy(float)\n        + np.maximum(0.0, final_kick_flow.to_numpy(float)) * 0.13\n        + win_upside_flow * 0.20,\n        0.0, 1.0,\n    )\n    position_hold_flow = pd.to_numeric(\n        df.get("好位置維持指数", pd.Series(0.50, index=df.index)), errors="coerce"\n    ).fillna(0.50).clip(0.0, 1.0).to_numpy(float)\n    rear_chase_flow = pd.to_numeric(\n        df.get("後方追上げ指数", pd.Series(0.35, index=df.index)), errors="coerce"\n    ).fillna(0.35).clip(0.0, 1.0).to_numpy(float)\n    position_dependency_flow = pd.to_numeric(\n        df.get("位置依存指数", pd.Series(0.50, index=df.index)), errors="coerce"\n    ).fillna(0.50).clip(0.0, 1.0).to_numpy(float)\n    bipolar_flow = pd.to_numeric(\n        df.get("二極化指数", pd.Series(0.30, index=df.index)), errors="coerce"\n    ).fillna(0.30).clip(0.0, 1.0).to_numpy(float)\n\n    place_strength = pd.to_numeric(\n        df.get("連下安定指数", pd.Series(0.45, index=df.index)),\n        errors="coerce",\n    ).fillna(0.45).clip(0.0, 1.0).to_numpy(float)\n\n    zero_line_outer = (handicap == 0).astype(float) * same_line_outer_ratio * np.clip((line_count - 1.0) / 2.0, 0.0, 1.0)\n    outer_start_delay = outer_penalty * 0.060 + zero_line_outer * 0.065\n\n    st_speed_risk = np.clip((st - 0.14) / 0.16, 0.0, 1.0)\n    st_variation_risk = np.clip((st_std - 0.025) / 0.065, 0.0, 1.0)\n    outer_risk = np.clip(outer_penalty / 4.5 + zero_line_outer * 0.75, 0.0, 1.0)\n    crowd_risk = np.clip((line_count - 2.0) / 4.0, 0.0, 1.0)\n    conversion_risk = np.clip(1.0 - traffic_conversion, 0.0, 1.0)\n    normal_late_prob = np.clip(\n        0.040 + st_speed_risk * 0.066 + st_variation_risk * 0.098\n        + outer_risk * 0.076 + crowd_risk * 0.032\n        + conversion_risk * same_line_outer_ratio * 0.090,\n        0.030, 0.29,\n    )\n    severe_late_prob = np.clip(\n        0.009 + st_speed_risk * 0.021 + st_variation_risk * 0.042\n        + outer_risk * 0.032\n        + conversion_risk * same_line_outer_ratio * 0.045,\n        0.006, 0.12,\n    )\n\n    zero_inner_escape = (handicap == 0).astype(float) * same_line_inner_ratio * np.clip((line_count - 1.0) / 2.0, 0.0, 1.0)\n    escape_success_prob = pd.to_numeric(df.get("逃げ成功率", pd.Series(0.0, index=df.index)), errors="coerce").fillna(0.0).clip(0.0, 0.90).to_numpy(float)\n\n    top3_rate = pd.to_numeric(df.get("3着以内率", pd.Series(0.35, index=df.index)), errors="coerce").fillna(0.35).clip(0.0, 1.0).to_numpy(float)\n    quinella_rate = pd.to_numeric(df.get("連対率", pd.Series(0.20, index=df.index)), errors="coerce").fillna(0.20).clip(0.0, 1.0).to_numpy(float)\n    st_stability = pd.to_numeric(df.get("ST安定性", pd.Series(0.50, index=df.index)), errors="coerce").fillna(0.50).clip(0.0, 1.0).to_numpy(float)\n\n    # Ver7.4: 点数モデルで算出した内枠残存率をシミュレーションにも使用。\n    # 当日の混雑度とST安定性を少しだけ再加味する。\n    stored_inner_hold = pd.to_numeric(\n        df.get("内枠残存率", pd.Series(0.0, index=df.index)),\n        errors="coerce",\n    ).fillna(0.0).clip(0.0, 0.90).to_numpy(float)\n    inner_hold_prob = np.clip(\n        stored_inner_hold * 0.82\n        + zero_inner_escape * 0.08\n        + st_stability * 0.06\n        + top3_rate * 0.04,\n        0.08, 0.88,\n    )\n    front_handicap = float(np.min(handicap))\n    front_line = handicap == front_handicap\n    inner_hold_prob *= front_line.astype(float)\n\n    # Ver8.3: 勝ち切り確率は「最内だから」ではなく能力の裏付けを必須にする。\n    # z_scores・実戦変換力が低い車は、内を守れても1着への強制昇格を抑える。\n    ability_gate = 1.0 / (1.0 + np.exp(-np.clip(z_scores, -3.0, 3.0)))\n    front_win_prob = np.clip(\n        escape_success_prob * 0.30\n        + inner_hold_prob * 0.10\n        + quinella_rate * 0.17\n        + recent_form_strength * 0.13\n        + st_stability * 0.11\n        + execution_quality * 0.10\n        + ability_gate * 0.07\n        + win_upside_flow * 0.14,\n        0.03, 0.74,\n    ) * front_line.astype(float)\n    base_start_key = handicap * 0.10 + cars.astype(float) * 0.002 + st * 0.88 + outer_start_delay - zero_inner_escape * 0.010 - escape_success_prob * 0.010\n\n    return {\n        "cars": cars, "handicap": handicap, "st": st, "st_std": st_std,\n        "z_scores": z_scores, "finish_sd": finish_sd,\n        "outer_start_delay": outer_start_delay,\n        "same_line_outer_ratio": same_line_outer_ratio,\n        "same_line_inner_ratio": same_line_inner_ratio,\n        "zero_inner_escape": zero_inner_escape,\n        "normal_late_prob": normal_late_prob,\n        "severe_late_prob": severe_late_prob,\n        "base_start_key": base_start_key,\n        "trial_outlier_confidence": trial_outlier_confidence,\n        "form_upside_prob": form_upside_prob,\n        "form_upside_power": form_upside_power,\n        "recent_form_strength": recent_form_strength,\n        "execution_quality": execution_quality,\n        "trial_trust_flow": trial_trust_flow,\n        "st_stability_flow": st_stability_flow,\n        "traffic_conversion": traffic_conversion,\n        "road_stability": road_stability,\n        "current_day_strength": current_day_strength,\n        "explosion_strength": explosion_strength,\n        "trial_dominance_flow": trial_dominance_flow,\n        "front_command_flow": front_command_flow,\n        "pursuit_failure_risk": pursuit_failure_risk,\n        "own_trial_form": own_trial_form,\n        "today_win_conversion": today_win_conversion,\n        "volatility_profile": volatility_profile,\n        "form_leader_strength": np.clip(\n            form_upside_prob * 0.38\n            + recent_form_strength * 0.24\n            + execution_quality * 0.14\n            + closing_strength * 0.20\n            + front_command_flow * 0.12,\n            0.0, 1.0,\n        ),\n        "closing_strength": closing_strength,\n        "win_upside_flow": win_upside_flow,\n        "final_kick_flow": final_kick_flow.to_numpy(float),\n        "place_strength": place_strength,\n        "position_hold_flow": position_hold_flow,\n        "rear_chase_flow": rear_chase_flow,\n        "position_dependency_flow": position_dependency_flow,\n        "bipolar_flow": bipolar_flow,\n        "escape_success_prob": escape_success_prob,\n        "inner_hold_prob": inner_hold_prob,\n        "front_win_prob": front_win_prob,\n        "front_handicap": front_handicap,\n        "front_line": front_line,\n    }\n\n\ndef simulate_detailed(df, trials, seed, track_temp=30.0):\n    """Ver12.2: 大人数同ハンデ線を含む6周モデルに、安定上位と突破機会を追加する。"""\n    rng = np.random.default_rng(seed)\n    arr = prepare_simulation_arrays(df)\n    cars = arr["cars"]; n = len(cars)\n    handicap = arr["handicap"]; st = arr["st"]\n    z_scores = arr["z_scores"]; finish_sd = arr["finish_sd"]\n    explosion_strength = arr["explosion_strength"]\n    trial_dominance_flow = arr["trial_dominance_flow"]\n    front_command_flow = arr.get("front_command_flow", np.zeros(n, dtype=float))\n    pursuit_failure_risk = arr.get("pursuit_failure_risk", np.zeros(n, dtype=float))\n    own_trial_form = arr.get("own_trial_form", np.full(n, 0.5, dtype=float))\n    today_win_conversion = arr.get("today_win_conversion", np.full(n, 0.5, dtype=float))\n    outer_start_delay = arr["outer_start_delay"]\n    same_line_outer_ratio = arr["same_line_outer_ratio"]\n    same_line_inner_ratio = arr["same_line_inner_ratio"]\n    normal_late_prob = arr["normal_late_prob"]; severe_late_prob = arr["severe_late_prob"]\n    base_start_key = arr["base_start_key"]\n    form_upside_prob = arr["form_upside_prob"]; form_upside_power = arr["form_upside_power"]\n    recent_form_strength = arr["recent_form_strength"]\n    execution_quality = arr["execution_quality"]\n    trial_trust_flow = arr["trial_trust_flow"]\n    st_stability_flow = arr["st_stability_flow"]\n    traffic_conversion = arr["traffic_conversion"]\n    road_stability = arr["road_stability"]\n    current_day_strength = arr.get("current_day_strength", np.full(n, 0.50, dtype=float))\n    volatility_profile = arr["volatility_profile"]\n    form_leader_strength = arr["form_leader_strength"]\n    closing_strength = arr["closing_strength"]\n    win_upside_flow = arr.get("win_upside_flow", np.zeros(n, dtype=float))\n    final_kick_flow = arr.get("final_kick_flow", np.zeros(n, dtype=float))\n    place_strength = arr["place_strength"]\n    position_hold_flow = arr.get("position_hold_flow", np.full(n, 0.50))\n    rear_chase_flow = arr.get("rear_chase_flow", np.full(n, 0.35))\n    position_dependency_flow = arr.get("position_dependency_flow", np.full(n, 0.50))\n    bipolar_flow = arr.get("bipolar_flow", np.full(n, 0.30))\n    escape_success_prob = arr["escape_success_prob"]\n    inner_hold_prob = arr["inner_hold_prob"]\n    front_win_prob = arr["front_win_prob"]\n    front_line = arr["front_line"]\n    min_handicap = arr["front_handicap"]\n\n    # Ver11.1: 全車同ハンデは横一列スタートとして明示的に評価する。\n    # 車番固定ではなく、平均ST・ST安定性・今回試走・実戦変換力から\n    # 「スタート一気指数」を作り、外枠でも本当に切れる選手は先頭争いへ出せる。\n    all_same_handicap = bool(n >= 3 and np.nanmax(handicap) - np.nanmin(handicap) < 1e-9)\n    # Ver12.2: 全車横一列だけでなく、0m単騎＋10mに多数など「大人数の同ハンデ線」も混戦モデル対象。\n    unique_h, h_counts = np.unique(handicap, return_counts=True)\n    dense_idx = int(np.argmax(h_counts)) if len(h_counts) else 0\n    dense_handicap = float(unique_h[dense_idx]) if len(unique_h) else float(np.nanmin(handicap))\n    dense_line_size = int(h_counts[dense_idx]) if len(h_counts) else n\n    dense_line_race = bool(dense_line_size >= 4)\n    dense_line_mask = np.isclose(handicap, dense_handicap)\n    trial_values = pd.to_numeric(\n        df.get("試走換算", pd.Series(np.nan, index=df.index)),\n        errors="coerce",\n    ).to_numpy(float)\n    valid_trial = trial_values[np.isfinite(trial_values)]\n    trial_fill = float(np.nanmedian(valid_trial)) if len(valid_trial) else 3.50\n    trial_values = np.where(np.isfinite(trial_values), trial_values, trial_fill)\n\n    def _lower_rank_strength(values):\n        values = np.asarray(values, dtype=float)\n        order_idx = np.argsort(values, kind="stable")\n        strength = np.zeros(len(values), dtype=float)\n        if len(values) <= 1:\n            strength[:] = 0.5\n        else:\n            strength[order_idx] = 1.0 - np.arange(len(values), dtype=float) / (len(values) - 1.0)\n        return strength\n\n    st_launch_strength = _lower_rank_strength(st)\n    trial_launch_strength = _lower_rank_strength(trial_values)\n    same_line_launch_strength = np.clip(\n        st_launch_strength * 0.46\n        + st_stability_flow * 0.19\n        + trial_launch_strength * 0.22\n        + execution_quality * 0.13,\n        0.0, 1.0,\n    )\n\n    # Ver11.3: 好スタートを「実際の先行位置」に変換し、その位置を短時間維持する力。\n    # STだけではなく、本走への変換力・試走信頼度・走路安定性を組み合わせる。\n    # 能力点を直接持ち上げず、スタート成功時だけ展開へ作用させる。\n    trial_trust_flow = pd.to_numeric(\n        df.get("試走信頼度", pd.Series(0.50, index=df.index)),\n        errors="coerce",\n    ).fillna(0.50).clip(0.0, 1.0).to_numpy(float)\n    lead_conversion_strength = np.clip(\n        same_line_launch_strength * 0.43\n        + st_stability_flow * 0.13\n        + execution_quality * 0.18\n        + traffic_conversion * 0.14\n        + road_stability * 0.08\n        + trial_trust_flow * 0.04,\n        0.0, 1.0,\n    )\n\n    finish_matrix = np.zeros((n, n), dtype=np.int64)\n    trifecta = Counter()\n    trio = Counter()\n    exacta = Counter()\n    quinella = Counter()\n    scenario_counts = Counter()\n    phases = 3\n\n    # Ver12.2: 熱による前残りと、スタート後の隊列形成による残りを分離する。\n    # 41℃前後でも横一列スタートで内枠が先に隊列を作れば残れる一方、\n    # 温度だけで全ての前残りを強化しない。\n    track_temp = float(track_temp if track_temp is not None else 30.0)\n    if track_temp < 44.0:\n        heat_index = 0.0\n    elif track_temp < 48.0:\n        heat_index = 0.05 + (track_temp - 44.0) / 4.0 * 0.20\n    elif track_temp < 50.0:\n        heat_index = 0.25 + (track_temp - 48.0) / 2.0 * 0.25\n    else:\n        heat_index = 0.50 + np.clip((track_temp - 50.0) / 10.0, 0.0, 1.0) * 0.50\n    heat_index = float(np.clip(heat_index, 0.0, 1.0))\n    chaos_index = float(np.clip((track_temp - 48.0) / 16.0, 0.0, 1.0))\n    front_temp_gate = 0.20 + heat_index * 0.80\n    escape_success_prob = escape_success_prob * (0.34 + heat_index * 0.66)\n    inner_hold_prob = inner_hold_prob * front_temp_gate\n    front_win_prob = front_win_prob * (0.24 + heat_index * 0.76)\n    normal_pass_margin = 0.19 + heat_index * 0.15\n    incumbent_bonus = 0.060 + heat_index * 0.070\n\n    # 50℃未満でも温度由来の前残りは弱いが、横一列の隊列残存は別処理で発生する。\n    scenario_names = np.array(["先行縦長", "前残り", "混戦", "追い込み"], dtype=object)\n    scenario_prob = np.array([0.36, 0.12, 0.28, 0.24], dtype=float)\n    scenario_prob += heat_index * np.array([-0.055, 0.185, -0.040, -0.090])\n    scenario_prob = np.clip(scenario_prob, 0.03, None)\n    scenario_prob /= scenario_prob.sum()\n    scenario_ids = rng.choice(4, size=int(trials), p=scenario_prob)\n    scenario_noise_map = np.array([0.88, 0.82, 1.28, 1.06])\n    leader_factor_map = np.array([0.90, 1.24, 1.12, 0.70])\n    escape_factor_map = np.array([1.08, 1.22, 0.82, 0.72]) * (1.0 + heat_index * 0.07)\n    hold_factor_map = np.array([1.08, 1.20, 0.90, 0.78]) * (1.0 + heat_index * 0.12)\n    win_factor_map = np.array([1.04, 1.16, 0.82, 0.64]) * (1.0 + heat_index * 0.06)\n\n    for scenario_id in scenario_ids:\n        scenario = scenario_names[int(scenario_id)]\n        scenario_counts[str(scenario)] += 1\n        draw = rng.random(n)\n        severe = draw < severe_late_prob\n        normal = (~severe) & (draw < severe_late_prob + normal_late_prob)\n        late_delay = np.zeros(n, dtype=float)\n        if severe.any():\n            late_delay[severe] = rng.uniform(0.30, 0.62, int(severe.sum()))\n            mask = severe & (handicap == 0)\n            late_delay[mask] += same_line_outer_ratio[mask] * 0.34\n        if normal.any():\n            late_delay[normal] = rng.uniform(0.12, 0.28, int(normal.sum()))\n            mask = normal & (handicap == 0)\n            late_delay[mask] += same_line_outer_ratio[mask] * 0.20\n\n        start_noise_sd = 0.034 * (1.0 + chaos_index * (0.35 + volatility_profile * 0.65))\n        start_key = base_start_key + rng.normal(0, start_noise_sd, n) + late_delay\n\n        if all_same_handicap:\n            # 横一列では内外の距離差より、切った直後の加速とST再現性を優先する。\n            # 55℃付近ではグリップ低下により全体のブレは増える一方、\n            # 好スタートを決めた選手が先にコースを選べる価値も少し高まる。\n            launch_value = 0.040 + heat_index * 0.022\n            start_key -= same_line_launch_strength * launch_value\n            # 横一列では最内側ほど最初の進路を確保しやすい。\n            # ただしST・再現性が低い選手を車番だけで救済しない。\n            inner_lane_ratio = 1.0 - same_line_outer_ratio\n            lane_launch_gate = np.clip(\n                same_line_launch_strength * 0.58 + st_stability_flow * 0.22\n                + execution_quality * 0.20, 0.0, 1.0\n            )\n            start_key -= inner_lane_ratio * lane_launch_gate * 0.024\n\n            # 上位2名程度にだけ「スタート一気」の分岐を持たせる。\n            # 毎回固定せず、指数・安定性・熱走路の組合せで発生させる。\n            launch_rank = np.argsort(-same_line_launch_strength)\n            for launch_order, idx in enumerate(launch_rank[:min(3, n)]):\n                burst_prob = np.clip(\n                    0.07\n                    + same_line_launch_strength[idx] * 0.24\n                    + st_stability_flow[idx] * 0.08\n                    + heat_index * 0.05\n                    - launch_order * 0.035,\n                    0.04, 0.38,\n                )\n                if not severe[idx] and rng.random() < burst_prob:\n                    start_key[idx] -= rng.uniform(0.025, 0.065) * (\n                        0.75 + same_line_launch_strength[idx] * 0.50\n                    )\n\n            # 外枠一律減点を少し緩和。ただしSTが悪い外枠は救済しない。\n            start_key -= same_line_outer_ratio * same_line_launch_strength * (\n                0.010 + heat_index * 0.008\n            )\n\n        order = np.argsort(start_key)\n\n        # Ver11.3: 横一列でスタート上位に入った選手だけ、先行転換を成立させる。\n        # 「指数が高いだけ」では発動せず、実際のスタート順が前方であることを条件にする。\n        launch_retention = np.zeros(n, dtype=float)\n        launch_pair = None\n        if all_same_handicap:\n            front_window = [int(i) for i in order[:min(3, n)]]\n            for start_pos, idx in enumerate(front_window):\n                inner_lane_ratio = 1.0 - same_line_outer_ratio[idx]\n                formation_value = inner_lane_ratio * (\n                    0.55 * lead_conversion_strength[idx]\n                    + 0.45 * same_line_launch_strength[idx]\n                )\n                conversion_prob = np.clip(\n                    0.05\n                    + lead_conversion_strength[idx] * 0.42\n                    + same_line_launch_strength[idx] * 0.15\n                    + formation_value * 0.16\n                    + heat_index * 0.06\n                    - start_pos * 0.055\n                    - (0.10 if normal[idx] else 0.0),\n                    0.03, 0.70,\n                )\n                if not severe[idx] and rng.random() < conversion_prob:\n                    launch_retention[idx] = np.clip(\n                        0.16\n                        + lead_conversion_strength[idx] * 0.34\n                        + formation_value * 0.16\n                        + position_hold_flow[idx] * 0.27\n                        + position_dependency_flow[idx] * 0.09\n                        + heat_index * 0.06\n                        - start_pos * 0.045,\n                        0.16, 0.93,\n                    )\n\n            # 先頭と番手がともに高い先行転換力を持つ場合、短い隊列を形成。\n            # これにより「先頭が行き、もう1車が追走して残る」を自然に再現する。\n            if n >= 2:\n                lead_idx, second_idx = int(order[0]), int(order[1])\n                pair_quality = min(lead_conversion_strength[lead_idx], lead_conversion_strength[second_idx])\n                leader_inner = 1.0 - same_line_outer_ratio[lead_idx]\n                pair_prob = np.clip(\n                    0.04 + pair_quality * 0.27 + leader_inner * 0.08\n                    + heat_index * 0.05,\n                    0.03, 0.43,\n                )\n                if (launch_retention[lead_idx] > 0\n                        and launch_retention[second_idx] > 0\n                        and rng.random() < pair_prob):\n                    launch_pair = (lead_idx, second_idx)\n\n        # 能力の直接支配をさらに弱め、近況・スタート・位置取りを相対的に強化。\n        scenario_noise = scenario_noise_map[int(scenario_id)]\n        # 熱走路では滑りやライン乱れによる上下振れを増やす。\n        # 安定型は増幅を小さく、ムラ型は大きくする。\n        race_noise_sd = (0.16 + finish_sd * 0.075) * scenario_noise\n        race_noise_sd *= 1.0 + chaos_index * (0.22 + volatility_profile * 0.72)\n        race_noise = rng.normal(0, race_noise_sd)\n        base_pace = (\n            z_scores * 0.78\n            + race_noise\n            - (st - 0.20) * 0.58\n            + (execution_quality - 0.50) * 0.20\n            + chaos_index * (road_stability - 0.50) * 0.12\n            + (current_day_strength - 0.50) * (0.10 + heat_index * 0.12)\n            + front_command_flow * (0.13 + heat_index * 0.10)\n            + (own_trial_form - 0.50) * 0.10\n            - pursuit_failure_risk * (0.12 + heat_index * 0.10)\n        )\n\n        # 後方追走不成立は毎回固定減点せず、条件が悪い試行で大敗側へ分岐する。\n        pursuit_fail = rng.random(n) < np.clip(\n            pursuit_failure_risk * (0.24 + chaos_index * 0.18), 0.0, 0.42\n        )\n        if pursuit_fail.any():\n            base_pace[pursuit_fail] -= rng.uniform(0.20, 0.48, int(pursuit_fail.sum()))\n\n        # Ver12.2: 突出試走は毎回固定加点せず、当日の動きが本走で発現する試行だけ強く出す。\n        # 高温時は機力差が展開へ出やすくなる一方、不発試行も残す。\n        explosion_hit_prob = np.clip(\n            0.03 + explosion_strength * (0.30 + heat_index * 0.12)\n            + trial_dominance_flow * (0.12 + heat_index * 0.08),\n            0.03, 0.62,\n        )\n        explosion_hit = rng.random(n) < explosion_hit_prob\n        if explosion_hit.any():\n            explosion_power = (\n                0.10\n                + explosion_strength[explosion_hit] * (0.26 + heat_index * 0.10)\n                + trial_dominance_flow[explosion_hit] * (0.12 + heat_index * 0.06)\n            )\n            base_pace[explosion_hit] += explosion_power\n\n        # 突出気配が発現した車は終盤の勝ち切り分岐にも乗りやすくする。\n        explosion_finish_boost = np.zeros(n, dtype=float)\n        explosion_finish_boost[explosion_hit] = np.clip(\n            explosion_strength[explosion_hit] * 0.16\n            + trial_dominance_flow[explosion_hit] * 0.10,\n            0.0, 0.25,\n        )\n        # 先行転換成立時は序盤の巡航へ小幅加点。後半まで能力差を無効化しない。\n        retained_mask = launch_retention > 0\n        if retained_mask.any():\n            base_pace[retained_mask] += (\n                launch_retention[retained_mask] * (0.10 + heat_index * 0.045)\n            )\n        # Ver12.2: 実際に前方スタートを取れた時だけ、過去の位置維持型を反映する。\n        start_rank = np.empty(n, dtype=int)\n        start_rank[order] = np.arange(n)\n        good_position_now = start_rank <= min(2, n - 1)\n        if good_position_now.any():\n            base_pace[good_position_now] += (\n                position_hold_flow[good_position_now] * 0.10\n                + position_dependency_flow[good_position_now] * 0.045\n            )\n        # 二極化型がスタート後方になった場合は、後方追上げ力が低いほど大敗側へ振れる。\n        poor_position_now = start_rank >= max(4, n - 3)\n        poor_branch_prob = np.clip(\n            0.05 + bipolar_flow * 0.25 + position_dependency_flow * 0.18\n            - rear_chase_flow * 0.18, 0.02, 0.48\n        )\n        poor_collapse = poor_position_now & (rng.random(n) < poor_branch_prob)\n        if poor_collapse.any():\n            base_pace[poor_collapse] -= (\n                0.10 + bipolar_flow[poor_collapse] * 0.18\n                + position_dependency_flow[poor_collapse] * 0.10\n            )\n\n        if launch_pair is not None:\n            pair_leader, pair_follower = launch_pair\n            base_pace[pair_leader] += 0.035 + heat_index * 0.018\n            base_pace[pair_follower] += 0.055 + heat_index * 0.025\n\n        if scenario == "先行縦長":\n            base_pace[front_line] += 0.08 + execution_quality[front_line] * 0.05\n            chase_mask = handicap > min_handicap\n            base_pace[chase_mask] += 0.05 + traffic_conversion[chase_mask] * 0.07\n        elif scenario == "前残り":\n            base_pace[front_line] += 0.14 + recent_form_strength[front_line] * 0.08\n            base_pace[~front_line] -= 0.05\n        elif scenario == "混戦":\n            base_pace += rng.normal(0, 0.10, n)\n        elif scenario == "追い込み":\n            chase_mask = handicap > min_handicap\n            base_pace[chase_mask] += 0.15 + traffic_conversion[chase_mask] * 0.12\n            base_pace[front_line] -= 0.07\n\n        # 好調選手は毎回固定加点せず、その日の展開で動きが出る確率を上げる。\n        flow_hit_prob = np.clip(0.07 + recent_form_strength * 0.38 + form_upside_prob * 0.12, 0.07, 0.50)\n        flow_hit = rng.random(n) < flow_hit_prob\n        if flow_hit.any():\n            base_pace[flow_hit] += rng.uniform(0.04, 0.13, int(flow_hit.sum())) * (0.65 + recent_form_strength[flow_hit])\n\n        form_upside = rng.random(n) < form_upside_prob\n        if form_upside.any():\n            base_pace[form_upside] += rng.uniform(form_upside_power[form_upside] * 0.82, form_upside_power[form_upside] * 1.30)\n\n        # Ver8.1: 近況主役候補。前残り・混戦では、試走外れから本走で戻して\n        # 1着まで押し上がる独立分岐を持たせる。車番固定ではない。\n        leader_candidate = int(np.argmax(form_leader_strength))\n        scenario_leader_factor = leader_factor_map[int(scenario_id)]\n        leader_breakout = (\n            not severe[leader_candidate]\n            and rng.random() < np.clip(\n                0.04\n                + form_leader_strength[leader_candidate] * 0.38 * scenario_leader_factor\n                + closing_strength[leader_candidate] * 0.14,\n                0.04, 0.54,\n            )\n        )\n        if leader_breakout:\n            base_pace[leader_candidate] += (\n                rng.uniform(0.24, 0.48)\n                + form_upside_prob[leader_candidate] * 0.48\n                + recent_form_strength[leader_candidate] * 0.10\n                + closing_strength[leader_candidate] * 0.12\n            )\n\n        recovery = np.ones(n, dtype=float)\n        if severe.any():\n            recovery[severe] = rng.uniform(0.40, 0.64, int(severe.sum()))\n            base_pace[severe] -= rng.uniform(0.28, 0.56, int(severe.sum()))\n        if normal.any():\n            recovery[normal] = rng.uniform(0.64, 0.84, int(normal.sum()))\n            base_pace[normal] -= rng.uniform(0.10, 0.27, int(normal.sum()))\n\n        passed = np.zeros((n, n), dtype=bool)\n        # Ver8.4: 一度抜かれた後の隊列崩壊を記録する。\n        # collapse_level が高いほど、その後の巡航力と抵抗力が落ちる。\n        overtaken_count = np.zeros(n, dtype=np.int8)\n        collapse_level = np.zeros(n, dtype=float)\n        first_overtaken_phase = np.full(n, phases, dtype=np.int8)\n        delay_total = late_delay + outer_start_delay\n        initial_leader = int(order[0])\n        clean_escape = front_line[initial_leader] and not severe[initial_leader] and not normal[initial_leader] and rng.random() < np.clip(escape_success_prob[initial_leader] * escape_factor_map[int(scenario_id)], 0.02, 0.96)\n\n        # 最内前線車は、逃げ失敗でも「内枠残り」へ分岐できる。\n        front_candidates = [int(i) for i in order if front_line[int(i)]]\n        inner_front = min(front_candidates, key=lambda i: cars[i]) if front_candidates else initial_leader\n        inner_hold = (not clean_escape and not severe[inner_front] and rng.random() < np.clip(inner_hold_prob[inner_front] * hold_factor_map[int(scenario_id)], 0.02, 0.97))\n\n        # Ver8.3: 展開成立時だけ勝ち切り抽選を行う。\n        # 内枠残存だけの場合は、クリーンな逃げより勝ち切り条件を厳しくする。\n        win_setup_factor = 1.0 if clean_escape else 0.66\n        front_win = (\n            not severe[inner_front]\n            and (clean_escape or inner_hold)\n            and rng.random() < np.clip(\n                front_win_prob[inner_front] * win_factor_map[int(scenario_id)] * win_setup_factor,\n                0.02, 0.76,\n            )\n        )\n        if front_win:\n            base_pace[inner_front] += rng.uniform(0.12, 0.25) + escape_success_prob[inner_front] * 0.11\n\n        boxed = (not clean_escape and not inner_hold and (normal[inner_front] or rng.random() < 0.18 + same_line_outer_ratio[inner_front] * 0.10))\n\n        if clean_escape:\n            base_pace[initial_leader] += 0.13 + escape_success_prob[initial_leader] * 0.32 + recent_form_strength[initial_leader] * 0.08\n        elif inner_hold:\n            base_pace[inner_front] += 0.15 + inner_hold_prob[inner_front] * 0.24 + recent_form_strength[inner_front] * 0.07\n            inner_pos = int(np.where(order == inner_front)[0][0])\n            if inner_pos <= 2 and rng.random() < 0.20 + inner_hold_prob[inner_front] * 0.22:\n                order = np.delete(order, inner_pos)\n                order = np.insert(order, 0, inner_front)\n        elif boxed:\n            base_pace[inner_front] -= rng.uniform(0.10, 0.24)\n\n        # Ver7.4: 前線の最内車が残る展開では、その後ろの「実戦変換力が高い車」が\n        # コースを拾って続く。一方、外側で変換力が低い車は包まれやすい。\n        front_indices = np.where(front_line)[0]\n        follower = None\n        if (clean_escape or inner_hold) and len(front_indices) >= 3:\n            follower_score = (\n                traffic_conversion[front_indices] * 0.25\n                + form_upside_prob[front_indices] * 0.65\n                + recent_form_strength[front_indices] * 0.15\n                - same_line_outer_ratio[front_indices] * 0.04\n            )\n            follower_score[front_indices == inner_front] = -999.0\n            follower = int(front_indices[int(np.argmax(follower_score))])\n            if rng.random() < np.clip(0.24 + traffic_conversion[follower] * 0.34 + form_upside_prob[follower] * 0.50, 0.20, 0.72):\n                base_pace[follower] += rng.uniform(0.12, 0.25) + form_upside_prob[follower] * 0.22\n\n        weak_outer = front_indices[\n            (same_line_outer_ratio[front_indices] >= 0.70)\n            & (traffic_conversion[front_indices] < 0.43)\n        ]\n        for idx in weak_outer:\n            if rng.random() < 0.24 + (0.43 - traffic_conversion[idx]) * 0.75:\n                base_pace[idx] -= rng.uniform(0.10, 0.23)\n\n        for phase in range(phases):\n            phase_recovery = (phase + 1) / phases\n            # 抜かれた後はライン・リズムを失い、残り周回ほど後退が連鎖しやすい。\n            # ただし実戦変換力が高い選手は崩れ幅を抑える。\n            collapse_drag = collapse_level * (0.72 + 0.12 * phase)\n            pace = (\n                base_pace\n                + rng.normal(0, (0.075 + finish_sd * 0.027) * (1.0 + chaos_index * (0.15 + volatility_profile * 0.48)))\n                + z_scores * (1.0 - recovery) * phase_recovery * 0.34\n                - collapse_drag\n            )\n            pos = n - 1\n            while pos > 0:\n                trailing = int(order[pos]); ahead = int(order[pos - 1])\n                margin = normal_pass_margin + incumbent_bonus\n                # 熱走路では路面を使って前へ出る余地が小さくなるため、\n                # 後方からの仕掛けほど追加の速度差を必要とする。\n                position_heat = heat_index * (0.018 + 0.012 * min(pos, 5))\n                margin += position_heat\n                # 実戦変換力が低い追走車は、速い試走があっても混戦で抜きづらい。\n                margin += max(0.0, 0.50 - traffic_conversion[trailing]) * 0.34\n                # 実戦変換力が高い車は前を捌く際の必要差を少し小さくする。\n                margin -= max(0.0, traffic_conversion[trailing] - 0.58) * 0.16\n                # 前団主導指数が高い10m勢は、同ハンデ集団を捌いて前線へ出る力として扱う。\n                margin -= max(0.0, front_command_flow[trailing] - 0.42) * 0.24\n                # すでに抜かれて崩れた車は、後続への抵抗力も低下する。\n                margin -= min(0.24, collapse_level[ahead] * 0.30)\n\n                if clean_escape and ahead == initial_leader and pos - 1 == 0:\n                    margin += max(0.11, 0.15 - phase * 0.022 + escape_success_prob[ahead] * 0.24)\n                    margin += heat_index * (0.055 - phase * 0.007)\n                if inner_hold and ahead == inner_front and pos - 1 <= 2:\n                    # 先頭でなくてもインを守る車を抜くには余分な差が必要。\n                    margin += max(0.09, 0.16 - phase * 0.017 + inner_hold_prob[ahead] * 0.19 + recent_form_strength[ahead] * 0.035)\n                if front_win and ahead == inner_front and pos - 1 == 0:\n                    margin += max(0.12, 0.20 - phase * 0.020 + front_win_prob[ahead] * 0.22)\n                if follower is not None and trailing == follower and pos <= 4:\n                    margin -= 0.07 + form_upside_prob[follower] * 0.12\n\n                # Ver11.3: 先行転換に成功した車は1〜2周目だけ抜かれにくくする。\n                # 熱走路ほど追抜側にも負荷がかかるが、後半には効果を減衰させる。\n                if launch_retention[ahead] > 0 and pos - 1 <= 2:\n                    early_retention = max(0.0, 1.0 - phase / max(1, phases - 1))\n                    margin += launch_retention[ahead] * early_retention * (\n                        0.095 + heat_index * 0.060\n                    )\n                # 前に出た時に抜かれにくい選手は、序盤限定ではなく中終盤まで抵抗する。\n                if start_rank[ahead] <= 2 and pos - 1 <= 3:\n                    phase_keep = 0.70 + 0.30 * max(0.0, 1.0 - phase / max(1, phases - 1))\n                    margin += position_hold_flow[ahead] * position_dependency_flow[ahead] * phase_keep * 0.105\n                if launch_pair is not None:\n                    pair_leader, pair_follower = launch_pair\n                    if ahead == pair_leader and trailing == pair_follower and pos - 1 == 0:\n                        # 番手車は先頭を無理に早仕掛けせず、隊列を保ちやすい。\n                        margin += 0.055 + heat_index * 0.025\n                    elif ahead == pair_follower and pos - 1 <= 1:\n                        # 先頭・番手が形成された時、後続が番手を抜く障壁を少し増やす。\n                        margin += 0.045 + heat_index * 0.040\n\n                close_count = sum(abs(pace[int(order[k])] - pace[trailing]) < 0.30 for k in range(pos + 1, min(n, pos + 4)))\n                delayed_nearby = sum(delay_total[int(order[k])] >= 0.16 for k in range(max(0, pos - 1), min(n, pos + 3)))\n                margin += min(0.23, close_count * 0.055 + delayed_nearby * 0.042)\n\n                if passed[ahead, trailing]:\n                    margin += same_or_stronger_repass_penalty(pace[trailing], pace[ahead])\n                if severe[trailing]: margin += 0.16 * (1.0 - phase_recovery)\n                elif normal[trailing]: margin += 0.08 * (1.0 - phase_recovery)\n                if handicap[trailing] == 0 and same_line_outer_ratio[trailing] > 0 and (severe[trailing] or normal[trailing]):\n                    margin += same_line_outer_ratio[trailing] * (0.78 - phase * 0.11)\n\n                if pace[trailing] > pace[ahead] + margin:\n                    # 抜かれた側の崩れ判定。能力差が大きい、早い周回、\n                    # 実戦変換力が低い、すでに一度抜かれているほど連鎖しやすい。\n                    ability_gap = max(0.0, pace[trailing] - pace[ahead] - margin)\n                    early_factor = (phases - phase) / phases\n                    stability_guard = (\n                        0.52 * execution_quality[ahead]\n                        + 0.30 * traffic_conversion[ahead]\n                        + 0.18 * recent_form_strength[ahead]\n                    )\n                    collapse_prob = np.clip(\n                        0.10\n                        + ability_gap * 0.34\n                        + early_factor * 0.13\n                        + overtaken_count[ahead] * 0.16\n                        + max(0.0, 0.55 - stability_guard) * 0.48,\n                        0.06, 0.82,\n                    )\n                    if rng.random() < collapse_prob:\n                        collapse_add = (\n                            0.10\n                            + min(0.32, ability_gap * 0.24)\n                            + early_factor * 0.07\n                            + max(0.0, 0.55 - stability_guard) * 0.16\n                        )\n                        collapse_level[ahead] = min(0.78, collapse_level[ahead] + collapse_add)\n                    overtaken_count[ahead] += 1\n                    if first_overtaken_phase[ahead] == phases:\n                        first_overtaken_phase[ahead] = phase\n\n                    order[pos - 1], order[pos] = trailing, ahead\n                    passed[trailing, ahead] = True\n                pos -= 1\n\n        # Ver7.4: 前線最内が残ったレースでは、隊列が崩れにくい展開も作る。\n        # 番手候補は車番固定ではなく、試走外れ復調・実戦変換力・近況から選ぶ。\n        if (clean_escape or inner_hold) and int(order[0]) == inner_front and follower is not None:\n            line_lock_prob = np.clip(\n                0.18 + inner_hold_prob[inner_front] * 0.16\n                + form_upside_prob[follower] * 0.48\n                + traffic_conversion[follower] * 0.10\n                + heat_index * 0.14,\n                0.18, 0.66,\n            )\n            if rng.random() < line_lock_prob:\n                cur_f = int(np.where(order == follower)[0][0])\n                if cur_f != 1:\n                    order = np.delete(order, cur_f)\n                    order = np.insert(order, 1, follower)\n\n                remaining_front = [\n                    int(i) for i in np.where(front_line)[0]\n                    if int(i) not in (inner_front, follower)\n                ]\n                if remaining_front:\n                    third_score = np.array([\n                        recent_form_strength[i] * 0.44\n                        + traffic_conversion[i] * 0.36\n                        + execution_quality[i] * 0.20\n                        for i in remaining_front\n                    ])\n                    third_candidate = remaining_front[int(np.argmax(third_score))]\n                    cur_t = int(np.where(order == third_candidate)[0][0])\n                    if cur_t != 2 and rng.random() < 0.62:\n                        order = np.delete(order, cur_t)\n                        order = np.insert(order, 2, third_candidate)\n\n        # Ver8.1: シナリオごとの最終隊列調整。\n        if scenario == "先行縦長":\n            front_now = [int(i) for i in order if front_line[int(i)]]\n            if len(front_now) >= 2:\n                selected_front = sorted(front_now, key=lambda i: (-base_pace[i], int(cars[i])))[:2]\n                for target_pos, idx in enumerate(selected_front):\n                    cur = int(np.where(order == idx)[0][0])\n                    order = np.delete(order, cur); order = np.insert(order, target_pos, idx)\n            chase_candidates = [int(i) for i in order if handicap[int(i)] > min_handicap and not severe[int(i)]]\n            if chase_candidates:\n                chase_score = np.array([base_pace[i] + traffic_conversion[i] * 0.20 for i in chase_candidates])\n                chaser = chase_candidates[int(np.argmax(chase_score))]\n                cur = int(np.where(order == chaser)[0][0])\n                if cur > 2 and rng.random() < (0.48 + traffic_conversion[chaser] * 0.24) * (1.0 - heat_index * 0.38):\n                    order = np.delete(order, cur); order = np.insert(order, 2, chaser)\n        elif scenario == "追い込み":\n            chase_candidates = [int(i) for i in order if handicap[int(i)] > min_handicap and not severe[int(i)]]\n            if chase_candidates:\n                chase_score = np.array([base_pace[i] + traffic_conversion[i] * 0.26 for i in chase_candidates])\n                chaser = chase_candidates[int(np.argmax(chase_score))]\n                cur = int(np.where(order == chaser)[0][0])\n                target = 1 if rng.random() < 0.35 * (1.0 - heat_index * 0.42) else 2\n                if cur > target:\n                    order = np.delete(order, cur); order = np.insert(order, target, chaser)\n        elif scenario == "混戦":\n            front_top = sum(front_line[int(i)] for i in order[:4])\n            if front_top >= 3 and rng.random() < 0.24 * (1.0 - heat_index * 0.30):\n                chase_candidates = [int(i) for i in order if handicap[int(i)] > min_handicap and traffic_conversion[int(i)] >= 0.45]\n                if chase_candidates:\n                    chaser = max(chase_candidates, key=lambda i: base_pace[i] + traffic_conversion[i] * 0.18)\n                    cur = int(np.where(order == chaser)[0][0])\n                    target = int(rng.integers(1, 4))\n                    if cur > target:\n                        order = np.delete(order, cur); order = np.insert(order, target, chaser)\n\n        # Ver8.4: 隊列崩壊の最終連鎖。\n        # 早い段階で能力上位に抜かれ、後方にも強い車が複数いる場合は、\n        # 2～3着で踏みとどまらず着外まで飲み込まれる展開を作る。\n        for idx in range(n):\n            if overtaken_count[idx] <= 0 or collapse_level[idx] < 0.16:\n                continue\n            cur = int(np.where(order == idx)[0][0])\n            if cur >= n - 1:\n                continue\n            behind = [int(j) for j in order[cur + 1:]]\n            stronger_behind = [\n                j for j in behind\n                if (z_scores[j] - z_scores[idx] > 0.28)\n                and traffic_conversion[j] >= 0.42\n                and not severe[j]\n            ]\n            if not stronger_behind:\n                continue\n            early = 1.0 - first_overtaken_phase[idx] / max(1, phases)\n            cascade_prob = np.clip(\n                0.08\n                + collapse_level[idx] * 0.62\n                + min(3, len(stronger_behind)) * 0.10\n                + early * 0.14\n                - execution_quality[idx] * 0.16\n                - heat_index * 0.10,\n                0.04, 0.78,\n            )\n            if rng.random() < cascade_prob:\n                max_drop = min(len(stronger_behind), 3)\n                drop = 1 + int(rng.integers(0, max_drop))\n                target = min(n - 1, cur + drop)\n                order = np.delete(order, cur)\n                order = np.insert(order, target, idx)\n\n        # Ver8.3: 勝ち切り抽選成立でも無条件に先頭へ固定しない。\n        # クリーン逃げ、能力、現在位置が揃った場合に限って先頭へ戻す。\n        if front_win:\n            cur_w = int(np.where(order == inner_front)[0][0])\n            finish_win_prob = np.clip(\n                (0.46 if clean_escape else 0.24)\n                + front_win_prob[inner_front] * 0.34\n                + execution_quality[inner_front] * 0.12\n                - max(0, cur_w - 2) * 0.10\n                - collapse_level[inner_front] * 0.44\n                - overtaken_count[inner_front] * 0.08,\n                0.08, 0.88,\n            )\n            if cur_w <= 3 and rng.random() < finish_win_prob:\n                order = np.delete(order, cur_w)\n                order = np.insert(order, 0, inner_front)\n\n        # Ver8.1: 近況主役候補が展開をつかんだ場合は、最終的に1～2着へ進出。\n        # 前残りでは先頭まで、先行縦長・混戦では1～2番手を抽選する。\n        if leader_breakout and not severe[leader_candidate]:\n            cur_l = int(np.where(order == leader_candidate)[0][0])\n            # 能力が高くても勝ち切り指数が低い選手は、1着固定ではなく2～3着へ。\n            # ただし最前線の逃げ車は front_win 側で別評価するため、ここでは後方主役候補を調整。\n            close = float(closing_strength[leader_candidate])\n            if scenario == "前残り":\n                first_prob = 0.30 + close * 0.48\n                target_l = 0 if rng.random() < first_prob else 1\n            elif scenario == "混戦":\n                first_prob = 0.18 + close * 0.42\n                target_l = 0 if rng.random() < first_prob else 1\n            elif scenario == "先行縦長":\n                first_prob = 0.12 + close * 0.34\n                target_l = 0 if rng.random() < first_prob else 1\n            else:\n                target_l = 1 if rng.random() < (0.48 + place_strength[leader_candidate] * 0.30) else 2\n            if front_win and leader_candidate != inner_front:\n                target_l = max(1, target_l)\n            if cur_l > target_l:\n                order = np.delete(order, cur_l)\n                order = np.insert(order, target_l, leader_candidate)\n\n        # Ver9.0: 最終勝ち切り判定。\n        # 後方から能力で先頭へ来たものの勝ち切り実績が弱い選手は、\n        # 勝負強い2～3番手候補に差される場合を作る。\n        # 1番など最前線車の clean_escape / front_win は保護して、逃げ残りを消さない。\n        current_leader = int(order[0])\n        protected_escape = (\n            current_leader == inner_front\n            and (clean_escape or front_win)\n            and collapse_level[current_leader] < 0.22\n        )\n        if not protected_escape and closing_strength[current_leader] < 0.48:\n            challengers = [\n                int(i) for i in order[1:4]\n                if not severe[int(i)]\n                and closing_strength[int(i)] > closing_strength[current_leader] + 0.10\n                and base_pace[int(i)] > base_pace[current_leader] - 0.18\n            ]\n            if challengers:\n                finisher = max(\n                    challengers,\n                    key=lambda i: closing_strength[i] * 0.58 + base_pace[i] * 0.24 + place_strength[i] * 0.18,\n                )\n                conversion_prob = np.clip(\n                    0.06\n                    + (closing_strength[finisher] - closing_strength[current_leader]) * 0.42\n                    + max(0.0, base_pace[finisher] - base_pace[current_leader]) * 0.18\n                    - heat_index * (0.09 - np.maximum(0.0, final_kick_flow[finisher]) * 0.035),\n                    0.02, 0.34,\n                )\n                if rng.random() < conversion_prob:\n                    cur_f = int(np.where(order == finisher)[0][0])\n                    order = np.delete(order, cur_f)\n                    order = np.insert(order, 0, finisher)\n\n        # Ver10.7: 能力順位が中位でも、格・ST・勝ち切り力が揃う選手には\n        # 展開がほどけた際の一発逆転を残す。現在2～5番手にいることを条件とし、\n        # 後方から無条件に先頭へ飛ばす処理にはしない。\n        upset_pool = [\n            int(i) for i in order[1:5]\n            if not severe[int(i)]\n            and win_upside_flow[int(i)] >= 0.34\n            and closing_strength[int(i)] >= 0.24\n            and base_pace[int(i)] > base_pace[int(order[0])] - 0.38\n        ]\n        if upset_pool and not protected_escape:\n            upsetter = max(\n                upset_pool,\n                key=lambda i: (\n                    win_upside_flow[i] * 0.30\n                    + today_win_conversion[i] * 0.32\n                    + closing_strength[i] * 0.22\n                    + execution_quality[i] * 0.10\n                    + st_stability_flow[i] * 0.06\n                ),\n            )\n            upset_prob = np.clip(\n                0.015\n                + win_upside_flow[upsetter] * 0.070\n                + today_win_conversion[upsetter] * 0.085\n                + closing_strength[upsetter] * 0.055\n                + max(0.0, base_pace[upsetter] - base_pace[current_leader]) * 0.06\n                - heat_index * 0.035,\n                0.015, 0.145,\n            )\n            if rng.random() < upset_prob:\n                cur_u = int(np.where(order == upsetter)[0][0])\n                order = np.delete(order, cur_u)\n                order = np.insert(order, 0, upsetter)\n\n        # 包まれた内枠車は1着候補から落ちやすいが、完全着外固定にはしない。\n        if boxed:\n            cur = int(np.where(order == inner_front)[0][0])\n            target = min(n - 1, cur + int(rng.integers(1, 4)))\n            if target > cur:\n                order = np.delete(order, cur); order = np.insert(order, target, inner_front)\n\n        for idx in range(n):\n            if handicap[idx] == 0 and same_line_outer_ratio[idx] > 0 and (severe[idx] or normal[idx]):\n                cur = int(np.where(order == idx)[0][0])\n                drop = int(rng.integers(4, 7)) if severe[idx] else int(rng.integers(3, 6))\n                drop = max(1, int(round(drop * (0.70 + same_line_outer_ratio[idx]))))\n                target = min(n - 1, cur + drop)\n                if target > cur:\n                    order = np.delete(order, cur); order = np.insert(order, target, idx)\n\n        # Ver10.3: 能力順位とは別に、スタート後の「隊列連鎖」を再現する。\n        # 車番固定ではなく、ハンデ構成・内外位置・逃げ力・追走力・レール残存力から選ぶ。\n        unique_lines = np.unique(handicap)\n        front_members = [int(i) for i in np.where(front_line)[0]]\n        back_members = [int(i) for i in np.where(handicap > min_handicap)[0]]\n\n        # A) 複数ハンデ線: 前線の外寄りで逃げ力が高い車が主導権を取ると、\n        # 後方線の追走上位が2番手、前線の総合力上位が3番手に収まる隊列。\n        if len(unique_lines) >= 2 and len(front_members) >= 3 and back_members:\n            launch_pool = [\n                i for i in front_members\n                if same_line_outer_ratio[i] >= 0.45 and not severe[i]\n            ]\n            if launch_pool:\n                launch_scores = {\n                    i: (\n                        escape_success_prob[i] * 0.34\n                        + execution_quality[i] * 0.18\n                        + recent_form_strength[i] * 0.14\n                        + closing_strength[i] * 0.12\n                        + place_strength[i] * 0.10\n                        + (1.0 / (1.0 + np.exp(-z_scores[i]))) * 0.12\n                    )\n                    for i in launch_pool\n                }\n                flow_leader = max(launch_pool, key=lambda i: launch_scores[i])\n                flow_pos = int(np.where(order == flow_leader)[0][0])\n                chase_pool = [i for i in back_members if not severe[i]]\n                if (\n                    chase_pool\n                    and flow_pos <= 3\n                    and escape_success_prob[flow_leader] >= 0.76\n                    and max(place_strength[i] for i in chase_pool) >= 0.50\n                ):\n                    chase_scores = {\n                        i: (\n                            place_strength[i] * 0.34\n                            + traffic_conversion[i] * 0.28\n                            + execution_quality[i] * 0.18\n                            + closing_strength[i] * 0.12\n                            + recent_form_strength[i] * 0.08\n                            - same_line_outer_ratio[i] * 0.04\n                        )\n                        for i in chase_pool\n                    }\n                    chaser = max(chase_pool, key=lambda i: chase_scores[i])\n                    hold_pool = [i for i in front_members if i != flow_leader and not severe[i]]\n                    if hold_pool:\n                        hold_scores = {\n                            i: (\n                                (1.0 / (1.0 + np.exp(-z_scores[i]))) * 0.34\n                                + place_strength[i] * 0.22\n                                + execution_quality[i] * 0.18\n                                + recent_form_strength[i] * 0.14\n                                + inner_hold_prob[i] * 0.08\n                                + same_line_inner_ratio[i] * 0.04\n                            )\n                            for i in hold_pool\n                        }\n                        holder = max(hold_pool, key=lambda i: hold_scores[i])\n                        chain_prob = np.clip(\n                            0.18 + launch_scores[flow_leader] * 0.18\n                            + chase_scores[chaser] * 0.12\n                            + (0.05 if scenario in ("先行縦長", "前残り") else 0.0)\n                            - flow_pos * 0.025,\n                            0.16, 0.42,\n                        )\n                        if rng.random() < chain_prob:\n                            selected = [flow_leader, chaser, holder]\n                            rest = [int(i) for i in order if int(i) not in selected]\n                            order = np.array(selected + rest, dtype=int)\n\n        # B) 全車同一ハンデ線: 外寄りの先行車が切り込み、\n        # その内側の勝負強い車が追走し、最内寄りの安定車が3着へ残る隊列。\n        if len(unique_lines) == 1 and n >= 6:\n            outer_candidates = [\n                i for i in range(n)\n                if 0.55 <= same_line_outer_ratio[i] < 0.99 and not severe[i]\n            ]\n            if outer_candidates:\n                launch_scores = {\n                    i: (\n                        escape_success_prob[i] * 0.28\n                        + closing_strength[i] * 0.20\n                        + place_strength[i] * 0.16\n                        + execution_quality[i] * 0.16\n                        + recent_form_strength[i] * 0.12\n                        + traffic_conversion[i] * 0.08\n                    )\n                    for i in outer_candidates\n                }\n                launch_leader = max(outer_candidates, key=lambda i: launch_scores[i])\n                follower_pool = [\n                    i for i in range(n)\n                    if i != launch_leader\n                    and cars[i] < cars[launch_leader]\n                    and same_line_outer_ratio[i] >= 0.30\n                    and not severe[i]\n                ]\n                rail_pool = [\n                    i for i in range(n)\n                    if i != launch_leader\n                    and same_line_inner_ratio[i] >= 0.70\n                    and not severe[i]\n                ]\n                if follower_pool and rail_pool:\n                    follower_scores = {\n                        i: (\n                            closing_strength[i] * 0.38\n                            + place_strength[i] * 0.22\n                            + execution_quality[i] * 0.17\n                            + traffic_conversion[i] * 0.13\n                            + recent_form_strength[i] * 0.10\n                            - abs(float(cars[launch_leader] - cars[i]) - 2.0) * 0.025\n                        )\n                        for i in follower_pool\n                    }\n                    follower2 = max(follower_pool, key=lambda i: follower_scores[i])\n                    rail_pool = [i for i in rail_pool if i != follower2]\n                    if rail_pool:\n                        rail_scores = {\n                            i: (\n                                same_line_inner_ratio[i] * 0.42\n                                + trial_trust_flow[i] * 0.20\n                                + recent_form_strength[i] * 0.16\n                                + place_strength[i] * 0.12\n                                + st_stability_flow[i] * 0.10\n                            )\n                            for i in rail_pool\n                        }\n                        rail3 = max(rail_pool, key=lambda i: rail_scores[i])\n                        formation_prob = np.clip(\n                            0.11 + launch_scores[launch_leader] * 0.16\n                            + follower_scores[follower2] * 0.11\n                            + (0.04 if scenario in ("先行縦長", "混戦") else 0.0),\n                            0.11, 0.30,\n                        )\n                        # 現在の先頭が選定先行車、または先頭争い圏内のときだけ成立。\n                        launch_pos = int(np.where(order == launch_leader)[0][0])\n                        if launch_pos <= 2 and rng.random() < formation_prob:\n                            selected = [launch_leader, follower2, rail3]\n                            rest = [int(i) for i in order if int(i) not in selected]\n                            order = np.array(selected + rest, dtype=int)\n\n        # Ver12.2: 全車同ハンデ限定の6周簡易モデル。\n        # 現在の順位を1周目隊列として、隣接する相手だけを抜く現実寄りの処理を重ねる。\n        # 前団が密集すると後方ほど突破障壁が増えるが、集団突破力の高い選手は越えられる。\n        if dense_line_race and n >= 5:\n            group_breakthrough = pd.to_numeric(\n                df.get("集団突破力", pd.Series(0.50, index=df.index)), errors="coerce"\n            ).fillna(0.50).clip(0, 1).to_numpy(float)\n\n            # 大人数同ハンデ線では、能力順位より先にスタートで線内隊列を作る。\n            # 内側は進路を取りやすいが、スタート一気・先行転換力が高い外側は前へ出られる。\n            if not all_same_handicap:\n                dense_members = [int(i) for i in order if dense_line_mask[int(i)]]\n                other_members = [int(i) for i in order if not dense_line_mask[int(i)]]\n                if len(dense_members) >= 4:\n                    launch_metric = {}\n                    for ii in dense_members:\n                        launch_metric[ii] = (\n                            same_line_launch_strength[ii] * 0.34\n                            + lead_conversion_strength[ii] * 0.26\n                            + st_stability_flow[ii] * 0.14\n                            + trial_launch_strength[ii] * 0.10\n                            + same_line_inner_ratio[ii] * 0.16\n                            + rng.normal(0.0, 0.075)\n                        )\n                    dense_members = sorted(dense_members, key=lambda ii: launch_metric[ii], reverse=True)\n                    # 最前ハンデ車は序盤の前、密集線はその直後へ置く。\n                    front_others = sorted(other_members, key=lambda ii: (handicap[ii], list(order).index(ii)))\n                    order = np.array(front_others + dense_members, dtype=int)\n\n            # 1周目の隊列を強く残す。好スタート内枠だけ小さな進路確保を持つ。\n            first_lap_pos = np.empty(n, dtype=int)\n            for p0, i0 in enumerate(order):\n                first_lap_pos[int(i0)] = p0\n            front_pack_size = int(min(n - 1, max(3, min(5, dense_line_size // 2 + 1))))\n\n            for lap in range(2, 7):\n                # 2〜4周目は前団内の攻防中心、5〜6周目は突破力と終盤力を強める。\n                late_phase = max(0.0, (lap - 4) / 2.0)\n                new_order = order.copy()\n                # 前からではなく後ろから判定し、1周に何人も瞬間移動するのを防ぐ。\n                moved = set()\n                for pos in range(n - 1, 0, -1):\n                    chaser = int(new_order[pos])\n                    leader = int(new_order[pos - 1])\n                    if chaser in moved or leader in moved:\n                        continue\n\n                    # 先頭3台付近が固まるほど壁が厚くなる。後方から前団入口へ来た選手に強く作用。\n                    front_density = min(1.0, front_pack_size / 4.0)\n                    entering_front = pos <= front_pack_size + 1\n                    congestion = (0.13 + 0.24 * front_density) if entering_front else (0.04 + 0.08 * front_density)\n                    if dense_line_mask[chaser] and lap <= 4:\n                        congestion += same_line_outer_ratio[chaser] * 0.10\n\n                    # 内側の恩恵は「好スタートで前団に入れた」場合のみ。固定の能力加点ではない。\n                    leader_early_front = first_lap_pos[leader] <= 2\n                    inner_route_hold = (same_line_inner_ratio[leader] ** 1.4) * (0.10 if leader_early_front else 0.025)\n\n                    chase_power = (\n                        group_breakthrough[chaser] * (0.32 + 0.20 * late_phase)\n                        + closing_strength[chaser] * (0.15 + 0.17 * late_phase)\n                        + traffic_conversion[chaser] * 0.18\n                        + execution_quality[chaser] * 0.12\n                        + rear_chase_flow[chaser] * 0.10\n                        + current_day_strength[chaser] * 0.08\n                    )\n                    stable_top_flow = pd.to_numeric(\n                        df.get("安定上位指数", pd.Series(0.50, index=df.index)), errors="coerce"\n                    ).fillna(0.50).clip(0, 1).to_numpy(float)\n                    hold_power = (\n                        position_hold_flow[leader] * 0.21\n                        + execution_quality[leader] * 0.16\n                        + place_strength[leader] * 0.15\n                        + traffic_conversion[leader] * 0.10\n                        + st_stability_flow[leader] * 0.08\n                        + stable_top_flow[leader] * 0.18\n                        + inner_route_hold\n                    )\n\n                    # Ver12.2: 能力だけでは抜けず、前団が崩れる「突破機会」が必要。\n                    # 前走者の位置維持が弱い、前団内に能力差がある、終盤になるほど隙間が生まれる。\n                    leader_instability = np.clip(\n                        0.55 - position_hold_flow[leader] * 0.26\n                        - execution_quality[leader] * 0.14\n                        + volatility_profile[leader] * 0.18,\n                        0.08, 0.72\n                    )\n                    pack_spread = np.clip(\n                        abs(execution_quality[chaser] - execution_quality[leader]) * 0.35\n                        + abs(current_day_strength[chaser] - current_day_strength[leader]) * 0.25,\n                        0.0, 0.35\n                    )\n                    breakthrough_opportunity = np.clip(\n                        0.12 + leader_instability * 0.34 + pack_spread\n                        + late_phase * 0.22 - congestion * 0.30,\n                        0.035, 0.72\n                    )\n                    congestion_after_skill = congestion * (1.0 - 0.42 * group_breakthrough[chaser])\n                    margin = chase_power - hold_power - congestion_after_skill\n                    pass_prob = 1.0 / (1.0 + np.exp(-(margin - 0.085) * 6.6))\n                    # 適性と機会を掛け合わせる。強い選手でも混戦が開かなければ待たされる。\n                    pass_prob *= breakthrough_opportunity * (0.72 + 0.38 * late_phase)\n                    if severe[chaser]:\n                        pass_prob *= 0.20\n                    elif normal[chaser]:\n                        pass_prob *= 0.62\n                    if bipolar_flow[chaser] > 0.65 and first_lap_pos[chaser] >= n - 2:\n                        pass_prob *= 0.78\n\n                    if rng.random() < np.clip(pass_prob, 0.01, 0.72):\n                        new_order[pos - 1], new_order[pos] = chaser, leader\n                        moved.add(chaser); moved.add(leader)\n\n                order = new_order\n\n                # 前団内では、好位置維持だけでなく実戦能力のある選手が少しずつ上がれる。\n                # これにより3番が2番、続いて1番を抜くような段階的変動を許す。\n                if lap <= 4 and front_pack_size >= 2:\n                    for pos in range(min(front_pack_size, n - 1), 0, -1):\n                        chaser = int(order[pos]); leader = int(order[pos - 1])\n                        internal_edge = (\n                            group_breakthrough[chaser] * 0.28 + execution_quality[chaser] * 0.24\n                            + current_day_strength[chaser] * 0.18 + closing_strength[chaser] * 0.12\n                            - position_hold_flow[leader] * 0.22 - execution_quality[leader] * 0.10\n                        )\n                        internal_prob = np.clip(0.07 + internal_edge * 0.30, 0.015, 0.30)\n                        if rng.random() < internal_prob:\n                            order[pos - 1], order[pos] = chaser, leader\n                            break\n\n        for rank, idx in enumerate(order):\n            finish_matrix[int(idx), rank] += 1\n        top3 = tuple(int(cars[int(order[i])]) for i in range(3))\n        top2 = tuple(int(cars[int(order[i])]) for i in range(2))\n        trifecta[top3] += 1\n        trio[tuple(sorted(top3))] += 1\n        exacta[top2] += 1\n        quinella[tuple(sorted(top2))] += 1\n\n    finish_counts = {int(car): Counter({rank + 1: int(finish_matrix[i, rank]) for rank in range(n) if finish_matrix[i, rank] > 0}) for i, car in enumerate(cars)}\n    bet_counts = {\n        "三連単": trifecta,\n        "三連複": trio,\n        "2車単": exacta,\n        "2車複": quinella,\n    }\n    return finish_counts, bet_counts\n\n\n\ndef simulate(df, trials, seed, track_temp=30.0):\n    """Ver12.3: 高速ベクトル型の6周近似モデル。\n\n    1周目のST反応とスタート後の伸びを分離し、最終周には僅差の差し判定を追加する。\n    車番固定の結果合わせは行わず、全選手共通の指標から確率的に発生させる。\n    """\n    rng = np.random.default_rng(int(seed))\n    arr = prepare_simulation_arrays(df)\n    cars = np.asarray(arr["cars"], dtype=int)\n    n = len(cars); trials = int(trials)\n    handicap = np.asarray(arr["handicap"], dtype=float)\n    z = np.asarray(arr["z_scores"], dtype=float)\n    finish_sd = np.asarray(arr["finish_sd"], dtype=float)\n    st = np.asarray(arr["st"], dtype=float)\n    st_stability = np.asarray(arr["st_stability_flow"], dtype=float)\n    execution = np.asarray(arr["execution_quality"], dtype=float)\n    traffic = np.asarray(arr["traffic_conversion"], dtype=float)\n    closing = np.asarray(arr["closing_strength"], dtype=float)\n    current = np.asarray(arr.get("current_day_strength", np.full(n, .5)), dtype=float)\n    position_hold = np.asarray(arr.get("position_hold_flow", np.full(n, .5)), dtype=float)\n    rear_chase = np.asarray(arr.get("rear_chase_flow", np.full(n, .35)), dtype=float)\n    dependency = np.asarray(arr.get("position_dependency_flow", np.full(n, .5)), dtype=float)\n    bipolar = np.asarray(arr.get("bipolar_flow", np.full(n, .3)), dtype=float)\n    volatility = np.asarray(arr.get("volatility_profile", np.full(n, .5)), dtype=float)\n    final_kick = np.asarray(arr.get("final_kick_flow", closing), dtype=float)\n    recent_form = np.asarray(arr.get("recent_form_strength", current), dtype=float)\n\n    stable = pd.to_numeric(df.get("安定上位指数", pd.Series(.5,index=df.index)), errors="coerce").fillna(.5).clip(0,1).to_numpy(float)\n    breakthrough = pd.to_numeric(df.get("集団突破力", pd.Series(.5,index=df.index)), errors="coerce").fillna(.5).clip(0,1).to_numpy(float)\n    trial = pd.to_numeric(df.get("試走換算", pd.Series(np.nan,index=df.index)), errors="coerce").to_numpy(float)\n    fill=float(np.nanmedian(trial[np.isfinite(trial)])) if np.isfinite(trial).any() else 3.50\n    trial=np.where(np.isfinite(trial),trial,fill)\n    trial_strength=(trial.max()-trial)/(max(1e-6,trial.max()-trial.min())) if trial.max()>trial.min() else np.full(n,.5)\n    st_fill=float(np.nanmedian(st[np.isfinite(st)])) if np.isfinite(st).any() else .15\n    st2=np.where(np.isfinite(st),st,st_fill)\n    st_strength=(st2.max()-st2)/(max(1e-6,st2.max()-st2.min())) if st2.max()>st2.min() else np.full(n,.5)\n\n    temp=float(track_temp if track_temp is not None else 30.0)\n    heat=np.clip((temp-44.0)/12.0,0,1)\n    unique_h, counts=np.unique(handicap,return_counts=True)\n    dense_h=float(unique_h[np.argmax(counts)])\n    dense_mask=np.isclose(handicap,dense_h)\n    dense_size=int(counts.max()); same_line=dense_size>=4\n    line_indices=np.where(dense_mask)[0]\n    lane_ratio=np.zeros(n)\n    if len(line_indices)>1:\n        order_line=line_indices[np.argsort(cars[line_indices])]\n        lane_ratio[order_line]=np.linspace(1.0,0.0,len(order_line))\n\n    # 評価の中心。安定上位は残すが、格だけで外枠が自動突破しないよう抑える。\n    base=(z*.76 + stable*.30 + execution*.15 + current*.13 + trial_strength*.11)\n    noise_sd=np.clip(.68 + finish_sd*.50 + volatility*.30, .52, 1.45)\n    score=base[None,:] + rng.normal(0,noise_sd,size=(trials,n))\n\n    # ST反応と「その後の伸び」を分離。\n    reaction = st_strength*.54 + st_stability*.25 + execution*.13 + trial_strength*.08\n    start_draw = reaction[None,:] + rng.normal(0,.25,size=(trials,n))\n    stretch_index=np.clip(trial_strength*.30 + execution*.25 + st_stability*.16 + current*.14 + recent_form*.10 + traffic*.05,0,1)\n    # 上振れは全員に起こり得るが、指数が高いほど頻度と幅が増える。\n    stretch_event = rng.random((trials,n)) < np.clip(.06 + stretch_index[None,:]*.24, .05, .30)\n    stretch_power = stretch_event * rng.uniform(.12,.58,size=(trials,n)) * (.55 + stretch_index[None,:])\n    first_lap = start_draw + stretch_power\n\n    if same_line:\n        gate=1/(1+np.exp(-(first_lap-.48)*8.0))\n        score += gate * lane_ratio[None,:] * .32\n        front_idx=np.argpartition(-first_lap, min(2,n-1), axis=1)[:,:min(3,n)]\n        front_mask=np.zeros((trials,n),dtype=bool)\n        front_mask[np.arange(trials)[:,None],front_idx]=True\n        score += front_mask*(.20 + position_hold[None,:]*.35 + stable[None,:]*.15)\n        # スタート伸び上振れ車は、外枠でも前団へ入った価値を上乗せ。\n        score += front_mask * stretch_power * .42\n        gap=rng.beta(2.0,3.2,size=(trials,1))\n        opportunity=np.clip(gap-(dense_size-3)*.035,0,1)\n        chase=(~front_mask)*opportunity*(breakthrough[None,:]*.48 + rear_chase[None,:]*.27 + closing[None,:]*.20)\n        score += chase\n        bad=first_lap < .37\n        score -= bad*(dependency[None,:]*.32 + bipolar[None,:]*rng.uniform(.04,.31,size=(trials,n)))\n    else:\n        min_h=np.nanmin(handicap); front=np.isclose(handicap,min_h)\n        score[:,front] += (.12+heat*.22)*position_hold[None,front]\n        score[:,~front] += rear_chase[None,~front]*.18 + closing[None,~front]*.12\n        score += stretch_power*.20\n\n    # 中盤の突破機会。\n    finish_gap=rng.beta(2.2,2.8,size=(trials,1))\n    score += finish_gap*(closing[None,:]*.18 + breakthrough[None,:]*.16)\n\n    # 最終周のゴール前伸び。大逆転ではなく、近い相手だけを僅差で交わせるようにする。\n    goal_kick_index=np.clip(final_kick*.36 + closing*.25 + execution*.15 + current*.10 + trial_strength*.08 + rear_chase*.06,0,1)\n    goal_event=rng.random((trials,n)) < np.clip(.08 + goal_kick_index[None,:]*.22, .06, .29)\n    goal_power=goal_event*rng.uniform(.05,.26,size=(trials,n))*(.65+goal_kick_index[None,:])\n    score += goal_power\n    score += rng.normal(0,.055,size=(trials,n))\n\n    # いったん順位化し、僅差の隣接2台だけ最終周差しを追加。\n    order=np.argsort(-score,axis=1)\n    sorted_score=np.take_along_axis(score,order,axis=1)\n    for rank in range(n-1):\n        lead_idx=order[:,rank]; chase_idx=order[:,rank+1]\n        margin=sorted_score[:,rank]-sorted_score[:,rank+1]\n        chase_kick=goal_power[np.arange(trials),chase_idx]\n        lead_hold=position_hold[lead_idx]*.11 + stable[lead_idx]*.06\n        pass_mask=(margin < .22) & ((chase_kick-lead_hold) > margin*.55) & (rng.random(trials)<.52)\n        if np.any(pass_mask):\n            a=order[pass_mask,rank].copy(); b=order[pass_mask,rank+1].copy()\n            order[pass_mask,rank]=b; order[pass_mask,rank+1]=a\n\n    finish_matrix=np.zeros((n,n),dtype=np.int64)\n    for rank in range(n): finish_matrix[:,rank]=np.bincount(order[:,rank],minlength=n)\n    top3=cars[order[:,:3]]; top2=cars[order[:,:2]]\n    def make_counter(a):\n        vals,cnt=np.unique(a,axis=0,return_counts=True)\n        return Counter({tuple(map(int,v)):int(c) for v,c in zip(vals,cnt)})\n    finish_counts={int(car):Counter({rank+1:int(finish_matrix[i,rank]) for rank in range(n) if finish_matrix[i,rank]>0}) for i,car in enumerate(cars)}\n    return finish_counts,{"三連単":make_counter(top3),"三連複":make_counter(np.sort(top3,axis=1)),"2車単":make_counter(top2),"2車複":make_counter(np.sort(top2,axis=1))}\n\ndef style_header(ws, row, last_col):\n    fill = PatternFill("solid", fgColor="4472C4")\n    font = Font(color="FFFFFF", bold=True)\n    for c in range(1, last_col + 1):\n        cell = ws.cell(row, c)\n        cell.fill = fill\n        cell.font = font\n        cell.alignment = Alignment(horizontal="center")\n\n\ndef auto_width(ws):\n    for col_idx in range(1, ws.max_column + 1):\n        max_len = 0\n        for row_idx in range(1, ws.max_row + 1):\n            cell = ws.cell(row_idx, col_idx)\n            if isinstance(cell, MergedCell):\n                continue\n            if cell.value is not None:\n                max_len = max(max_len, len(str(cell.value)))\n        ws.column_dimensions[get_column_letter(col_idx)].width = min(\n            max(max_len + 2, 10), 24\n        )\n\n\ndef create_result_excel(content, filename, df, finish_counts, bet_counts, trials, track_temp=30.0):\n    wb = load_workbook(io.BytesIO(content), data_only=False)\n\n    generated_sheets = [\n        "Colab計算結果", "着順分布",\n        "三連単確率", "三連単期待値", "三連複確率", "三連複期待値",\n        "2車単確率", "2車単期待値", "2車複確率", "2車複期待値",\n    ]\n    for sheet_name in generated_sheets:\n        if sheet_name in wb.sheetnames:\n            del wb[sheet_name]\n\n    result = wb.create_sheet("Colab計算結果")\n    dist = wb.create_sheet("着順分布")\n\n    result.merge_cells("A1:AV1")\n    result["A1"] = f"AutoRaceAI Ver12.2 走路環境結果（走路温度 {track_temp:.0f}℃）"\n    result["A1"].fill = PatternFill("solid", fgColor="17365D")\n    result["A1"].font = Font(color="FFFFFF", bold=True, size=14)\n    result["A1"].alignment = Alignment(horizontal="center")\n\n    output_cols = [\n        "車", "選手名", "ハンデ", "試走換算", "直近5差", "平均ST",\n        "ST標準偏差", "ST安定性", "同ハンデ差", "開催場差", "走路差", "着順指数", "勝率",\n        "連対率", "3着以内率", "平均着順", "着順安定度",\n        "審査P", "現ランク", "上昇度", "再現性", "今回ハンデ勝率",\n        "試走点", "試走単発救済点", "近況主役点", "最近点", "ST点", "ハンデ点", "開催場点",\n        "走路点", "位置点", "内枠展開点", "最内勝ち切り点", "着順点", "勝率点", "連対点",\n        "上昇点", "再現性点", "ハンデ勝率点", "3着以内率点", "平均着順点", "着順安定度点",\n        "審査Pランク点", "試走信頼度", "レース巧者指数", "レース巧者点",\n        "逃げ成功率", "内枠残存率", "位置活用率", "同ハンデST優位度", "レースタイプ", "改良総合点",\n        "改良順位", "前走ハンデ", "直近5走平均ハンデ", "ハンデ改善量", "ハンデ変化点", "同ハンデ人数",\n        "同ハンデ外順", "外枠不利補正", "直近3走好調補正", "巻き返し余地補正", "実戦再現力補正",\n        "勝負余力補正", "内側高信頼補正", "近況単独過大補正", "選手タイプ総合補正",\n        "中位粘り率", "中位粘り補正", "条件改善再現補正", "中位内容総合補正",\n        "上昇カーブ指数", "上昇カーブ補正", "終盤指数", "終盤力補正", "相手レベル耐性補正",\n        "Ver10_6総合補正", "選手タイプVer2", "改善後総合点", "改善後順位",\n    ]\n    output_cols = [c for c in output_cols if c in df.columns]\n    # シミュレーションと同じ定義で安定指数を表示する。\n    finish_sd = pd.to_numeric(df.get("着順標準偏差", pd.Series(1.5, index=df.index)), errors="coerce").fillna(1.5).clip(0.8, 3.0)\n    finish_consistency = (1.0 - (finish_sd - 0.8) / 2.2).clip(0.0, 1.0)\n    df = df.copy()\n    df["走路安定指数"] = (\n        0.42 * finish_consistency\n        + 0.31 * pd.to_numeric(df.get("ST安定性", 0.5), errors="coerce").fillna(0.5).clip(0, 1)\n        + 0.17 * pd.to_numeric(df.get("試走信頼度", 0.5), errors="coerce").fillna(0.5).clip(0.2, 1)\n        + 0.10 * pd.to_numeric(df.get("近況信頼度", 0.5), errors="coerce").fillna(0.5).clip(0, 1)\n    ).clip(0, 1)\n    df["走路安定タイプ"] = pd.cut(df["走路安定指数"], [-0.01, 0.48, 0.66, 1.01], labels=["ムラ型", "標準型", "安定型"]).astype(str)\n    output_cols += ["走路安定指数", "走路安定タイプ"]\n\n    for c, name in enumerate(output_cols, 1): result.cell(3, c, name)\n    style_header(result, 3, len(output_cols))\n    for r, (_, row) in enumerate(df.iterrows(), 4):\n        for c, name in enumerate(output_cols, 1):\n            value = row.get(name)\n            result.cell(r, c, None if pd.isna(value) else value)\n    percent_names = {"勝率", "連対率", "3着以内率", "今回ハンデ勝率", "走路安定指数"}\n    for c, name in enumerate(output_cols, 1):\n        if name in percent_names:\n            for r in range(4, 4 + len(df)): result.cell(r, c).number_format = "0.0%"\n\n    dist.merge_cells("A1:J1")\n    dist["A1"] = "着順分布"\n    dist["A1"].fill = PatternFill("solid", fgColor="17365D")\n    dist["A1"].font = Font(color="FFFFFF", bold=True, size=14)\n    dist["A1"].alignment = Alignment(horizontal="center")\n    dist_headers = ["車", "選手名"] + [f"{rank}着率" for rank in range(1, 9)]\n    for c, name in enumerate(dist_headers, 1): dist.cell(3, c, name)\n    style_header(dist, 3, len(dist_headers))\n    for r, (_, row) in enumerate(df.iterrows(), 4):\n        car = int(row["車"])\n        values = [car, row["選手名"]] + [finish_counts[car][rank] / trials for rank in range(1, 9)]\n        for c, value in enumerate(values, 1):\n            dist.cell(r, c, value)\n            if c >= 3: dist.cell(r, c).number_format = "0.0%"\n\n    def make_probability_and_ev(ticket_name, counter):\n        prob = wb.create_sheet(f"{ticket_name}確率")\n        ev = wb.create_sheet(f"{ticket_name}期待値")\n        ordered = ticket_name in ("三連単", "2車単")\n        pick_count = 3 if ticket_name.startswith("三連") else 2\n        labels = (["1着", "2着", "3着"] if pick_count == 3 and ordered else\n                  ["車1", "車2", "車3"] if pick_count == 3 else\n                  ["1着", "2着"] if ordered else ["車1", "車2"])\n        prob.merge_cells(start_row=1, start_column=1, end_row=1, end_column=pick_count + 3)\n        prob["A1"] = f"{ticket_name}確率（Ver11.1走路環境補正）"\n        prob["A1"].fill = PatternFill("solid", fgColor="17365D")\n        prob["A1"].font = Font(color="FFFFFF", bold=True, size=14)\n        headers = ["順位"] + labels + ["組合せ", "確率"]\n        for c, name in enumerate(headers, 1): prob.cell(3, c, name)\n        style_header(prob, 3, len(headers))\n        sorted_items = sorted(counter.items(), key=lambda x: x[1], reverse=True)\n        for rank, (combo, count) in enumerate(sorted_items, 1):\n            row = rank + 3\n            values = [rank] + list(combo) + ["-".join(map(str, combo)), count / trials]\n            for c, value in enumerate(values, 1): prob.cell(row, c, value)\n            prob.cell(row, len(headers)).number_format = "0.000%"\n\n        ev.merge_cells(start_row=1, start_column=1, end_row=1, end_column=pick_count + 7)\n        ev["A1"] = f"{ticket_name}期待値（確率 × 実オッズ）"\n        ev["A1"].fill = PatternFill("solid", fgColor="17365D")\n        ev["A1"].font = Font(color="FFFFFF", bold=True, size=14)\n        ev_headers = ["順位"] + labels + ["組合せ", "補正確率", "適正オッズ", "実オッズ入力", "期待値倍率", "期待値判定"]\n        for c, name in enumerate(ev_headers, 1): ev.cell(3, c, name)\n        style_header(ev, 3, len(ev_headers))\n        prob_col = 2 + pick_count + 1\n        fair_col = prob_col + 1; odds_col = fair_col + 1; value_col = odds_col + 1; judge_col = value_col + 1\n        from openpyxl.utils import get_column_letter\n        for rank, (combo, count) in enumerate(sorted_items, 1):\n            row = rank + 3; p = count / trials\n            values = [rank] + list(combo) + ["-".join(map(str, combo)), p, (1/p if p > 0 else None)]\n            for c, value in enumerate(values, 1): ev.cell(row, c, value)\n            ev.cell(row, prob_col).number_format = "0.000%"\n            ev.cell(row, fair_col).number_format = "0.00"\n            ev.cell(row, odds_col).number_format = "0.00"\n            ev.cell(row, value_col, f\'=IF({get_column_letter(odds_col)}{row}="","",{get_column_letter(prob_col)}{row}*{get_column_letter(odds_col)}{row})\')\n            ev.cell(row, value_col).number_format = "0.000"\n            ev.cell(row, judge_col, f\'=IF({get_column_letter(value_col)}{row}="","",IF({get_column_letter(value_col)}{row}>=1,"期待値あり","期待値不足"))\')\n        return prob, ev\n\n    sheets = [result, dist]\n    for ticket in ["三連単", "三連複", "2車単", "2車複"]:\n        sheets.extend(make_probability_and_ev(ticket, bet_counts[ticket]))\n    for ws in sheets:\n        ws.freeze_panes = "A4"\n        auto_width(ws)\n\n    path = f"/content/{Path(filename).stem}_AutoRaceAI_Ver12_0_位置依存_二極化タイプ.xlsx"\n    wb.save(path)\n    return path\n\n\ndef run_model(content, filename, trials, seed, track_temp=30.0):\n    wb = load_workbook(io.BytesIO(content), data_only=True)\n\n    required = ["レース予測", "設定"] + [f"選手{i}" for i in range(1, 9)]\n    missing = [name for name in required if name not in wb.sheetnames]\n    if missing:\n        raise ValueError("不足シート: " + ", ".join(missing))\n\n    race = read_race(wb["レース予測"])\n    settings = read_settings(wb["設定"])\n\n    metrics = []\n    for car in range(1, 9):\n        ws = wb[f"選手{car}"]\n        current = current_player(ws)\n        history = read_history(ws)\n        metrics.append(\n            player_metrics(car, current, history, race, settings)\n        )\n\n    df = calculate_excel_model(metrics, settings)\n\n    # Ver12.2 診断列。シミュレーションと同じ考え方で、スタート一気と先行転換を表示する。\n    st_v = pd.to_numeric(df["平均ST"], errors="coerce").fillna(0.20).to_numpy(float)\n    trial_v = pd.to_numeric(df["試走換算"], errors="coerce").fillna(pd.to_numeric(df["試走換算"], errors="coerce").median()).to_numpy(float)\n    def _rank_lower(v):\n        v = np.asarray(v, dtype=float); idx = np.argsort(v, kind="stable")\n        out = np.zeros(len(v), dtype=float)\n        out[idx] = 0.5 if len(v) <= 1 else 1.0 - np.arange(len(v), dtype=float) / (len(v) - 1.0)\n        return out\n    st_stable_v = pd.to_numeric(df["ST安定性"], errors="coerce").fillna(0.50).clip(0, 1).to_numpy(float)\n    execution_v = pd.to_numeric(df["実戦変換力"], errors="coerce").fillna(0.50).clip(0, 1).to_numpy(float) if "実戦変換力" in df.columns else pd.to_numeric(df["レース巧者指数"], errors="coerce").fillna(0.50).clip(0, 1).to_numpy(float)\n    traffic_v = pd.to_numeric(df["レース巧者指数"], errors="coerce").fillna(0.50).clip(0, 1).to_numpy(float)\n    road_v = pd.to_numeric(df.get("走路安定指数", pd.Series(0.50, index=df.index)), errors="coerce").fillna(0.50).clip(0, 1).to_numpy(float)\n    trust_v = pd.to_numeric(df["試走信頼度"], errors="coerce").fillna(0.50).clip(0, 1).to_numpy(float)\n    df["スタート一気指数"] = np.clip(_rank_lower(st_v) * 0.46 + st_stable_v * 0.19 + _rank_lower(trial_v) * 0.22 + execution_v * 0.13, 0, 1)\n    df["先行転換力"] = np.clip(df["スタート一気指数"].to_numpy(float) * 0.43 + st_stable_v * 0.13 + execution_v * 0.18 + traffic_v * 0.14 + road_v * 0.08 + trust_v * 0.04, 0, 1)\n    same_line_diag = bool(len(df) >= 3 and pd.to_numeric(df["ハンデ"], errors="coerce").max() - pd.to_numeric(df["ハンデ"], errors="coerce").min() < 1e-9)\n    if same_line_diag:\n        inner_ratio_diag = 1.0 - (pd.to_numeric(df["車"], errors="coerce").fillna(1.0).to_numpy(float) - 1.0) / max(1.0, len(df) - 1.0)\n        df["隊列残存指数"] = np.clip(inner_ratio_diag * (df["スタート一気指数"].to_numpy(float) * 0.48 + df["先行転換力"].to_numpy(float) * 0.52), 0, 1)\n    else:\n        df["隊列残存指数"] = 0.0\n\n    # Ver12.2: 能力が高い選手は混戦の壁を越えられるよう、全選手共通の集団突破力を作る。\n    # 内枠・車番はここには入れず、実戦能力・終盤・交通処理・勝負強さ・後方追上げで構成する。\n    def _norm01_col(col, default=50.0):\n        v = pd.to_numeric(df.get(col, pd.Series(default, index=df.index)), errors="coerce").fillna(default).to_numpy(float)\n        if np.nanmax(v) <= 1.5:\n            return np.clip(v, 0, 1)\n        lo, hi = np.nanpercentile(v, 10), np.nanpercentile(v, 90)\n        if hi - lo < 1e-9:\n            return np.full(len(v), 0.5)\n        return np.clip((v - lo) / (hi - lo), 0, 1)\n\n    practical_v = _norm01_col("実戦能力点")\n    closing_v = _norm01_col("終盤指数")\n    traffic_v2 = _norm01_col("レース巧者指数")\n    clutch_v = _norm01_col("勝負強さ点")\n    rear_v = np.clip(pd.to_numeric(df.get("後方追上げ指数", pd.Series(0.35, index=df.index)), errors="coerce").fillna(0.35).to_numpy(float), 0, 1)\n    trial_v2 = _norm01_col("当日レース指数")\n    stable_v2 = np.clip(pd.to_numeric(df.get("安定上位指数", pd.Series(0.50, index=df.index)), errors="coerce").fillna(0.50).to_numpy(float), 0, 1)\n    recent_poor_v2 = np.clip(pd.to_numeric(df.get("直近5走生凡走率", pd.Series(0.25, index=df.index)), errors="coerce").fillna(0.25).to_numpy(float), 0, 1)\n\n    raw_breakthrough = (\n        practical_v * 0.23 + closing_v * 0.20 + traffic_v2 * 0.18\n        + clutch_v * 0.10 + rear_v * 0.13 + trial_v2 * 0.08\n        + stable_v2 * 0.08\n    )\n    # 能力差を0～1へ広げすぎず、強い選手でも「機会がなければ抜けない」範囲へ圧縮。\n    df["混戦突破適性"] = np.clip(\n        0.26 + raw_breakthrough * 0.50 - recent_poor_v2 * 0.08,\n        0.18, 0.78\n    )\n    df["集団突破力"] = df["混戦突破適性"]\n\n    # Ver12.3 診断列：ST反応とは別のスタート伸びと、最終周の伸びを表示。\n    _trial_s = _norm01_col("当日レース指数")\n    _exec_s = _norm01_col("実戦能力点")\n    _ststab_s = np.clip(pd.to_numeric(df.get("ST安定性", pd.Series(.5,index=df.index)), errors="coerce").fillna(.5).to_numpy(float),0,1)\n    _current_s = np.clip(pd.to_numeric(df.get("当日状態指数", pd.Series(.5,index=df.index)), errors="coerce").fillna(.5).to_numpy(float),0,1)\n    _recent_s = np.clip(pd.to_numeric(df.get("近況信頼度", pd.Series(.5,index=df.index)), errors="coerce").fillna(.5).to_numpy(float),0,1)\n    df["スタート伸び指数"] = np.clip(_trial_s*.30 + _exec_s*.25 + _ststab_s*.16 + _current_s*.14 + _recent_s*.10 + traffic_v2*.05,0,1)\n    _final_s = _norm01_col("終盤指数")\n    df["ゴール前伸び指数"] = np.clip(_final_s*.36 + closing_v*.25 + _exec_s*.15 + _current_s*.10 + _trial_s*.08 + rear_v*.06,0,1)\n\n    finish_counts, bet_counts = simulate(df, trials, seed, track_temp=track_temp)\n    output = create_result_excel(\n        content, filename, df, finish_counts, bet_counts, trials, track_temp=track_temp\n    )\n    return df, bet_counts, output\n\n\nupload = widgets.FileUpload(\n    accept=".xlsx",\n    multiple=False,\n    description="Excelを選択",\n    button_style="info",\n)\n\ntrial_slider = widgets.IntSlider(\n    value=10000,\n    min=1000,\n    max=100000,\n    step=1000,\n    description="試行回数（超高速10000）",\n    continuous_update=False,\n    layout=widgets.Layout(width="600px"),\n    style={"description_width": "initial"},\n)\n\nseed_input = widgets.IntText(\n    value=20260719,\n    description="乱数シード",\n    style={"description_width": "initial"},\n)\n\ntrack_temp_slider = widgets.FloatSlider(\n    value=30.0,\n    min=15.0,\n    max=70.0,\n    step=1.0,\n    description="走路温度(℃)",\n    continuous_update=False,\n    layout=widgets.Layout(width="600px"),\n    style={"description_width": "initial"},\n)\n\nrun_button = widgets.Button(\n    description="Ver12.3 高速計算",\n    button_style="success",\n    icon="calculator",\n    layout=widgets.Layout(width="280px", height="44px"),\n)\n\noutput_area = widgets.Output()\n\n\ndef clicked(_):\n    with output_area:\n        _hidden_clear_output()\n\n        try:\n            import time\n            _calc_started = time.perf_counter()\n            filename, content = uploaded_file(upload.value)\n            if not content:\n                print("先にExcelを選択してください。")\n                return\n\n            heat = float(np.clip((float(track_temp_slider.value) - 50.0) / 8.0, 0.0, 1.0))\n            chaos = float(np.clip((float(track_temp_slider.value) - 48.0) / 16.0, 0.0, 1.0))\n            heat_label = "通常" if heat <= 0 else ("やや前残り" if heat < 0.50 else "前残り強め")\n            if chaos >= 0.65:\n                heat_label += "・荒れ強め"\n            elif chaos > 0:\n                heat_label += "・荒れ注意"\n            print(f"AutoRaceAI Ver12.3 高速版を計算中（走路温度 {track_temp_slider.value:.0f}℃・{heat_label}）...")\n            df, bet_counts, output = run_model(\n                content,\n                filename,\n                min(int(trial_slider.value), 20000),\n                int(seed_input.value),\n                float(track_temp_slider.value),\n            )\n            globals()["LATEST_PREDICTION_DF"] = df.copy()\n            globals()["LATEST_BET_COUNTS"] = bet_counts\n            globals()["LATEST_TRACK_TEMP"] = float(track_temp_slider.value)\n            globals()["LATEST_PREDICTION_FILENAME"] = filename\n\n            requested_display_cols = [\n                "車", "選手名", "レースタイプ", "逃げ成功率", "内枠残存率", "位置活用率",\n                "同ハンデST優位度", "試走信頼度", "レース巧者指数",\n                "直近5走生3着内率", "直近5走生凡走率", "近況信頼度", "短期上振れ抑制",\n                "近況信頼度補正", "直近内容補正", "速度実戦変換補正",\n                "基礎スピード点", "実戦能力点", "勝負強さ点", "展開適性点",\n                "選手タイプ", "選手タイプVer2", "上昇カーブ指数", "上昇カーブ補正",\n                "終盤指数", "終盤力補正", "相手レベル耐性補正",\n                "中位粘り率", "中位粘り補正", "条件改善再現補正",\n                "スタート一気指数", "スタート伸び指数", "先行転換力", "隊列残存指数", "ゴール前伸び指数", "安定上位指数", "安定上位評価補正", "混戦突破適性", "集団突破力", "好位置維持指数", "後方追上げ指数", "位置依存指数", "二極化指数", "自己比試走指数", "自己試走変化秒", "当日状態指数", "試走突出度", "爆発指数", "爆発能力補正", "前団主導指数", "当日勝ち切り指数", "後方追走不成立リスク", "当日レース指数", "基礎履歴総合点", "改良総合点", "改善後総合点", "改善後順位"\n            ]\n            # Ver10で廃止・変更された旧列があっても表示処理を止めない\n            display_cols = [col for col in requested_display_cols if col in df.columns]\n            missing_display_cols = [col for col in requested_display_cols if col not in df.columns]\n            if missing_display_cols:\n                print("表示対象から除外した未生成列:", ", ".join(missing_display_cols))\n\n            _hidden_display(\n                df.loc[:, display_cols]\n                .sort_values(["改善後順位", "車"])\n                .reset_index(drop=True)\n                .style.format({\n                    "逃げ成功率": "{:.1%}",\n                    "内枠残存率": "{:.1%}",\n                    "位置活用率": "{:.1%}",\n                    "同ハンデST優位度": "{:.3f}",\n                    "試走信頼度": "{:.3f}",\n                    "レース巧者指数": "{:.3f}",\n                    "スタート一気指数": "{:.3f}",\n                    "先行転換力": "{:.3f}",\n                    "当日状態指数": "{:.3f}",\n                    "試走突出度": "{:.3f}",\n                    "爆発指数": "{:.3f}",\n                    "爆発能力補正": "{:.3f}",\n                    "当日レース指数": "{:.3f}",\n                    "基礎履歴総合点": "{:.3f}",\n                    "直近5走生3着内率": "{:.1%}",\n                    "直近5走生凡走率": "{:.1%}",\n                    "近況信頼度": "{:.3f}",\n                    "短期上振れ抑制": "{:.3f}",\n                    "近況信頼度補正": "{:.3f}",\n                    "直近内容補正": "{:.3f}",\n                    "速度実戦変換補正": "{:.3f}",\n                    "改良総合点": "{:.3f}",\n                    "ハンデ変化点": "{:.3f}",\n                    "改善後総合点": "{:.3f}",\n                })\n            )\n\n            print("\\n三連単確率 上位10通り")\n            for rank, (combo, count) in enumerate(\n                sorted(bet_counts["三連単"].items(), key=lambda x: x[1], reverse=True)[:10],\n                1,\n            ):\n                probability = count / int(trial_slider.value) * 100\n                print(\n                    f"{rank:>2}. {combo[0]}-{combo[1]}-{combo[2]} "\n                    f"{probability:.3f}%"\n                )\n\n            for ticket in ["三連複", "2車単", "2車複"]:\n                print(f"\\n{ticket}確率 上位5通り")\n                for rank, (combo, count) in enumerate(\n                    sorted(bet_counts[ticket].items(), key=lambda x: x[1], reverse=True)[:5], 1\n                ):\n                    probability = count / int(trial_slider.value) * 100\n                    print(f"{rank:>2}. {\'-\'.join(map(str, combo))} {probability:.3f}%")\n\n            _elapsed = time.perf_counter() - _calc_started\n            print(f"\\n計算時間: {_elapsed:.1f}秒")\n            print("結果Excelをダウンロードします。")\n            files.download(output)\n\n        except Exception as exc:\n            print("エラー:", exc)\n\n\nrun_button.on_click(clicked)\n\n_hidden_display(\n    widgets.VBox([\n        upload,\n        trial_slider,\n        seed_input,\n        track_temp_slider,\n        run_button,\n        output_area,\n    ])\n)\n\n# ============================================================\n# Ver13.1 周回順位貼り付け・展開分析\n# ============================================================\nLAP_STATUS_GOAL = \'ゴール線通過\'\n\ndef init_lap_tables():\n    mount_and_init_db()\n    with sqlite3.connect(DB_PATH) as con:\n        con.executescript("""\n        CREATE TABLE IF NOT EXISTS races (\n            race_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            race_key TEXT NOT NULL UNIQUE,\n            race_date TEXT, venue TEXT, race_no TEXT, race_name TEXT,\n            distance_m INTEGER, total_laps INTEGER,\n            weather TEXT, surface_condition TEXT,\n            track_temp REAL, air_temp REAL, humidity REAL,\n            start_time_text TEXT, source TEXT,\n            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n            updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP\n        );\n        CREATE TABLE IF NOT EXISTS lap_history (\n            lap_history_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            race_id INTEGER NOT NULL,\n            lap_index INTEGER NOT NULL,\n            lap_label TEXT NOT NULL,\n            position INTEGER NOT NULL,\n            car_no INTEGER NOT NULL,\n            player_id INTEGER,\n            is_goal INTEGER NOT NULL DEFAULT 0,\n            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n            UNIQUE(race_id, lap_index, position),\n            UNIQUE(race_id, lap_index, car_no),\n            FOREIGN KEY(race_id) REFERENCES races(race_id),\n            FOREIGN KEY(player_id) REFERENCES players(player_id)\n        );\n        CREATE TABLE IF NOT EXISTS lap_features (\n            feature_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            race_id INTEGER NOT NULL,\n            car_no INTEGER NOT NULL,\n            player_id INTEGER,\n            first_lap_pos INTEGER, final_pos INTEGER, best_pos INTEGER, worst_pos INTEGER,\n            net_gain INTEGER, overtakes INTEGER, passed_by INTEGER,\n            front_pack_laps INTEGER, front_pack_rate REAL,\n            late_gain INTEGER, late_peak_gain INTEGER,\n            lead_laps INTEGER, position_variance REAL,\n            finish_vs_first INTEGER,\n            UNIQUE(race_id, car_no),\n            FOREIGN KEY(race_id) REFERENCES races(race_id),\n            FOREIGN KEY(player_id) REFERENCES players(player_id)\n        );\n        CREATE INDEX IF NOT EXISTS idx_lap_player ON lap_history(player_id, race_id, lap_index);\n        CREATE INDEX IF NOT EXISTS idx_lap_race ON lap_history(race_id, lap_index, position);\n        """)\n\ndef _normalize_paste(text):\n    return str(text or \'\').replace(\'\\r\\n\',\'\\n\').replace(\'\\r\',\'\\n\').replace(\'\\u3000\',\' \')\n\ndef parse_lap_paste(text):\n    text=_normalize_paste(text)\n    lines=[x.strip() for x in text.split(\'\\n\') if x.strip()]\n    rows=[]\n    for line in lines:\n        parts=[p for p in re.split(r\'[\\t ,，]+\',line) if p]\n        if len(parts)<9 or not (parts[0] in [\'ゴール線通過\',\'ゴール\'] or re.fullmatch(r\'\\d+周目\',parts[0])):\n            continue\n        label=parts[0]\n        cars=[int(x) for x in parts[1:9] if re.fullmatch(r\'\\d+\',x)]\n        if len(cars)!=8 or sorted(cars)!=list(range(1,9)):\n            raise ValueError(f\'「{line}」の車番が1～8の並びになっていません\')\n        is_goal=label in [\'ゴール線通過\',\'ゴール\']\n        lap_no=10**6 if is_goal else int(re.search(r\'\\d+\',label).group())\n        rows.append({\'label\':LAP_STATUS_GOAL if is_goal else label,\'lap_no\':lap_no,\'is_goal\':is_goal,\'cars\':cars})\n    if not rows:\n        raise ValueError(\'周回順位表を読み取れませんでした。見出しを含めてそのまま貼り付けてください。\')\n    unique={r[\'label\']:r for r in rows}\n    normal=sorted([r for r in unique.values() if not r[\'is_goal\']],key=lambda r:r[\'lap_no\'])\n    goals=[r for r in unique.values() if r[\'is_goal\']]\n    ordered=normal+goals\n    for i,r in enumerate(ordered,1): r[\'lap_index\']=i\n    meta={}\n    m=re.search(r\'(\\d{4})年(\\d{1,2})月(\\d{1,2})日\',text)\n    if m: meta[\'race_date\']=f\'{int(m.group(1)):04d}-{int(m.group(2)):02d}-{int(m.group(3)):02d}\'\n    m=re.search(r\'([^\\n]*?(?:優勝戦|一般戦|選抜戦|準決勝戦|特別選抜戦|予選|最終予選|二次予選))\\s*(\\d+)m\\s*\\((\\d+)周\\)\',text)\n    if m:\n        meta[\'race_name\']=m.group(1).strip(); meta[\'distance_m\']=int(m.group(2)); meta[\'total_laps\']=int(m.group(3))\n    else:\n        m=re.search(r\'(\\d+)m\\s*\\((\\d+)周\\)\',text)\n        if m: meta[\'distance_m\']=int(m.group(1)); meta[\'total_laps\']=int(m.group(2))\n    patterns={\n        \'weather\':r\'天候[：:]\\s*([^\\s\u3000]+)\',\n        \'surface_condition\':r\'走路状況[：:]\\s*([^\\s\u3000]+)\',\n        \'track_temp\':r\'走路温度[：:]\\s*([\\d.]+)\',\n        \'air_temp\':r\'気温[：:]\\s*([\\d.]+)\',\n        \'humidity\':r\'湿度[：:]\\s*([\\d.]+)\',\n        \'start_time_text\':r\'発走時間\\s*([0-2]?\\d:[0-5]\\d)\'\n    }\n    for k,p in patterns.items():\n        m=re.search(p,text)\n        if m: meta[k]=float(m.group(1)) if k in [\'track_temp\',\'air_temp\',\'humidity\'] else m.group(1)\n    return ordered,meta\n\ndef extract_result_mapping(content):\n    if not content: return {},{}\n    wb=load_workbook(io.BytesIO(content),data_only=True)\n    mapping={}; meta={}\n    if \'結果入力\' in wb.sheetnames:\n        ws=wb[\'結果入力\']; h={_s(ws.cell(1,c).value):c for c in range(1,ws.max_column+1)}\n        for r in range(2,ws.max_row+1):\n            name=_s(ws.cell(r,h.get(\'選手名\',1)).value)\n            if name: mapping[r-1]=name\n        if ws.max_row>=2:\n            for col,key in [(\'開催日\',\'race_date\'),(\'開催場\',\'venue\'),(\'レース\',\'race_no\')]:\n                if col in h:\n                    v=ws.cell(2,h[col]).value\n                    meta[key]=_date_text(v) if col==\'開催日\' else _s(v)\n    else:\n        players=[w for w in wb.worksheets if re.fullmatch(r\'選手\\d+\',w.title)]\n        for i,w in enumerate(players,1): mapping[i]=_player_name(w)\n    return mapping,meta\n\ndef _race_key(meta, rows):\n    core=[_s(meta.get(\'race_date\')),_s(meta.get(\'venue\')),_s(meta.get(\'race_no\')),_s(meta.get(\'race_name\')),\n          _s(meta.get(\'distance_m\')),_s(meta.get(\'total_laps\'))]\n    if not any(core[:4]): core.append(\'|\'.join(\',\'.join(map(str,r[\'cars\'])) for r in rows))\n    return hashlib.sha256(\'|\'.join(core).encode()).hexdigest()\n\ndef calculate_lap_features(rows):\n    positions={c:[] for c in range(1,9)}\n    for r in rows:\n        for pos,c in enumerate(r[\'cars\'],1): positions[c].append(pos)\n    n=len(rows); late_start=max(0,n-3); feats={}\n    for c,p in positions.items():\n        gains=[p[i-1]-p[i] for i in range(1,len(p))]\n        front=sum(x<=3 for x in p)\n        feats[c]={\n            \'first_lap_pos\':p[0],\'final_pos\':p[-1],\'best_pos\':min(p),\'worst_pos\':max(p),\n            \'net_gain\':p[0]-p[-1],\'overtakes\':sum(max(0,x) for x in gains),\n            \'passed_by\':sum(max(0,-x) for x in gains),\'front_pack_laps\':front,\n            \'front_pack_rate\':round(front/len(p),4),\'late_gain\':p[late_start]-p[-1],\n            \'late_peak_gain\':sum(max(0,p[i-1]-p[i]) for i in range(max(1,late_start+1),len(p))),\n            \'lead_laps\':sum(x==1 for x in p),\'position_variance\':round(float(np.var(p)),4),\n            \'finish_vs_first\':p[0]-p[-1]\n        }\n    return feats\n\ndef save_lap_data(rows, meta, player_mapping=None, source=\'貼り付け\'):\n    init_lap_tables(); player_mapping=player_mapping or {}; key=_race_key(meta,rows)\n    with sqlite3.connect(DB_PATH) as con:\n        con.execute("""INSERT INTO races(race_key,race_date,venue,race_no,race_name,distance_m,total_laps,\n          weather,surface_condition,track_temp,air_temp,humidity,start_time_text,source)\n          VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?)\n          ON CONFLICT(race_key) DO UPDATE SET race_date=excluded.race_date,venue=excluded.venue,\n          race_no=excluded.race_no,race_name=excluded.race_name,distance_m=excluded.distance_m,\n          total_laps=excluded.total_laps,weather=excluded.weather,surface_condition=excluded.surface_condition,\n          track_temp=excluded.track_temp,air_temp=excluded.air_temp,humidity=excluded.humidity,\n          start_time_text=excluded.start_time_text,source=excluded.source,updated_at=CURRENT_TIMESTAMP""",\n          (key,meta.get(\'race_date\'),meta.get(\'venue\'),meta.get(\'race_no\'),meta.get(\'race_name\'),meta.get(\'distance_m\'),\n           meta.get(\'total_laps\'),meta.get(\'weather\'),meta.get(\'surface_condition\'),meta.get(\'track_temp\'),\n           meta.get(\'air_temp\'),meta.get(\'humidity\'),meta.get(\'start_time_text\'),source))\n        race_id=con.execute(\'SELECT race_id FROM races WHERE race_key=?\',(key,)).fetchone()[0]\n        pids={int(car):_ensure_player(con,name) for car,name in player_mapping.items() if name}\n        con.execute(\'DELETE FROM lap_history WHERE race_id=?\',(race_id,)); con.execute(\'DELETE FROM lap_features WHERE race_id=?\',(race_id,))\n        for r in rows:\n            for pos,car in enumerate(r[\'cars\'],1):\n                con.execute("INSERT INTO lap_history(race_id,lap_index,lap_label,position,car_no,player_id,is_goal) VALUES(?,?,?,?,?,?,?)",\n                            (race_id,r[\'lap_index\'],r[\'label\'],pos,car,pids.get(car),int(r[\'is_goal\'])))\n        feats=calculate_lap_features(rows)\n        for car,f in feats.items():\n            con.execute("""INSERT INTO lap_features(race_id,car_no,player_id,first_lap_pos,final_pos,best_pos,worst_pos,\n                net_gain,overtakes,passed_by,front_pack_laps,front_pack_rate,late_gain,late_peak_gain,\n                lead_laps,position_variance,finish_vs_first) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",\n                (race_id,car,pids.get(car),f[\'first_lap_pos\'],f[\'final_pos\'],f[\'best_pos\'],f[\'worst_pos\'],f[\'net_gain\'],\n                 f[\'overtakes\'],f[\'passed_by\'],f[\'front_pack_laps\'],f[\'front_pack_rate\'],f[\'late_gain\'],f[\'late_peak_gain\'],\n                 f[\'lead_laps\'],f[\'position_variance\'],f[\'finish_vs_first\']))\n    return race_id,feats\n\ndef lap_analysis_dataframe(feats, mapping=None):\n    mapping=mapping or {}; data=[]\n    for c,f in feats.items():\n        typ=[]\n        if f[\'first_lap_pos\']<=3 and f[\'front_pack_rate\']>=0.6: typ.append(\'前団維持\')\n        if f[\'net_gain\']>=3: typ.append(\'追上げ\')\n        if f[\'late_gain\']>=2: typ.append(\'終盤伸び\')\n        if f[\'first_lap_pos\']<=3 and f[\'final_pos\']>=6: typ.append(\'先行失速\')\n        if f[\'position_variance\']>=3: typ.append(\'展開変動大\')\n        data.append({\'車番\':c,\'選手名\':mapping.get(c,\'\'),\'1周目\':f[\'first_lap_pos\'],\'最終\':f[\'final_pos\'],\n                     \'最高順位\':f[\'best_pos\'],\'最低順位\':f[\'worst_pos\'],\'純上昇\':f[\'net_gain\'],\n                     \'追抜き量\':f[\'overtakes\'],\'被追抜き量\':f[\'passed_by\'],\'前団率\':f[\'front_pack_rate\'],\n                     \'終盤上昇\':f[\'late_gain\'],\'先頭周回\':f[\'lead_laps\'],\'展開タイプ\':\'・\'.join(typ) or \'中間型\'})\n    return pd.DataFrame(data).sort_values(\'最終\').reset_index(drop=True)\n\ndef player_lap_profile(name):\n    init_lap_tables()\n    with sqlite3.connect(DB_PATH) as con:\n        return pd.read_sql_query("""SELECT COUNT(*) AS レース数,\n          ROUND(AVG(first_lap_pos),2) AS 一周目平均順位, ROUND(AVG(final_pos),2) AS ゴール平均順位,\n          ROUND(AVG(net_gain),2) AS 平均順位上昇, ROUND(AVG(front_pack_rate)*100,1) AS 前団滞在率,\n          ROUND(AVG(late_gain),2) AS 終盤平均上昇, ROUND(AVG(overtakes),2) AS 平均追抜き量,\n          ROUND(AVG(CASE WHEN first_lap_pos<=3 AND final_pos<=3 THEN 1.0 ELSE 0 END)*100,1) AS 好位置維持率,\n          ROUND(AVG(CASE WHEN first_lap_pos>=6 AND final_pos<=3 THEN 1.0 ELSE 0 END)*100,1) AS 後方三着内率\n          FROM lap_features f JOIN players p ON p.player_id=f.player_id WHERE p.player_name=?""",con,params=(name,))\n\ninit_lap_tables()\nlap_excel_upload=widgets.FileUpload(accept=\'.xlsx\',multiple=False,description=\'結果Excel（任意）\')\nlap_paste=widgets.Textarea(value=\'\',placeholder=\'公式ページの「周回・順位」からレース情報まで、そのまま貼り付け\',\n    description=\'周回表:\',layout=widgets.Layout(width=\'100%\',height=\'330px\'),style={\'description_width\':\'70px\'})\nvenue_box=widgets.Text(description=\'開催場:\',placeholder=\'Excelがない場合に入力\')\nrace_no_box=widgets.Text(description=\'レース:\',placeholder=\'例：12R\')\nsave_lap_button=widgets.Button(description=\'周回データを解析・DB保存\',button_style=\'success\',icon=\'database\')\nlap_out=widgets.Output()\n\ndef on_save_laps(_):\n    with lap_out:\n        _hidden_clear_output()\n        try:\n            rows,meta=parse_lap_paste(lap_paste.value)\n            fname,xbytes=file_bytes(lap_excel_upload)\n            mapping,xmeta=extract_result_mapping(xbytes) if xbytes else ({},{})\n            for k,v in xmeta.items():\n                if v and not meta.get(k): meta[k]=v\n            if venue_box.value.strip(): meta[\'venue\']=venue_box.value.strip()\n            if race_no_box.value.strip(): meta[\'race_no\']=race_no_box.value.strip()\n            race_id,feats=save_lap_data(rows,meta,mapping,source=fname or \'公式周回表貼付\')\n            print(f\'保存完了：race_id={race_id} / {len(rows)}地点 × 8選手\')\n            print(\'抽出したレース情報:\',meta)\n            _hidden_display(lap_analysis_dataframe(feats,mapping))\n            if not mapping: print(\'※結果入力Excelも指定すると、車番と選手名を結び付けて保存できます。\')\n        except Exception as e: print(\'エラー:\',e)\n\nsave_lap_button.on_click(on_save_laps)\n_hidden_display(widgets.HTML(\'<h3>⑤ 周回順位表をそのまま貼り付け</h3><p>結果入力Excelを指定すると、1行目＝1号車として選手名を自動対応します。</p>\'))\n_hidden_display(lap_excel_upload,widgets.HBox([venue_box,race_no_box]),lap_paste,save_lap_button,lap_out)\n\n# ============================================================\n# Ver13.2 公式結果全文貼り付け・一括登録\n# ============================================================\n\ndef _split_result_line(line):\n    line = line.replace(\'\\u3000\', \' \').strip()\n    if \'\\t\' in line:\n        return [x.strip() for x in line.split(\'\\t\')]\n    return [x.strip() for x in re.split(r\'\\s{2,}\', line) if x.strip()]\n\n\ndef _normalize_status(accident=\'\', abnormal=\'\'):\n    raw = (_s(accident) + \' \' + _s(abnormal)).strip()\n    if not raw:\n        return \'通常\'\n    aliases = [\n        (\'反妨\', \'反妨\'), (\'妨害\', \'反妨\'), (\'落車\', \'落車\'), (\'他落\', \'他落\'),\n        (\'故障\', \'故障\'), (\'反則\', \'反則\'), (\'不成立\', \'不成立\'), (\'欠車\', \'欠車\'),\n        (\'停止\', \'停止\'), (\'再試走\', \'再試走\')\n    ]\n    for key, val in aliases:\n        if key in raw:\n            return val\n    return raw\n\n\ndef parse_full_official_result(text):\n    if not _s(text):\n        raise ValueError(\'貼り付け内容が空です\')\n    clean = text.replace(\'\\r\\n\', \'\\n\').replace(\'\\r\', \'\\n\')\n    lap_rows, meta = parse_lap_paste(clean)\n\n    m = re.search(r\'1着賞金\\s*[\\u3000 ]*([\\d,]+)円\', clean)\n    if m:\n        meta[\'first_prize_yen\'] = int(m.group(1).replace(\',\', \'\'))\n\n    lines = [x.rstrip() for x in clean.split(\'\\n\')]\n    header_idx = None\n    for i, line in enumerate(lines):\n        compact = line.replace(\'\\u3000\', \' \').replace(\' \', \'\')\n        if all(x in compact for x in [\'着\', \'車\', \'選手名\', \'試走タイム\', \'競走タイム\']):\n            header_idx = i\n            break\n    if header_idx is None:\n        raise ValueError(\'着順結果表の見出しが見つかりません\')\n\n    results = []\n    for line in lines[header_idx + 1:]:\n        if \'グランドノート\' in line or \'周回・順位\' in line:\n            break\n        if not line.strip():\n            continue\n        if \'\\t\' in line:\n            parts = [x.strip() for x in line.replace(\'\\u3000\', \' \').split(\'\\t\')]\n            parts += [\'\'] * max(0, 12 - len(parts))\n            finish, accident, car, name, age_term, lg, handicap, trial, race_t, st, abnormal, popularity = parts[:12]\n        else:\n            parts = _split_result_line(line)\n            if len(parts) < 9 or not re.fullmatch(r\'\\d+\', parts[0]) or not re.fullmatch(r\'\\d+\', parts[1]):\n                continue\n            finish, car = parts[0], parts[1]\n            name = parts[2]\n            age_term = parts[3] if len(parts) > 3 else \'\'\n            lg = parts[4] if len(parts) > 4 else \'\'\n            handicap = parts[5] if len(parts) > 5 else \'\'\n            trial = parts[6] if len(parts) > 6 else \'\'\n            race_t = parts[7] if len(parts) > 7 else \'\'\n            st = parts[8] if len(parts) > 8 else \'\'\n            popularity = parts[-1] if len(parts) > 9 and re.fullmatch(r\'\\d+\', parts[-1]) else \'\'\n            accident = \'\'\n            abnormal = \'\'\n        if not re.fullmatch(r\'\\d+\', _s(finish)) or not re.fullmatch(r\'\\d+\', _s(car)):\n            continue\n        results.append({\n            \'着順\': int(finish), \'事故\': _s(accident), \'車番\': int(car), \'選手名\': _s(name),\n            \'年齢期\': _s(age_term), \'LG\': _s(lg), \'ハンデ\': _s(handicap),\n            \'試走T\': _num(trial), \'競走T\': _num(race_t), \'ST\': _num(st),\n            \'異常\': _s(abnormal), \'人気\': _num(popularity),\n            \'結果区分\': _normalize_status(accident, abnormal)\n        })\n    if not results:\n        raise ValueError(\'選手結果を読み取れませんでした\')\n    cars = sorted(r[\'車番\'] for r in results)\n    if cars != list(range(1, len(results) + 1)):\n        raise ValueError(f\'車番の読取結果が不正です: {cars}\')\n    return meta, results, lap_rows\n\n\ndef init_full_result_tables():\n    init_lap_tables()\n    with sqlite3.connect(DB_PATH) as con:\n        con.executescript("""\n        CREATE TABLE IF NOT EXISTS race_entries (\n            entry_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            race_id INTEGER NOT NULL,\n            car_no INTEGER NOT NULL,\n            player_id INTEGER,\n            finish INTEGER,\n            accident TEXT,\n            age_term TEXT,\n            lg TEXT,\n            handicap TEXT,\n            trial_time REAL,\n            race_time REAL,\n            start_time REAL,\n            abnormal TEXT,\n            popularity INTEGER,\n            result_status TEXT,\n            UNIQUE(race_id, car_no),\n            FOREIGN KEY(race_id) REFERENCES races(race_id),\n            FOREIGN KEY(player_id) REFERENCES players(player_id)\n        );\n        CREATE INDEX IF NOT EXISTS idx_race_entries_player ON race_entries(player_id);\n        """)\n\n\ndef save_full_official_result(text, venue=\'\', race_no=\'\', source=\'公式結果全文貼付\'):\n    init_full_result_tables()\n    meta, results, lap_rows = parse_full_official_result(text)\n    if _s(venue):\n        meta[\'venue\'] = _s(venue)\n    if _s(race_no):\n        meta[\'race_no\'] = _s(race_no)\n    mapping = {r[\'車番\']: r[\'選手名\'] for r in results}\n    race_id, features = save_lap_data(lap_rows, meta, mapping, source=source)\n\n    added_history = 0\n    skipped_history = 0\n    with sqlite3.connect(DB_PATH) as con:\n        for r in results:\n            pid = _ensure_player(con, r[\'選手名\'])\n            con.execute("""INSERT INTO race_entries(\n                race_id,car_no,player_id,finish,accident,age_term,lg,handicap,\n                trial_time,race_time,start_time,abnormal,popularity,result_status)\n                VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?)\n                ON CONFLICT(race_id,car_no) DO UPDATE SET\n                player_id=excluded.player_id,finish=excluded.finish,accident=excluded.accident,\n                age_term=excluded.age_term,lg=excluded.lg,handicap=excluded.handicap,\n                trial_time=excluded.trial_time,race_time=excluded.race_time,\n                start_time=excluded.start_time,abnormal=excluded.abnormal,\n                popularity=excluded.popularity,result_status=excluded.result_status""",\n                (race_id, r[\'車番\'], pid, r[\'着順\'], r[\'事故\'], r[\'年齢期\'], r[\'LG\'], r[\'ハンデ\'],\n                 r[\'試走T\'], r[\'競走T\'], r[\'ST\'], r[\'異常\'], r[\'人気\'], r[\'結果区分\']))\n\n    starters = len(results)\n    surface = meta.get(\'surface_condition\', \'\')\n    for r in results:\n        row = {\n            \'開催日\': meta.get(\'race_date\', \'\'), \'開催場\': meta.get(\'venue\', \'\'),\n            \'レース\': meta.get(\'race_no\') or meta.get(\'race_name\', \'\'),\n            \'着順\': r[\'着順\'], \'出走\': starters, \'走路\': surface, \'ハンデ\': r[\'ハンデ\'],\n            \'試走T\': r[\'試走T\'], \'競走T\': r[\'競走T\'], \'ST\': r[\'ST\'], \'結果区分\': r[\'結果区分\']\n        }\n        a, s = add_history_rows(r[\'選手名\'], [row], source)\n        added_history += a\n        skipped_history += s\n    return race_id, meta, results, features, added_history, skipped_history\n\n\ndef full_result_summary(results, features):\n    rows = []\n    for r in sorted(results, key=lambda x: x[\'着順\']):\n        f = features.get(r[\'車番\'], {})\n        rows.append({\n            \'着\': r[\'着順\'], \'車\': r[\'車番\'], \'選手名\': r[\'選手名\'], \'結果区分\': r[\'結果区分\'],\n            \'試走T\': r[\'試走T\'], \'競走T\': r[\'競走T\'], \'ST\': r[\'ST\'], \'人気\': r[\'人気\'],\n            \'1周目\': f.get(\'first_lap_pos\'), \'最終\': f.get(\'final_pos\'),\n            \'純上昇\': f.get(\'net_gain\'), \'前団率\': f.get(\'front_pack_rate\'),\n            \'終盤上昇\': f.get(\'late_gain\'), \'追抜き量\': f.get(\'overtakes\')\n        })\n    return pd.DataFrame(rows)\n\n\ninit_full_result_tables()\nfull_paste = widgets.Textarea(\n    value=\'\',\n    placeholder=\'公式結果ページの「レース情報～着順表～グランドノート～周回順位」をまとめて貼り付け\',\n    description=\'結果全文:\',\n    layout=widgets.Layout(width=\'100%\', height=\'520px\'),\n    style={\'description_width\': \'80px\'}\n)\nfull_venue = widgets.Text(description=\'開催場:\', placeholder=\'本文に無い場合のみ 例：川口\')\nfull_race_no = widgets.Text(description=\'レース:\', placeholder=\'本文に無い場合のみ 例：12R\')\nfull_save_btn = widgets.Button(description=\'結果・周回を一括DB登録\', button_style=\'success\', icon=\'database\')\nfull_out = widgets.Output()\n\n\ndef on_save_full_result(_):\n    with full_out:\n        _hidden_clear_output()\n        try:\n            race_id, meta, results, features, a, s = save_full_official_result(\n                full_paste.value, full_venue.value, full_race_no.value\n            )\n            print(f\'保存完了：race_id={race_id}\')\n            print(f\'選手履歴 新規追加={a} / 重複スキップ={s}\')\n            if not meta.get(\'venue\'):\n                print(\'注意：開催場が本文から取得できませんでした。開催場欄へ入力してください。\')\n            if not meta.get(\'race_no\'):\n                print(\'注意：レース番号が本文から取得できませんでした。レース欄へ入力してください。\')\n            print(\'抽出したレース情報:\', meta)\n            _hidden_display(full_result_summary(results, features))\n        except Exception as e:\n            print(\'エラー:\', e)\n\n\nfull_save_btn.on_click(on_save_full_result)\n_hidden_display(widgets.HTML(\n    \'<h3>⑥ 公式結果ページ全文をそのまま貼り付け</h3>\'\n    \'<p>着順表と周回順位を一度に保存します。開催場・レース番号が本文に無い場合だけ補助欄へ入力してください。</p>\'\n))\n_hidden_display(widgets.HBox([full_venue, full_race_no]), full_paste, full_save_btn, full_out)\n\n\n# ============================================================\n# Ver14.0 全結果再学習・設定変化表示\n# ============================================================\nimport json, sqlite3, hashlib, math\nfrom datetime import datetime\n\nLEARN_FEATURES = {\n    "基礎能力": ["基礎履歴総合点", "基礎スピード点", "実戦能力点", "勝負強さ点"],\n    "スタート": ["スタート一気指数", "スタート伸び指数", "先行転換力", "同ハンデST優位度"],\n    "前団維持": ["隊列残存指数", "好位置維持指数", "前団主導指数", "安定上位指数"],\n    "追い上げ": ["混戦突破適性", "集団突破力", "後方追上げ指数"],\n    "終盤": ["終盤指数", "ゴール前伸び指数"],\n    "当日気配": ["当日状態指数", "自己比試走指数", "試走突出度", "爆発指数"],\n    "安定性": ["近況信頼度", "試走信頼度", "レース巧者指数"],\n}\nINITIAL_WEIGHTS = {\n    "基礎能力": 1.00, "スタート": 0.72, "前団維持": 0.68,\n    "追い上げ": 0.62, "終盤": 0.58, "当日気配": 0.74, "安定性": 0.55,\n}\n\ndef init_learning_tables():\n    mount_and_init_db()\n    with sqlite3.connect(DB_PATH) as con:\n        con.executescript("""\n        CREATE TABLE IF NOT EXISTS prediction_races (\n            prediction_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            race_key TEXT NOT NULL UNIQUE,\n            race_date TEXT, venue TEXT, race_no TEXT,\n            track_temp REAL, created_at TEXT\n        );\n        CREATE TABLE IF NOT EXISTS prediction_features (\n            prediction_id INTEGER NOT NULL,\n            car_no INTEGER NOT NULL,\n            player_name TEXT,\n            features_json TEXT NOT NULL,\n            PRIMARY KEY(prediction_id, car_no)\n        );\n        CREATE TABLE IF NOT EXISTS learning_settings (\n            setting_name TEXT PRIMARY KEY,\n            initial_value REAL NOT NULL,\n            current_value REAL NOT NULL,\n            updated_at TEXT,\n            sample_races INTEGER DEFAULT 0,\n            reason TEXT\n        );\n        CREATE TABLE IF NOT EXISTS learning_history (\n            history_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            learned_at TEXT, sample_races INTEGER,\n            old_score REAL, new_score REAL,\n            settings_json TEXT, changes_json TEXT\n        );\n        """)\n        for k,v in INITIAL_WEIGHTS.items():\n            con.execute("""INSERT OR IGNORE INTO learning_settings\n                (setting_name,initial_value,current_value,updated_at,sample_races,reason)\n                VALUES(?,?,?,?,?,?)""",(k,v,v,datetime.now().isoformat(timespec=\'seconds\'),0,\'初期設定\'))\n        con.commit()\n\ndef _race_key(date_text, venue, race_no):\n    d=re.sub(r\'[^0-9]\',\'\',str(date_text))[:8]\n    r=re.sub(r\'[^0-9]\',\'\',str(race_no))\n    return f"{d}_{str(venue).strip()}_{r}R"\n\ndef _layer_value(row, cols):\n    vals=[]\n    for c in cols:\n        if c in row.index:\n            try:\n                x=float(row[c])\n                if np.isfinite(x): vals.append(x)\n            except: pass\n    return float(np.mean(vals)) if vals else 0.0\n\ndef save_latest_prediction(date_text, venue, race_no):\n    init_learning_tables()\n    if \'LATEST_PREDICTION_DF\' not in globals():\n        raise RuntimeError(\'先に予測セルで計算してください。\')\n    df=LATEST_PREDICTION_DF.copy()\n    key=_race_key(date_text,venue,race_no)\n    now=datetime.now().isoformat(timespec=\'seconds\')\n    with sqlite3.connect(DB_PATH) as con:\n        con.execute("""INSERT INTO prediction_races(race_key,race_date,venue,race_no,track_temp,created_at)\n          VALUES(?,?,?,?,?,?) ON CONFLICT(race_key) DO UPDATE SET\n          track_temp=excluded.track_temp,created_at=excluded.created_at""",\n          (key,str(date_text),str(venue),str(race_no),float(globals().get(\'LATEST_TRACK_TEMP\',np.nan)),now))\n        pid=con.execute(\'SELECT prediction_id FROM prediction_races WHERE race_key=?\',(key,)).fetchone()[0]\n        con.execute(\'DELETE FROM prediction_features WHERE prediction_id=?\',(pid,))\n        for _,row in df.iterrows():\n            car=int(float(row.get(\'車\',row.get(\'車番\',0))))\n            feats={layer:_layer_value(row,cols) for layer,cols in LEARN_FEATURES.items()}\n            # race内標準化は学習時に実施\n            con.execute(\'INSERT INTO prediction_features VALUES(?,?,?,?)\',\n                        (pid,car,str(row.get(\'選手名\',\'\')),json.dumps(feats,ensure_ascii=False)))\n        con.commit()\n    return key,len(df)\n\ndef _load_training():\n    init_learning_tables()\n    with sqlite3.connect(DB_PATH) as con:\n        prs=pd.read_sql_query(\'SELECT * FROM prediction_races\',con)\n        pfs=pd.read_sql_query(\'SELECT * FROM prediction_features\',con)\n        # Ver13.2 races/results schemaに合わせ、存在列を確認\n        tables={x[0] for x in con.execute("SELECT name FROM sqlite_master WHERE type=\'table\'")}\n        if \'race_results\' in tables:\n            res=pd.read_sql_query(\'SELECT * FROM race_results\',con)\n        elif \'results\' in tables:\n            res=pd.read_sql_query(\'SELECT * FROM results\',con)\n        else:\n            return []\n        races=pd.read_sql_query(\'SELECT * FROM races\',con) if \'races\' in tables else pd.DataFrame()\n    datasets=[]\n    for _,pr in prs.iterrows():\n        sub=pfs[pfs.prediction_id==pr.prediction_id].copy()\n        if sub.empty: continue\n        # race_key直接または日付/場/Rで結果レースを照合\n        rid=None\n        if not races.empty and \'race_key\' in races.columns:\n            m=races[races.race_key.astype(str)==str(pr.race_key)]\n            if not m.empty: rid=m.iloc[0].get(\'race_id\')\n        if rid is None and not races.empty:\n            m=races[(races.get(\'race_date\',\'\').astype(str)==str(pr.race_date)) &\n                    (races.get(\'venue\',\'\').astype(str)==str(pr.venue))]\n            if \'race_no\' in races.columns:\n                m=m[m.race_no.astype(str).str.replace(\'R\',\'\',regex=False)==str(pr.race_no).replace(\'R\',\'\')]\n            if not m.empty: rid=m.iloc[0].get(\'race_id\')\n        if rid is None: continue\n        rr=res[res.race_id==rid].copy() if \'race_id\' in res.columns else pd.DataFrame()\n        if rr.empty: continue\n        carcol=\'car_no\' if \'car_no\' in rr.columns else (\'車\' if \'車\' in rr.columns else None)\n        rankcol=\'finish_rank\' if \'finish_rank\' in rr.columns else (\'着順\' if \'着順\' in rr.columns else None)\n        statuscol=\'result_status\' if \'result_status\' in rr.columns else None\n        if not carcol or not rankcol: continue\n        merged=sub.merge(rr[[carcol,rankcol]+([statuscol] if statuscol else [])],left_on=\'car_no\',right_on=carcol)\n        if statuscol:\n            bad=\'反妨|落車|故障|反則|他落|不成立\'\n            merged=merged[~merged[statuscol].fillna(\'\').astype(str).str.contains(bad,regex=True)]\n        merged[rankcol]=pd.to_numeric(merged[rankcol],errors=\'coerce\')\n        merged=merged.dropna(subset=[rankcol])\n        if len(merged)>=5:\n            datasets.append((str(pr.race_key),merged,rankcol))\n    return datasets\n\ndef _matrix_for_race(df):\n    raw=np.array([[json.loads(x).get(k,0.0) for k in INITIAL_WEIGHTS] for x in df.features_json],float)\n    mu=np.nanmean(raw,axis=0); sd=np.nanstd(raw,axis=0)\n    sd=np.where(sd<1e-8,1.0,sd)\n    return np.nan_to_num((raw-mu)/sd)\n\ndef _loss(weights,datasets,reg_strength):\n    w=np.array([weights[k] for k in INITIAL_WEIGHTS],float)\n    losses=[]\n    for _,df,rankcol in datasets:\n        X=_matrix_for_race(df); score=X@w\n        ranks=df[rankcol].to_numpy(float)\n        # 全ペア順位logloss\n        for i in range(len(df)):\n            for j in range(i+1,len(df)):\n                if ranks[i]==ranks[j]: continue\n                y=1.0 if ranks[i]<ranks[j] else 0.0\n                z=np.clip(score[i]-score[j],-20,20)\n                p=1/(1+np.exp(-z))\n                losses.append(-(y*np.log(p+1e-12)+(1-y)*np.log(1-p+1e-12)))\n    base=np.mean(losses) if losses else 99.0\n    initial=np.array(list(INITIAL_WEIGHTS.values()))\n    return float(base + reg_strength*np.mean((w-initial)**2))\n\ndef learn_best_weights(search_rounds=3500,seed=42):\n    datasets=_load_training()\n    n=len(datasets)\n    if n<3:\n        raise RuntimeError(f\'学習可能な予測＋結果が{n}レースです。最低3レース蓄積してください。\')\n    init_learning_tables()\n    with sqlite3.connect(DB_PATH) as con:\n        cur={r[0]:float(r[1]) for r in con.execute(\'SELECT setting_name,current_value FROM learning_settings\')}\n    # 少数時は初期値へ強く寄せる\n    reg=max(0.015,0.30/(n**0.65))\n    rng=np.random.default_rng(seed)\n    best=dict(cur); best_loss=_loss(best,datasets,reg)\n    old_loss=best_loss\n    scales=np.array([0.42,0.35,0.38,0.38,0.34,0.40,0.30])\n    centers=np.array([cur[k] for k in INITIAL_WEIGHTS])\n    for t in range(int(search_rounds)):\n        shrink=0.25+0.75*(1-t/max(search_rounds,1))\n        cand_arr=centers+rng.normal(0,scales*shrink)\n        cand_arr=np.clip(cand_arr,0.05,1.80)\n        cand={k:float(v) for k,v in zip(INITIAL_WEIGHTS,cand_arr)}\n        loss=_loss(cand,datasets,reg)\n        if loss<best_loss:\n            best,best_loss=cand,loss\n            centers=cand_arr\n    now=datetime.now().isoformat(timespec=\'seconds\')\n    changes={k:best[k]-cur[k] for k in best}\n    with sqlite3.connect(DB_PATH) as con:\n        for k,v in best.items():\n            con.execute(\'UPDATE learning_settings SET current_value=?,updated_at=?,sample_races=?,reason=? WHERE setting_name=?\',\n                        (v,now,n,\'全登録レースの順位ペア誤差を最小化\',k))\n        con.execute(\'INSERT INTO learning_history(learned_at,sample_races,old_score,new_score,settings_json,changes_json) VALUES(?,?,?,?,?,?)\',\n                    (now,n,old_loss,best_loss,json.dumps(best,ensure_ascii=False),json.dumps(changes,ensure_ascii=False)))\n        con.commit()\n    return settings_table(),old_loss,best_loss,n\n\ndef settings_table():\n    init_learning_tables()\n    with sqlite3.connect(DB_PATH) as con:\n        df=pd.read_sql_query(\'SELECT setting_name AS 設定,initial_value AS 初期値,current_value AS 現在値,updated_at AS 最終更新,sample_races AS 採用レース数,reason AS 変更理由 FROM learning_settings ORDER BY rowid\',con)\n    df[\'変化量\']=df[\'現在値\']-df[\'初期値\']\n    df[\'変化率%\']=np.where(df[\'初期値\']!=0,df[\'変化量\']/df[\'初期値\']*100,0)\n    df[\'方向\']=np.where(df[\'変化量\']>0.005,\'↑ 強化\',np.where(df[\'変化量\']<-0.005,\'↓ 弱化\',\'→ ほぼ同じ\'))\n    return df[[\'設定\',\'初期値\',\'現在値\',\'変化量\',\'変化率%\',\'方向\',\'採用レース数\',\'最終更新\',\'変更理由\']]\n\ndef learning_history_table(limit=20):\n    init_learning_tables()\n    with sqlite3.connect(DB_PATH) as con:\n        return pd.read_sql_query(\'SELECT learned_at AS 学習日時,sample_races AS レース数,old_score AS 更新前誤差,new_score AS 更新後誤差,changes_json AS 変更内容 FROM learning_history ORDER BY history_id DESC LIMIT ?\',con,params=(int(limit),))\n\ndef reset_weights_to_initial():\n    init_learning_tables(); now=datetime.now().isoformat(timespec=\'seconds\')\n    with sqlite3.connect(DB_PATH) as con:\n        con.execute(\'UPDATE learning_settings SET current_value=initial_value,updated_at=?,sample_races=0,reason=?\',(now,\'初期設定へ手動復元\'))\n        con.commit()\n    return settings_table()\n\n# UI\nv14_date=widgets.Text(description=\'開催日\',placeholder=\'2026/07/20\')\nv14_venue=widgets.Text(description=\'開催場\',placeholder=\'川口\')\nv14_race=widgets.Text(description=\'レース\',placeholder=\'12\')\nv14_save=widgets.Button(description=\'最新予測をDB保存\',button_style=\'info\')\nv14_learn=widgets.Button(description=\'全結果から重み再学習\',button_style=\'success\')\nv14_show=widgets.Button(description=\'初期値との差を表示\')\nv14_history=widgets.Button(description=\'学習履歴を表示\')\nv14_reset=widgets.Button(description=\'初期設定へ戻す\',button_style=\'warning\')\nv14_out=widgets.Output()\n\ndef _v14_save(_):\n    with v14_out:\n        _hidden_clear_output()\n        try:\n            key,n=save_latest_prediction(v14_date.value,v14_venue.value,v14_race.value)\n            print(f\'保存完了: {key} / {n}選手\')\n            print(\'レース結果を全文登録した後、「全結果から重み再学習」を押してください。\')\n        except Exception as e: print(\'エラー:\',e)\ndef _v14_learn(_):\n    with v14_out:\n        _hidden_clear_output()\n        try:\n            df,a,b,n=learn_best_weights()\n            print(f\'{n}レースを使って再学習しました。誤差 {a:.4f} → {b:.4f}\')\n            _hidden_display(df.style.format({\'初期値\':\'{:.3f}\',\'現在値\':\'{:.3f}\',\'変化量\':\'{:+.3f}\',\'変化率%\':\'{:+.1f}%\'}))\n        except Exception as e: print(\'エラー:\',e)\ndef _v14_show(_):\n    with v14_out:\n        _hidden_clear_output(); _hidden_display(settings_table().style.format({\'初期値\':\'{:.3f}\',\'現在値\':\'{:.3f}\',\'変化量\':\'{:+.3f}\',\'変化率%\':\'{:+.1f}%\'}))\ndef _v14_hist(_):\n    with v14_out:\n        _hidden_clear_output(); _hidden_display(learning_history_table())\ndef _v14_reset(_):\n    with v14_out:\n        _hidden_clear_output(); print(\'初期設定へ戻しました。\'); _hidden_display(reset_weights_to_initial())\nv14_save.on_click(_v14_save); v14_learn.on_click(_v14_learn); v14_show.on_click(_v14_show); v14_history.on_click(_v14_hist); v14_reset.on_click(_v14_reset)\n_hidden_display(widgets.VBox([\n    widgets.HTML(\'<b>Ver14.0 全履歴再学習</b><br>予測直後に開催情報を入力して保存し、結果全文登録後に再学習します。\'),\n    widgets.HBox([v14_date,v14_venue,v14_race]),\n    widgets.HBox([v14_save,v14_learn,v14_show,v14_history,v14_reset]),v14_out\n]))\n\n# ============================================================\n# Ver14.1 条件依存特徴・選手別適用\n# ============================================================\nimport re, sqlite3, json, math\nfrom datetime import datetime\n\nCONDITION_DEFAULTS = {\n    \'min_condition_races\': 5,\n    \'min_other_races\': 8,\n    \'min_total_races\': 14,\n    \'min_effect\': 0.11,\n    \'min_confidence\': 0.72,\n    \'max_adjustment\': 0.38,\n    \'shrink_k\': 8.0,\n}\n\n\ndef init_condition_tables():\n    mount_and_init_db()\n    with sqlite3.connect(DB_PATH) as con:\n        con.executescript("""\n        CREATE TABLE IF NOT EXISTS player_condition_profiles (\n            player_id INTEGER NOT NULL,\n            condition_type TEXT NOT NULL,\n            condition_value TEXT NOT NULL,\n            sample_condition INTEGER NOT NULL,\n            sample_other INTEGER NOT NULL,\n            score_condition REAL,\n            score_other REAL,\n            raw_effect REAL,\n            shrunk_effect REAL,\n            confidence REAL,\n            adjustment REAL,\n            is_active INTEGER NOT NULL DEFAULT 0,\n            calculated_at TEXT,\n            PRIMARY KEY(player_id, condition_type, condition_value)\n        );\n        CREATE TABLE IF NOT EXISTS condition_learning_history (\n            history_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            calculated_at TEXT,\n            player_count INTEGER,\n            active_profile_count INTEGER,\n            settings_json TEXT\n        );\n        """)\n        # 既存prediction_racesを条件保存対応へ拡張\n        cols={r[1] for r in con.execute(\'PRAGMA table_info(prediction_races)\')}\n        additions={\n            \'start_time_text\':\'TEXT\',\'air_temp\':\'REAL\',\'humidity\':\'REAL\',\n            \'weather\':\'TEXT\',\'surface_condition\':\'TEXT\',\'distance_m\':\'INTEGER\',\n            \'total_laps\':\'INTEGER\',\'handicap_pattern\':\'TEXT\'\n        }\n        for name,typ in additions.items():\n            if name not in cols:\n                con.execute(f\'ALTER TABLE prediction_races ADD COLUMN {name} {typ}\')\n        con.commit()\n\n\ndef _num(x):\n    try:\n        v=float(x)\n        return v if np.isfinite(v) else np.nan\n    except Exception:\n        return np.nan\n\n\ndef _time_band(text):\n    m=re.search(r\'(\\d{1,2}):(\\d{2})\',str(text or \'\'))\n    if not m: return \'時間不明\'\n    h=int(m.group(1))\n    if h < 15: return \'昼\'\n    if h < 18: return \'夕方\'\n    return \'夜\'\n\n\ndef _temp_band(x, kind=\'走路\'):\n    x=_num(x)\n    if not np.isfinite(x): return f\'{kind}不明\'\n    if kind==\'走路\':\n        if x < 20: return \'低温(<20℃)\'\n        if x < 35: return \'中温(20-34℃)\'\n        if x < 45: return \'高温(35-44℃)\'\n        return \'酷熱(45℃以上)\'\n    if x < 15: return \'低温(<15℃)\'\n    if x < 28: return \'中温(15-27℃)\'\n    return \'高温(28℃以上)\'\n\n\ndef _humidity_band(x):\n    x=_num(x)\n    if not np.isfinite(x): return \'湿度不明\'\n    if x < 45: return \'低湿度(<45%)\'\n    if x < 70: return \'中湿度(45-69%)\'\n    return \'高湿度(70%以上)\'\n\n\ndef _handicap_num(x):\n    m=re.search(r\'-?\\d+\',str(x or \'\'))\n    return int(m.group()) if m else None\n\n\ndef _handicap_band(x):\n    n=_handicap_num(x)\n    if n is None: return \'ハンデ不明\'\n    if n <= 0: return \'0m\'\n    if n <= 10: return \'10m以内\'\n    if n <= 20: return \'20m以内\'\n    return \'30m以上\'\n\n\ndef _surface_group(x):\n    s=str(x or \'\').strip()\n    if not s: return \'走路不明\'\n    if \'良\' in s: return \'良走路\'\n    if \'湿\' in s or \'濡\' in s: return \'湿走路\'\n    if \'斑\' in s: return \'斑走路\'\n    return s\n\n\ndef _distance_band(x):\n    x=_num(x)\n    if not np.isfinite(x): return \'距離不明\'\n    if x <= 3100: return \'通常距離(3100m以下)\'\n    if x <= 4100: return \'長距離(3101-4100m)\'\n    return \'超長距離(4100m超)\'\n\n\ndef _performance_score(rank, field_size):\n    rank=_num(rank); field_size=_num(field_size)\n    if not np.isfinite(rank) or not np.isfinite(field_size) or field_size <= 1: return np.nan\n    return float(np.clip((field_size-rank)/(field_size-1),0,1))\n\n\ndef _load_condition_rows():\n    init_condition_tables()\n    with sqlite3.connect(DB_PATH) as con:\n        tables={r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type=\'table\'")}\n        if not {\'races\',\'race_entries\',\'players\'}.issubset(tables):\n            return pd.DataFrame()\n        q="""\n        SELECT e.race_id,e.player_id,p.player_name,e.car_no,e.finish,e.result_status,\n               e.handicap,e.trial_time,e.race_time,e.start_time,\n               r.race_date,r.venue,r.race_no,r.race_name,r.distance_m,r.total_laps,\n               r.weather,r.surface_condition,r.track_temp,r.air_temp,r.humidity,r.start_time_text\n        FROM race_entries e\n        JOIN races r ON r.race_id=e.race_id\n        LEFT JOIN players p ON p.player_id=e.player_id\n        """\n        df=pd.read_sql_query(q,con)\n    if df.empty: return df\n    bad=\'反妨|落車|故障|反則|他落|不成立\'\n    df=df[~df[\'result_status\'].fillna(\'\').astype(str).str.contains(bad,regex=True)].copy()\n    df[\'finish\']=pd.to_numeric(df[\'finish\'],errors=\'coerce\')\n    df=df.dropna(subset=[\'player_id\',\'finish\'])\n    sizes=df.groupby(\'race_id\')[\'car_no\'].transform(\'count\')\n    df[\'field_size\']=sizes\n    df[\'performance\']=[_performance_score(a,b) for a,b in zip(df.finish,df.field_size)]\n    # 同レース内ハンデ構成\n    def dense_line(g):\n        hs=g[\'handicap\'].map(_handicap_num)\n        counts=hs.value_counts(dropna=True)\n        return \'同ハンデ多数\' if (not counts.empty and counts.max()>=5) else \'通常ハンデ構成\'\n    patterns={rid:dense_line(g) for rid,g in df.groupby(\'race_id\')}\n    df[\'handicap_pattern\']=df[\'race_id\'].map(patterns).fillna(\'通常ハンデ構成\')\n    df[\'時間帯\']=df[\'start_time_text\'].map(_time_band)\n    df[\'走路温度帯\']=df[\'track_temp\'].map(lambda x:_temp_band(x,\'走路\'))\n    df[\'気温帯\']=df[\'air_temp\'].map(lambda x:_temp_band(x,\'気温\'))\n    df[\'湿度帯\']=df[\'humidity\'].map(_humidity_band)\n    df[\'走路状態\']=df[\'surface_condition\'].map(_surface_group)\n    df[\'天候条件\']=df[\'weather\'].fillna(\'天候不明\').astype(str).str.strip().replace(\'\',\'天候不明\')\n    df[\'開催場条件\']=df[\'venue\'].fillna(\'開催場不明\').astype(str)\n    df[\'ハンデ帯\']=df[\'handicap\'].map(_handicap_band)\n    df[\'ハンデ構成\']=df[\'handicap_pattern\']\n    df[\'距離帯\']=df[\'distance_m\'].map(_distance_band)\n    return df\n\n\nCONDITION_COLUMNS=[\'時間帯\',\'走路温度帯\',\'気温帯\',\'湿度帯\',\'走路状態\',\'天候条件\',\'開催場条件\',\'ハンデ帯\',\'ハンデ構成\',\'距離帯\']\n\n\ndef _confidence_from_effect(a,b,effect):\n    # Welch型の標準誤差から、0.5～0.99の扱いやすい信頼度へ変換\n    na,nb=len(a),len(b)\n    if na<2 or nb<2: return 0.0\n    se=math.sqrt(float(np.var(a,ddof=1))/na + float(np.var(b,ddof=1))/nb + 1e-12)\n    z=abs(effect)/se if se>0 else 0.0\n    return float(np.clip(0.5 + 0.49*(1-math.exp(-0.55*z)),0,0.99))\n\n\ndef rebuild_player_condition_profiles(settings=None):\n    cfg=dict(CONDITION_DEFAULTS)\n    if settings: cfg.update(settings)\n    df=_load_condition_rows()\n    if df.empty:\n        raise RuntimeError(\'公式結果データがまだありません。\')\n    now=datetime.now().isoformat(timespec=\'seconds\')\n    records=[]\n    for pid,g in df.groupby(\'player_id\'):\n        if len(g)<cfg[\'min_total_races\']: continue\n        for ctype in CONDITION_COLUMNS:\n            for cval,cg in g.groupby(ctype):\n                other=g[g[ctype]!=cval]\n                if len(cg)<cfg[\'min_condition_races\'] or len(other)<cfg[\'min_other_races\']: continue\n                a=cg.performance.dropna().to_numpy(float); b=other.performance.dropna().to_numpy(float)\n                if len(a)<cfg[\'min_condition_races\'] or len(b)<cfg[\'min_other_races\']: continue\n                ma=float(np.mean(a)); mb=float(np.mean(b)); raw=ma-mb\n                shrink=len(a)/(len(a)+cfg[\'shrink_k\'])\n                shrunk=raw*shrink\n                conf=_confidence_from_effect(a,b,raw)\n                active=(abs(shrunk)>=cfg[\'min_effect\'] and conf>=cfg[\'min_confidence\'])\n                adj=float(np.clip(shrunk*1.8,-cfg[\'max_adjustment\'],cfg[\'max_adjustment\'])) if active else 0.0\n                records.append((int(pid),ctype,str(cval),len(a),len(b),ma,mb,raw,shrunk,conf,adj,int(active),now))\n    with sqlite3.connect(DB_PATH) as con:\n        con.execute(\'DELETE FROM player_condition_profiles\')\n        con.executemany("""INSERT INTO player_condition_profiles(\n            player_id,condition_type,condition_value,sample_condition,sample_other,\n            score_condition,score_other,raw_effect,shrunk_effect,confidence,\n            adjustment,is_active,calculated_at) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)""",records)\n        active=sum(r[11] for r in records)\n        con.execute(\'INSERT INTO condition_learning_history(calculated_at,player_count,active_profile_count,settings_json) VALUES(?,?,?,?)\',\n                    (now,int(df.player_id.nunique()),int(active),json.dumps(cfg,ensure_ascii=False)))\n        con.commit()\n    return condition_profile_table(active_only=True),len(records),sum(r[11] for r in records)\n\n\ndef condition_profile_table(player_name=\'\',active_only=True):\n    init_condition_tables()\n    where=[]; params=[]\n    if active_only: where.append(\'c.is_active=1\')\n    if str(player_name).strip():\n        where.append(\'p.player_name LIKE ?\'); params.append(\'%\'+str(player_name).strip()+\'%\')\n    sql="""SELECT p.player_name AS 選手名,c.condition_type AS 条件種類,c.condition_value AS 条件,\n           c.sample_condition AS 条件内件数,c.sample_other AS 比較件数,\n           c.score_condition AS 条件内成績,c.score_other AS その他成績,\n           c.shrunk_effect AS 補正前効果,c.confidence AS 信頼度,\n           c.adjustment AS 予測補正,\n           CASE WHEN c.adjustment>0 THEN \'得意\' ELSE \'不得意\' END AS 判定,\n           c.calculated_at AS 算出日時\n           FROM player_condition_profiles c LEFT JOIN players p ON p.player_id=c.player_id"""\n    if where: sql+=\' WHERE \'+\' AND \'.join(where)\n    sql+=\' ORDER BY ABS(c.adjustment) DESC,c.sample_condition DESC\'\n    with sqlite3.connect(DB_PATH) as con:\n        return pd.read_sql_query(sql,con,params=params)\n\n\ndef _current_condition_map(meta):\n    return {\n        \'時間帯\':_time_band(meta.get(\'start_time_text\',\'\')),\n        \'走路温度帯\':_temp_band(meta.get(\'track_temp\'),\'走路\'),\n        \'気温帯\':_temp_band(meta.get(\'air_temp\'),\'気温\'),\n        \'湿度帯\':_humidity_band(meta.get(\'humidity\')),\n        \'走路状態\':_surface_group(meta.get(\'surface_condition\')),\n        \'天候条件\':str(meta.get(\'weather\') or \'天候不明\').strip(),\n        \'開催場条件\':str(meta.get(\'venue\') or \'開催場不明\').strip(),\n        \'ハンデ構成\':str(meta.get(\'handicap_pattern\') or \'通常ハンデ構成\'),\n        \'距離帯\':_distance_band(meta.get(\'distance_m\')),\n    }\n\n\ndef apply_condition_profiles(pred_df,meta):\n    init_condition_tables()\n    out=pred_df.copy()\n    name_col=\'選手名\' if \'選手名\' in out.columns else None\n    if not name_col: raise ValueError(\'予測表に選手名列がありません。\')\n    cmap=_current_condition_map(meta)\n    with sqlite3.connect(DB_PATH) as con:\n        prof=pd.read_sql_query("""SELECT p.player_name,c.* FROM player_condition_profiles c\n          JOIN players p ON p.player_id=c.player_id WHERE c.is_active=1""",con)\n    adjustments=[]; reasons=[]\n    for _,row in out.iterrows():\n        name=str(row[name_col]).strip(); ps=prof[prof.player_name.astype(str).str.strip()==name]\n        total=0.0; rs=[]\n        # 個人ハンデ帯も現在行から判定\n        local=dict(cmap); local[\'ハンデ帯\']=_handicap_band(row.get(\'ハンデ\',row.get(\'ハンデm\',\'\')))\n        for ctype,cval in local.items():\n            m=ps[(ps.condition_type==ctype)&(ps.condition_value.astype(str)==str(cval))]\n            if m.empty: continue\n            r=m.iloc[0]; a=float(r.adjustment)\n            total+=a\n            rs.append(f"{ctype}:{cval} {a:+.3f}(n={int(r.sample_condition)},信頼{float(r.confidence):.0%})")\n        total=float(np.clip(total,-0.55,0.55))\n        adjustments.append(total); reasons.append(\' / \'.join(rs) if rs else \'適用なし\')\n    out[\'条件依存補正\']=adjustments\n    out[\'条件補正理由\']=reasons\n    # 総合点系があれば表示用に反映。元列は残す\n    target=next((c for c in [\'総合点\',\'AI総合点\',\'予測スコア\'] if c in out.columns),None)\n    if target:\n        out[\'条件補正後\'+target]=pd.to_numeric(out[target],errors=\'coerce\').fillna(0)+out[\'条件依存補正\']\n    return out\n\n\ndef apply_conditions_to_latest_prediction(meta):\n    global LATEST_PREDICTION_DF\n    if \'LATEST_PREDICTION_DF\' not in globals():\n        raise RuntimeError(\'先に予測を実行してください。\')\n    LATEST_PREDICTION_DF=apply_condition_profiles(LATEST_PREDICTION_DF,meta)\n    return LATEST_PREDICTION_DF\n\n# UI\nv141_time=widgets.Text(description=\'発走時刻\',placeholder=\'20:45\')\nv141_track=widgets.FloatText(description=\'走路温度\',value=0.0)\nv141_air=widgets.FloatText(description=\'気温\',value=0.0)\nv141_hum=widgets.FloatText(description=\'湿度\',value=0.0)\nv141_weather=widgets.Text(description=\'天候\',placeholder=\'晴\')\nv141_surface=widgets.Text(description=\'走路\',placeholder=\'良走路\')\nv141_venue=widgets.Text(description=\'開催場\',placeholder=\'川口\')\nv141_distance=widgets.IntText(description=\'距離m\',value=3100)\nv141_pattern=widgets.Dropdown(description=\'ハンデ構成\',options=[\'通常ハンデ構成\',\'同ハンデ多数\'])\nv141_rebuild=widgets.Button(description=\'全結果から条件特徴を再分析\',button_style=\'success\')\nv141_apply=widgets.Button(description=\'今回予測へ条件補正を適用\',button_style=\'info\')\nv141_show=widgets.Button(description=\'有効な条件特徴を表示\')\nv141_player=widgets.Text(description=\'選手検索\',placeholder=\'空欄なら全員\')\nv141_out=widgets.Output()\n\ndef _v141_meta():\n    return {\'start_time_text\':v141_time.value,\'track_temp\':v141_track.value or np.nan,\n            \'air_temp\':v141_air.value or np.nan,\'humidity\':v141_hum.value or np.nan,\n            \'weather\':v141_weather.value,\'surface_condition\':v141_surface.value,\n            \'venue\':v141_venue.value,\'distance_m\':v141_distance.value,\n            \'handicap_pattern\':v141_pattern.value}\n\ndef _v141_rebuild(_):\n    with v141_out:\n        _hidden_clear_output()\n        try:\n            df,n,a=rebuild_player_condition_profiles()\n            print(f\'条件候補 {n}件を検証し、信頼条件を満たす特徴 {a}件を有効化しました。\')\n            if df.empty: print(\'まだ有効な条件特徴はありません。結果件数を蓄積してください。\')\n            else: _hidden_display(df.style.format({\'条件内成績\':\'{:.3f}\',\'その他成績\':\'{:.3f}\',\'補正前効果\':\'{:+.3f}\',\'信頼度\':\'{:.1%}\',\'予測補正\':\'{:+.3f}\'}))\n        except Exception as e: print(\'エラー:\',e)\n\ndef _v141_apply(_):\n    with v141_out:\n        _hidden_clear_output()\n        try:\n            df=apply_conditions_to_latest_prediction(_v141_meta())\n            cols=[c for c in [\'車\',\'車番\',\'選手名\',\'条件依存補正\',\'条件補正理由\',\'条件補正後総合点\',\'条件補正後AI総合点\',\'条件補正後予測スコア\'] if c in df.columns]\n            _hidden_display(df[cols] if cols else df)\n        except Exception as e: print(\'エラー:\',e)\n\ndef _v141_show(_):\n    with v141_out:\n        _hidden_clear_output()\n        df=condition_profile_table(v141_player.value,True)\n        if df.empty: print(\'該当する有効特徴はありません。\')\n        else: _hidden_display(df.style.format({\'条件内成績\':\'{:.3f}\',\'その他成績\':\'{:.3f}\',\'補正前効果\':\'{:+.3f}\',\'信頼度\':\'{:.1%}\',\'予測補正\':\'{:+.3f}\'}))\n\nv141_rebuild.on_click(_v141_rebuild)\nv141_apply.on_click(_v141_apply)\nv141_show.on_click(_v141_show)\n_hidden_display(widgets.VBox([\n    widgets.HTML(\'<b>Ver14.1 選手別・条件依存特徴</b><br>特定条件で差が繰り返し確認できた選手だけ補正します。\'),\n    widgets.HBox([v141_time,v141_venue,v141_distance]),\n    widgets.HBox([v141_track,v141_air,v141_hum]),\n    widgets.HBox([v141_weather,v141_surface,v141_pattern]),\n    widgets.HBox([v141_rebuild,v141_apply]),\n    widgets.HBox([v141_player,v141_show]),v141_out\n]))\n\n# ============================================================\n# AutoRaceAI Ver15.0\n# スマホ全文コピペ予測・選手履歴追加・結果登録\n# ============================================================\n\nimport re\nimport json\nimport math\nimport sqlite3\nimport hashlib\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport ipywidgets as widgets\nfrom IPython.display import display, clear_output, HTML\n\n# Ver14.1までのDB_PATHを優先して利用します。\nif "DB_PATH" not in globals():\n    DB_DIR = Path("/content/drive/MyDrive/AutoRaceAI")\nDB_DIR.mkdir(parents=True, exist_ok=True)\nDB_PATH = str(DB_DIR / "autorace_players.sqlite3")\n\nV15_STATE = {\n    "race_meta": {},\n    "entries": pd.DataFrame(),\n    "player_history": pd.DataFrame(),\n    "raw_prediction_text": "",\n    "raw_player_text": "",\n    "raw_result_text": "",\n}\n\nV15_ENTRY_COLUMNS = [\n    "車番", "選手名", "ハンデ", "試走T", "ST", "年齢", "級別", "期別"\n]\n\nV15_HISTORY_COLUMNS = [\n    "開催日", "開催場", "レース", "着順", "出走", "走路",\n    "ハンデ", "試走T", "競走T", "ST"\n]\n\n\n# ------------------------------\n# 共通ユーティリティ\n# ------------------------------\ndef v15_clean_text(text):\n    text = str(text or "")\n    text = text.replace("\\u3000", " ").replace("\\xa0", " ")\n    text = text.replace("℃", "℃ ")\n    text = re.sub(r"[ \\t]+", " ", text)\n    text = re.sub(r"\\r\\n?", "\\n", text)\n    return text.strip()\n\n\ndef v15_float(value):\n    if value is None:\n        return np.nan\n    m = re.search(r"-?\\d+(?:\\.\\d+)?", str(value).replace(",", ""))\n    return float(m.group()) if m else np.nan\n\n\ndef v15_int(value):\n    x = v15_float(value)\n    return int(x) if pd.notna(x) else None\n\n\ndef v15_normalize_name(name):\n    name = str(name or "").strip()\n    name = re.sub(r"\\s+", " ", name)\n    name = re.sub(r"^[0-9]+\\s*", "", name)\n    return name\n\n\ndef v15_hash(*parts):\n    joined = "|".join("" if p is None else str(p).strip() for p in parts)\n    return hashlib.sha256(joined.encode("utf-8")).hexdigest()[:24]\n\n\ndef v15_table_exists(con, table):\n    row = con.execute(\n        "SELECT name FROM sqlite_master WHERE type=\'table\' AND name=?",\n        (table,)\n    ).fetchone()\n    return row is not None\n\n\ndef v15_columns(con, table):\n    if not v15_table_exists(con, table):\n        return []\n    return [r[1] for r in con.execute(f"PRAGMA table_info({table})").fetchall()]\n\n\ndef v15_first_match(patterns, text, flags=0):\n    for pattern in patterns:\n        m = re.search(pattern, text, flags)\n        if m:\n            return m.group(1).strip()\n    return None\n\n\n# ------------------------------\n# レース情報解析\n# ------------------------------\ndef v15_parse_race_meta(text):\n    text = v15_clean_text(text)\n    compact = re.sub(r"\\s+", " ", text)\n\n    date_raw = v15_first_match([\n        r"((?:20)?\\d{2})年\\s*(\\d{1,2})月\\s*(\\d{1,2})日",\n    ], compact)\n\n    # 上のfirst_matchでは複数groupを扱えないため、日付のみ個別処理\n    date_iso = None\n    dm = re.search(r"((?:20)?\\d{2})年\\s*(\\d{1,2})月\\s*(\\d{1,2})日", compact)\n    if dm:\n        y = int(dm.group(1))\n        if y < 100:\n            y += 2000\n        date_iso = f"{y:04d}-{int(dm.group(2)):02d}-{int(dm.group(3)):02d}"\n    else:\n        dm = re.search(r"(20\\d{2})[./-](\\d{1,2})[./-](\\d{1,2})", compact)\n        if dm:\n            date_iso = f"{int(dm.group(1)):04d}-{int(dm.group(2)):02d}-{int(dm.group(3)):02d}"\n\n    distance = v15_int(v15_first_match([\n        r"(\\d{4})\\s*m",\n        r"距離[:：]?\\s*(\\d{4})",\n    ], compact, re.I))\n\n    laps = v15_int(v15_first_match([\n        r"\\((\\d+)\\s*周\\)",\n        r"(\\d+)\\s*周",\n    ], compact))\n\n    start_time = v15_first_match([\n        r"発走(?:予定)?時間\\s*[:：]?\\s*(\\d{1,2}:\\d{2})",\n        r"発走\\s*[:：]?\\s*(\\d{1,2}:\\d{2})",\n    ], compact)\n\n    weather = v15_first_match([\n        r"天候\\s*[:：]\\s*([^\\s\u3000]+)",\n    ], compact)\n\n    surface = v15_first_match([\n        r"走路状況\\s*[:：]\\s*([^\\s\u3000]+)",\n        r"走路状態\\s*[:：]\\s*([^\\s\u3000]+)",\n    ], compact)\n\n    track_temp = v15_float(v15_first_match([\n        r"走路温度\\s*[:：]\\s*(-?\\d+(?:\\.\\d+)?)",\n    ], compact))\n\n    air_temp = v15_float(v15_first_match([\n        r"(?<!走路)気温\\s*[:：]\\s*(-?\\d+(?:\\.\\d+)?)",\n    ], compact))\n\n    humidity = v15_float(v15_first_match([\n        r"湿度\\s*[:：]\\s*(\\d+(?:\\.\\d+)?)",\n    ], compact))\n\n    race_no = v15_int(v15_first_match([\n        r"(?:第\\s*)?(\\d{1,2})\\s*R",\n        r"レース番号\\s*[:：]?\\s*(\\d{1,2})",\n    ], compact, re.I))\n\n    venue = v15_first_match([\n        r"(川口|伊勢崎|浜松|山陽|飯塚)\\s*(?:オート|走路|開催)?",\n        r"開催場\\s*[:：]?\\s*([^\\s\u3000]+)",\n    ], compact)\n\n    race_type = None\n    race_type_candidates = [\n        "SG優勝戦", "G1優勝戦", "G2優勝戦", "優勝戦",\n        "準決勝戦", "準決勝", "選抜戦", "特別選抜戦",\n        "一般戦", "予選", "最終予選", "二次予選", "一次予選"\n    ]\n    for label in race_type_candidates:\n        if label in compact:\n            race_type = label\n            break\n\n    race_title = None\n    title_match = re.search(\n        r"((?:一般戦|準決勝戦?|優勝戦|選抜戦|特別選抜戦|予選)[^0-9\\n]{0,20})\\s*(\\d{4})\\s*m",\n        compact\n    )\n    if title_match:\n        race_title = title_match.group(1).strip()\n\n    prize = v15_int(v15_first_match([\n        r"1着賞金\\s*([\\d,]+)\\s*円",\n    ], compact))\n\n    time_band = None\n    if start_time:\n        hour = int(start_time.split(":")[0])\n        if hour < 16:\n            time_band = "昼"\n        elif hour < 18:\n            time_band = "夕方"\n        else:\n            time_band = "夜"\n\n    meta = {\n        "開催日": date_iso,\n        "開催場": venue,\n        "レース": race_no,\n        "レース名": race_title or race_type,\n        "レース種別": race_type,\n        "距離": distance,\n        "周回数": laps,\n        "発走時刻": start_time,\n        "時間帯": time_band,\n        "天候": weather,\n        "走路状態": surface,\n        "走路温度": track_temp,\n        "気温": air_temp,\n        "湿度": humidity,\n        "1着賞金": prize,\n    }\n    return meta\n\n\n# ------------------------------\n# 出走表解析\n# ------------------------------\ndef v15_parse_entry_line(line):\n    line = v15_clean_text(line)\n    if not line:\n        return None\n\n    # タブ区切りを優先\n    parts = [p.strip() for p in re.split(r"\\t+| {2,}", line) if p.strip()]\n    first = re.match(r"^\\s*([1-8])(?:\\s+|[\u3000\\t])", line)\n    if not first:\n        # コピー時に車番と選手名が連結する場合\n        first = re.match(r"^\\s*([1-8])\\s*([^\\d\\s].+)", line)\n    if not first:\n        return None\n\n    car_no = int(first.group(1))\n\n    # よくある数値を拾う\n    handicap_match = re.search(r"(?<!\\d)(0|10|20|30|40|50|60|70|80)\\s*m?(?!\\d)", line)\n    trial_match = re.search(r"(?<!\\d)(3\\.\\d{2})(?!\\d)", line)\n    st_match = re.search(r"(?:ST|スタート)\\s*[:：]?\\s*([+-]?\\d?\\.\\d{2,3})", line, re.I)\n\n    # 名前候補。車番の後から、ハンデ・タイム等の前まで\n    body = re.sub(r"^\\s*[1-8]\\s*", "", line).strip()\n    stop_positions = []\n    for pat in [\n        r"\\s(?:0|10|20|30|40|50|60|70|80)\\s*m?(?:\\s|$)",\n        r"\\s3\\.\\d{2}(?:\\s|$)",\n        r"\\s[AB]\\d(?:\\s|$)",\n        r"\\s\\d{1,2}期(?:\\s|$)",\n    ]:\n        m = re.search(pat, body)\n        if m:\n            stop_positions.append(m.start())\n    name_area = body[:min(stop_positions)] if stop_positions else body\n\n    # 級別・年齢・期別のような末尾情報を除外\n    name_area = re.sub(r"\\s+[AB]\\d.*$", "", name_area).strip()\n    name_area = re.sub(r"\\s+\\d{1,2}期.*$", "", name_area).strip()\n    name_area = re.sub(r"\\s+\\d{2}歳.*$", "", name_area).strip()\n\n    # 名前が取れないときは分割部品から探索\n    if len(name_area) < 2 and len(parts) >= 2:\n        name_area = parts[1]\n\n    name = v15_normalize_name(name_area)\n    if not name or name in {"車番", "選手名", "枠番"}:\n        return None\n\n    age = v15_int(v15_first_match([r"(\\d{2})\\s*歳"], line))\n    grade = v15_first_match([r"\\b([AB][12])\\b"], line)\n    term = v15_int(v15_first_match([r"(\\d{1,2})\\s*期"], line))\n\n    return {\n        "車番": car_no,\n        "選手名": name,\n        "ハンデ": v15_int(handicap_match.group(1)) if handicap_match else None,\n        "試走T": v15_float(trial_match.group(1)) if trial_match else np.nan,\n        "ST": v15_float(st_match.group(1)) if st_match else np.nan,\n        "年齢": age,\n        "級別": grade,\n        "期別": term,\n    }\n\n\ndef v15_parse_entries(text):\n    rows = []\n    seen = set()\n\n    for raw in v15_clean_text(text).splitlines():\n        row = v15_parse_entry_line(raw)\n        if not row:\n            continue\n        key = row["車番"]\n        if key in seen:\n            continue\n        seen.add(key)\n        rows.append(row)\n\n    df = pd.DataFrame(rows, columns=V15_ENTRY_COLUMNS)\n    if not df.empty:\n        df = df.sort_values("車番").reset_index(drop=True)\n    return df\n\n\ndef v15_parse_prediction_page(text):\n    meta = v15_parse_race_meta(text)\n    entries = v15_parse_entries(text)\n    return meta, entries\n\n\n# ------------------------------\n# 選手履歴解析\n# ------------------------------\ndef v15_guess_history_row(line):\n    original = line\n    line = v15_clean_text(line)\n\n    dm = re.search(r"(20\\d{2})[./年-](\\d{1,2})[./月-](\\d{1,2})", line)\n    if not dm:\n        return None\n    date_iso = f"{int(dm.group(1)):04d}-{int(dm.group(2)):02d}-{int(dm.group(3)):02d}"\n\n    venue = None\n    for v in ["川口", "伊勢崎", "浜松", "山陽", "飯塚"]:\n        if v in line:\n            venue = v\n            break\n\n    race_no = v15_int(v15_first_match([r"(\\d{1,2})\\s*R"], line, re.I))\n\n    rank = None\n    rank_match = re.search(r"(?:^|\\s)([1-8])\\s*着(?:\\s|$)", line)\n    if rank_match:\n        rank = int(rank_match.group(1))\n    else:\n        # 表形式では着順が単独数字のことがある\n        nums = re.findall(r"(?:^|\\s)([1-8])(?:\\s|$)", line)\n        if nums:\n            rank = int(nums[-1])\n\n    surface = None\n    for s in ["良走路", "湿走路", "斑走路", "風走路", "良", "湿", "斑"]:\n        if s in line:\n            surface = s\n            break\n\n    handicap = None\n    hm = re.search(r"(?<!\\d)(0|10|20|30|40|50|60|70|80)\\s*m?(?!\\d)", line)\n    if hm:\n        handicap = int(hm.group(1))\n\n    times = [float(x) for x in re.findall(r"(?<!\\d)(3\\.\\d{2}|4\\.\\d{2}|[12]\\.\\d{2})(?!\\d)", line)]\n    trial = times[0] if times else np.nan\n    race_time = times[1] if len(times) >= 2 else np.nan\n\n    st = np.nan\n    stm = re.search(r"(?:ST|スタート)\\s*[:：]?\\s*([+-]?\\d?\\.\\d{2,3})", line, re.I)\n    if stm:\n        st = float(stm.group(1))\n    elif len(times) >= 3:\n        st = times[-1]\n\n    starters = v15_int(v15_first_match([\n        r"(\\d)\\s*車",\n        r"出走\\s*[:：]?\\s*(\\d)",\n    ], line))\n\n    return {\n        "開催日": date_iso,\n        "開催場": venue,\n        "レース": race_no,\n        "着順": rank,\n        "出走": starters,\n        "走路": surface,\n        "ハンデ": handicap,\n        "試走T": trial,\n        "競走T": race_time,\n        "ST": st,\n        "_raw": original,\n    }\n\n\ndef v15_parse_player_history(text, player_name=None):\n    rows = []\n    for line in v15_clean_text(text).splitlines():\n        row = v15_guess_history_row(line)\n        if row:\n            rows.append(row)\n\n    df = pd.DataFrame(rows)\n    if df.empty:\n        return pd.DataFrame(columns=["選手名"] + V15_HISTORY_COLUMNS)\n\n    if player_name:\n        df.insert(0, "選手名", v15_normalize_name(player_name))\n    else:\n        # ページ先頭の氏名らしき文字列\n        guessed = v15_first_match([\n            r"選手名\\s*[:：]\\s*([^\\n]+)",\n            r"プロフィール\\s+([^\\n]+)",\n        ], v15_clean_text(text))\n        df.insert(0, "選手名", v15_normalize_name(guessed) if guessed else "")\n\n    return df[["選手名"] + V15_HISTORY_COLUMNS + ["_raw"]]\n\n\n# ------------------------------\n# Ver15用DB\n# ------------------------------\ndef v15_init_tables(db_path=DB_PATH):\n    with sqlite3.connect(db_path) as con:\n        con.execute("""\n            CREATE TABLE IF NOT EXISTS v15_race_inputs (\n                race_key TEXT PRIMARY KEY,\n                race_date TEXT,\n                venue TEXT,\n                race_no INTEGER,\n                race_name TEXT,\n                race_type TEXT,\n                distance INTEGER,\n                laps INTEGER,\n                start_time TEXT,\n                time_band TEXT,\n                weather TEXT,\n                surface TEXT,\n                track_temp REAL,\n                air_temp REAL,\n                humidity REAL,\n                first_prize INTEGER,\n                raw_text TEXT,\n                created_at TEXT\n            )\n        """)\n        con.execute("""\n            CREATE TABLE IF NOT EXISTS v15_race_entry_inputs (\n                race_key TEXT,\n                car_no INTEGER,\n                player_name TEXT,\n                handicap INTEGER,\n                trial_time REAL,\n                st REAL,\n                age INTEGER,\n                grade TEXT,\n                term INTEGER,\n                created_at TEXT,\n                PRIMARY KEY (race_key, car_no)\n            )\n        """)\n        con.execute("""\n            CREATE TABLE IF NOT EXISTS v15_similarity_weights (\n                feature_name TEXT PRIMARY KEY,\n                initial_weight REAL,\n                current_weight REAL,\n                updated_at TEXT,\n                reason TEXT\n            )\n        """)\n        con.execute("""\n            CREATE TABLE IF NOT EXISTS v15_player_history_imports (\n                history_key TEXT PRIMARY KEY,\n                player_name TEXT,\n                race_date TEXT,\n                venue TEXT,\n                race_no INTEGER,\n                rank INTEGER,\n                starters INTEGER,\n                surface TEXT,\n                handicap INTEGER,\n                trial_time REAL,\n                race_time REAL,\n                st REAL,\n                raw_line TEXT,\n                created_at TEXT\n            )\n        """)\n\n        defaults = {\n            "走路状態": 0.22,\n            "走路温度": 0.18,\n            "ハンデ構成": 0.18,\n            "開催場": 0.12,\n            "距離": 0.10,\n            "湿度": 0.08,\n            "気温": 0.06,\n            "レース種別": 0.04,\n            "発走時間": 0.02,\n        }\n        now = datetime.now().isoformat(timespec="seconds")\n        for name, weight in defaults.items():\n            con.execute("""\n                INSERT OR IGNORE INTO v15_similarity_weights\n                (feature_name, initial_weight, current_weight, updated_at, reason)\n                VALUES (?, ?, ?, ?, ?)\n            """, (name, weight, weight, now, "Ver15.0初期値"))\n        con.commit()\n\n\ndef v15_race_key(meta):\n    return v15_hash(\n        meta.get("開催日"), meta.get("開催場"), meta.get("レース"),\n        meta.get("距離"), meta.get("発走時刻")\n    )\n\n\ndef v15_save_race_input(meta, entries, raw_text, db_path=DB_PATH):\n    v15_init_tables(db_path)\n    key = v15_race_key(meta)\n    now = datetime.now().isoformat(timespec="seconds")\n\n    with sqlite3.connect(db_path) as con:\n        con.execute("""\n            INSERT OR REPLACE INTO v15_race_inputs (\n                race_key, race_date, venue, race_no, race_name, race_type,\n                distance, laps, start_time, time_band, weather, surface,\n                track_temp, air_temp, humidity, first_prize, raw_text, created_at\n            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n        """, (\n            key, meta.get("開催日"), meta.get("開催場"), meta.get("レース"),\n            meta.get("レース名"), meta.get("レース種別"), meta.get("距離"),\n            meta.get("周回数"), meta.get("発走時刻"), meta.get("時間帯"),\n            meta.get("天候"), meta.get("走路状態"), meta.get("走路温度"),\n            meta.get("気温"), meta.get("湿度"), meta.get("1着賞金"),\n            raw_text, now\n        ))\n\n        for _, row in entries.iterrows():\n            con.execute("""\n                INSERT OR REPLACE INTO v15_race_entry_inputs (\n                    race_key, car_no, player_name, handicap, trial_time,\n                    st, age, grade, term, created_at\n                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n            """, (\n                key, int(row["車番"]), row["選手名"],\n                None if pd.isna(row["ハンデ"]) else int(row["ハンデ"]),\n                None if pd.isna(row["試走T"]) else float(row["試走T"]),\n                None if pd.isna(row["ST"]) else float(row["ST"]),\n                None if pd.isna(row["年齢"]) else int(row["年齢"]),\n                None if pd.isna(row["級別"]) else str(row["級別"]),\n                None if pd.isna(row["期別"]) else int(row["期別"]),\n                now\n            ))\n        con.commit()\n    return key\n\n\ndef v15_save_player_history(df, db_path=DB_PATH):\n    if df is None or df.empty:\n        return 0, 0\n\n    v15_init_tables(db_path)\n    inserted = 0\n    skipped = 0\n    now = datetime.now().isoformat(timespec="seconds")\n\n    with sqlite3.connect(db_path) as con:\n        for _, row in df.iterrows():\n            key = v15_hash(\n                row.get("選手名"), row.get("開催日"), row.get("開催場"),\n                row.get("レース"), row.get("着順"), row.get("試走T"),\n                row.get("競走T")\n            )\n            exists = con.execute(\n                "SELECT 1 FROM v15_player_history_imports WHERE history_key=?",\n                (key,)\n            ).fetchone()\n            if exists:\n                skipped += 1\n                continue\n\n            con.execute("""\n                INSERT INTO v15_player_history_imports (\n                    history_key, player_name, race_date, venue, race_no,\n                    rank, starters, surface, handicap, trial_time,\n                    race_time, st, raw_line, created_at\n                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n            """, (\n                key, row.get("選手名"), row.get("開催日"),\n                row.get("開催場"),\n                None if pd.isna(row.get("レース")) else int(row.get("レース")),\n                None if pd.isna(row.get("着順")) else int(row.get("着順")),\n                None if pd.isna(row.get("出走")) else int(row.get("出走")),\n                row.get("走路"),\n                None if pd.isna(row.get("ハンデ")) else int(row.get("ハンデ")),\n                None if pd.isna(row.get("試走T")) else float(row.get("試走T")),\n                None if pd.isna(row.get("競走T")) else float(row.get("競走T")),\n                None if pd.isna(row.get("ST")) else float(row.get("ST")),\n                row.get("_raw"), now\n            ))\n            inserted += 1\n        con.commit()\n\n    return inserted, skipped\n\n\n# ------------------------------\n# 類似レース候補\n# ------------------------------\ndef v15_similarity_weights(db_path=DB_PATH):\n    v15_init_tables(db_path)\n    with sqlite3.connect(db_path) as con:\n        df = pd.read_sql_query(\n            "SELECT feature_name, current_weight FROM v15_similarity_weights",\n            con\n        )\n    return dict(zip(df["feature_name"], df["current_weight"]))\n\n\ndef v15_handicap_signature(entries):\n    if entries is None or entries.empty or "ハンデ" not in entries:\n        return {}\n    vals = pd.to_numeric(entries["ハンデ"], errors="coerce").dropna().astype(int)\n    return vals.value_counts().sort_index().to_dict()\n\n\ndef v15_numeric_similarity(a, b, scale):\n    if a is None or b is None or pd.isna(a) or pd.isna(b):\n        return None\n    return max(0.0, 1.0 - abs(float(a) - float(b)) / float(scale))\n\n\ndef v15_categorical_similarity(a, b):\n    if not a or not b:\n        return None\n    return 1.0 if str(a) == str(b) else 0.0\n\n\ndef v15_handicap_similarity(sig_a, sig_b):\n    if not sig_a or not sig_b:\n        return None\n    keys = sorted(set(sig_a) | set(sig_b))\n    va = np.array([sig_a.get(k, 0) for k in keys], dtype=float)\n    vb = np.array([sig_b.get(k, 0) for k in keys], dtype=float)\n    denom = max(1.0, va.sum(), vb.sum())\n    return max(0.0, 1.0 - np.abs(va - vb).sum() / denom)\n\n\ndef v15_find_similar_races(meta, entries, top_n=10, db_path=DB_PATH):\n    v15_init_tables(db_path)\n    weights = v15_similarity_weights(db_path)\n    current_key = v15_race_key(meta)\n    current_sig = v15_handicap_signature(entries)\n\n    with sqlite3.connect(db_path) as con:\n        races = pd.read_sql_query("SELECT * FROM v15_race_inputs", con)\n        all_entries = pd.read_sql_query(\n            "SELECT race_key, handicap FROM v15_race_entry_inputs", con\n        )\n\n    if races.empty:\n        return pd.DataFrame()\n\n    sig_map = {}\n    if not all_entries.empty:\n        for key, g in all_entries.groupby("race_key"):\n            vals = pd.to_numeric(g["handicap"], errors="coerce").dropna().astype(int)\n            sig_map[key] = vals.value_counts().sort_index().to_dict()\n\n    rows = []\n    for _, r in races.iterrows():\n        if r["race_key"] == current_key:\n            continue\n\n        feature_scores = {\n            "走路状態": v15_categorical_similarity(meta.get("走路状態"), r["surface"]),\n            "走路温度": v15_numeric_similarity(meta.get("走路温度"), r["track_temp"], 25),\n            "ハンデ構成": v15_handicap_similarity(current_sig, sig_map.get(r["race_key"], {})),\n            "開催場": v15_categorical_similarity(meta.get("開催場"), r["venue"]),\n            "距離": v15_numeric_similarity(meta.get("距離"), r["distance"], 1000),\n            "湿度": v15_numeric_similarity(meta.get("湿度"), r["humidity"], 50),\n            "気温": v15_numeric_similarity(meta.get("気温"), r["air_temp"], 20),\n            "レース種別": v15_categorical_similarity(meta.get("レース種別"), r["race_type"]),\n            "発走時間": None,\n        }\n\n        if meta.get("発走時刻") and r["start_time"]:\n            try:\n                h1, m1 = map(int, meta["発走時刻"].split(":"))\n                h2, m2 = map(int, str(r["start_time"]).split(":"))\n                feature_scores["発走時間"] = max(\n                    0.0, 1.0 - abs((h1 * 60 + m1) - (h2 * 60 + m2)) / 360\n                )\n            except Exception:\n                pass\n\n        usable = {\n            k: v for k, v in feature_scores.items()\n            if v is not None and k in weights\n        }\n        total_w = sum(weights[k] for k in usable)\n        if total_w <= 0:\n            continue\n\n        score = sum(weights[k] * usable[k] for k in usable) / total_w\n        strongest = sorted(\n            usable.items(),\n            key=lambda kv: weights[kv[0]] * kv[1],\n            reverse=True\n        )[:3]\n\n        rows.append({\n            "開催日": r["race_date"],\n            "開催場": r["venue"],\n            "レース": r["race_no"],\n            "レース種別": r["race_type"],\n            "距離": r["distance"],\n            "類似度": round(score * 100, 1),\n            "主な一致条件": "・".join(k for k, _ in strongest),\n        })\n\n    if not rows:\n        return pd.DataFrame()\n\n    return (\n        pd.DataFrame(rows)\n        .sort_values("類似度", ascending=False)\n        .head(top_n)\n        .reset_index(drop=True)\n    )\n\n\n# ------------------------------\n# 予測セルへの引き渡し\n# ------------------------------\ndef v15_apply_to_prediction(meta, entries):\n    global V15_RACE_META, V15_ENTRY_DF\n    V15_RACE_META = dict(meta)\n    V15_ENTRY_DF = entries.copy()\n\n    # 既存の予測ロジックが参照しやすい別名も用意\n    globals()["CURRENT_RACE_META"] = V15_RACE_META\n    globals()["CURRENT_ENTRY_DF"] = V15_ENTRY_DF\n\n    # 既存LATEST_TRACK_TEMPへ反映\n    if pd.notna(meta.get("走路温度")):\n        globals()["LATEST_TRACK_TEMP"] = float(meta["走路温度"])\n\n    return V15_RACE_META, V15_ENTRY_DF\n\n\n# ------------------------------\n# 結果全文をVer13.2へ渡す\n# ------------------------------\ndef v15_parse_result_preview(text):\n    if "parse_full_official_result" in globals():\n        return parse_full_official_result(text)\n    return {\n        "error": "Ver13.2のparse_full_official_result関数が見つかりません。",\n        "raw_text": text\n    }\n\n\ndef v15_save_result_using_existing(text):\n    if "parse_full_official_result" not in globals():\n        raise RuntimeError("Ver13.2の結果解析関数が読み込まれていません。上から順にセルを実行してください。")\n    parsed = parse_full_official_result(text)\n\n    if "save_full_official_result" not in globals():\n        raise RuntimeError("Ver13.2の結果保存関数が読み込まれていません。")\n\n    # 既存関数の引数差に耐える\n    try:\n        return save_full_official_result(parsed)\n    except TypeError:\n        return save_full_official_result(parsed, DB_PATH)\n\n\n# ------------------------------\n# UI\n# ------------------------------\nv15_init_tables()\n\nstyle = {"description_width": "110px"}\nwide_layout = widgets.Layout(width="100%", min_height="220px")\nbutton_layout = widgets.Layout(width="180px", height="42px")\n\nprediction_text = widgets.Textarea(\n    description="予測ページ全文",\n    placeholder="公式サイトのレース情報・出走表をまとめて貼り付け",\n    layout=wide_layout,\n    style=style,\n)\nbtn_prediction_preview = widgets.Button(\n    description="① 解析プレビュー",\n    button_style="info",\n    layout=button_layout,\n)\nbtn_prediction_save = widgets.Button(\n    description="② DB保存",\n    button_style="success",\n    layout=button_layout,\n)\nbtn_prediction_apply = widgets.Button(\n    description="③ 予測へ反映",\n    button_style="primary",\n    layout=button_layout,\n)\nbtn_similar = widgets.Button(\n    description="類似レース表示",\n    layout=button_layout,\n)\nout_prediction = widgets.Output()\n\nplayer_name_input = widgets.Text(\n    description="選手名",\n    placeholder="ページから取れない場合だけ入力",\n    layout=widgets.Layout(width="100%"),\n    style=style,\n)\nplayer_text = widgets.Textarea(\n    description="選手履歴全文",\n    placeholder="選手ページの過去成績表を貼り付け",\n    layout=wide_layout,\n    style=style,\n)\nbtn_player_preview = widgets.Button(\n    description="① 履歴プレビュー",\n    button_style="info",\n    layout=button_layout,\n)\nbtn_player_save = widgets.Button(\n    description="② 履歴DB追加",\n    button_style="success",\n    layout=button_layout,\n)\nout_player = widgets.Output()\n\nresult_text = widgets.Textarea(\n    description="結果ページ全文",\n    placeholder="公式結果ページを最初から周回順位までまとめて貼り付け",\n    layout=wide_layout,\n    style=style,\n)\nbtn_result_preview = widgets.Button(\n    description="① 結果プレビュー",\n    button_style="info",\n    layout=button_layout,\n)\nbtn_result_save = widgets.Button(\n    description="② 結果を登録",\n    button_style="success",\n    layout=button_layout,\n)\nout_result = widgets.Output()\n\n\ndef _v15_prediction_preview(_):\n    with out_prediction:\n        _hidden_clear_output()\n        text = prediction_text.value\n        meta, entries = v15_parse_prediction_page(text)\n        V15_STATE["race_meta"] = meta\n        V15_STATE["entries"] = entries\n        V15_STATE["raw_prediction_text"] = text\n\n        print("【解析したレース情報】")\n        _hidden_display(pd.DataFrame([meta]).T.rename(columns={0: "値"}))\n\n        print("\\n【解析した出走選手】")\n        if entries.empty:\n            print("出走選手を読み取れませんでした。車番から始まる行を含めて貼り付けてください。")\n        else:\n            _hidden_display(entries)\n\n        missing = [k for k in ["開催日", "開催場", "レース", "距離"] if not meta.get(k)]\n        if missing:\n            print("\\n⚠ 未取得項目:", "、".join(missing))\n        print("\\n内容を確認してから「DB保存」または「予測へ反映」を押してください。")\n\n\ndef _v15_prediction_save(_):\n    with out_prediction:\n        if not V15_STATE["race_meta"]:\n            print("先に解析プレビューを実行してください。")\n            return\n        key = v15_save_race_input(\n            V15_STATE["race_meta"],\n            V15_STATE["entries"],\n            V15_STATE["raw_prediction_text"],\n        )\n        print(f"✅ レース入力をDBへ保存しました。登録キー: {key}")\n\n\ndef _v15_prediction_apply(_):\n    with out_prediction:\n        if not V15_STATE["race_meta"]:\n            print("先に解析プレビューを実行してください。")\n            return\n        meta, entries = v15_apply_to_prediction(\n            V15_STATE["race_meta"],\n            V15_STATE["entries"],\n        )\n        print("✅ 解析内容を予測用変数へ反映しました。")\n        print("利用変数: CURRENT_RACE_META / CURRENT_ENTRY_DF")\n        if entries.empty:\n            print("⚠ 出走表が空です。既存Excel入力と組み合わせて利用してください。")\n\n\ndef _v15_similar(_):\n    with out_prediction:\n        if not V15_STATE["race_meta"]:\n            print("先に解析プレビューを実行してください。")\n            return\n        df = v15_find_similar_races(\n            V15_STATE["race_meta"],\n            V15_STATE["entries"],\n            top_n=10,\n        )\n        print("【類似レース候補】")\n        if df.empty:\n            print("比較できる保存済みレースがまだありません。")\n        else:\n            _hidden_display(df)\n            print("※ 類似度は初期重みによる参考値です。結果蓄積後に重みを再学習します。")\n\n\ndef _v15_player_preview(_):\n    with out_player:\n        _hidden_clear_output()\n        df = v15_parse_player_history(\n            player_text.value,\n            player_name=player_name_input.value.strip() or None,\n        )\n        V15_STATE["player_history"] = df\n        V15_STATE["raw_player_text"] = player_text.value\n        if df.empty:\n            print("履歴行を読み取れませんでした。開催日を含む成績表を貼り付けてください。")\n        else:\n            print(f"解析件数: {len(df)}件")\n            _hidden_display(df.drop(columns=["_raw"], errors="ignore"))\n            if (df["選手名"].fillna("").str.strip() == "").any():\n                print("⚠ 選手名が空です。上の選手名欄へ入力して、もう一度プレビューしてください。")\n\n\ndef _v15_player_save(_):\n    with out_player:\n        df = V15_STATE.get("player_history")\n        if df is None or df.empty:\n            print("先に履歴プレビューを実行してください。")\n            return\n        if (df["選手名"].fillna("").str.strip() == "").any():\n            print("選手名が空のため保存できません。")\n            return\n        inserted, skipped = v15_save_player_history(df)\n        print(f"✅ 新規追加: {inserted}件 / 重複スキップ: {skipped}件")\n\n\ndef _v15_result_preview(_):\n    with out_result:\n        _hidden_clear_output()\n        V15_STATE["raw_result_text"] = result_text.value\n        parsed = v15_parse_result_preview(result_text.value)\n        print("【結果解析プレビュー】")\n        if isinstance(parsed, dict):\n            summary = {}\n            for k, v in parsed.items():\n                if isinstance(v, pd.DataFrame):\n                    print(f"\\n{k}")\n                    _hidden_display(v)\n                elif isinstance(v, (list, tuple)) and v and isinstance(v[0], dict):\n                    print(f"\\n{k}")\n                    _hidden_display(pd.DataFrame(v))\n                elif k != "raw_text":\n                    summary[k] = v\n            if summary:\n                _hidden_display(pd.DataFrame([summary]).T.rename(columns={0: "値"}))\n        else:\n            _hidden_display(parsed)\n        print("\\n内容を確認してから「結果を登録」を押してください。")\n\n\ndef _v15_result_save(_):\n    with out_result:\n        if not result_text.value.strip():\n            print("結果ページ全文を貼り付けてください。")\n            return\n        try:\n            result = v15_save_result_using_existing(result_text.value)\n            print("✅ 公式結果をDBへ登録しました。")\n            if result is not None:\n                _hidden_display(result) if isinstance(result, pd.DataFrame) else print(result)\n        except Exception as exc:\n            print("❌ 登録できませんでした。")\n            print(type(exc).__name__, str(exc))\n\n\nbtn_prediction_preview.on_click(_v15_prediction_preview)\nbtn_prediction_save.on_click(_v15_prediction_save)\nbtn_prediction_apply.on_click(_v15_prediction_apply)\nbtn_similar.on_click(_v15_similar)\nbtn_player_preview.on_click(_v15_player_preview)\nbtn_player_save.on_click(_v15_player_save)\nbtn_result_preview.on_click(_v15_result_preview)\nbtn_result_save.on_click(_v15_result_save)\n\nprediction_box = widgets.VBox([\n    widgets.HTML("<h3>📋 予測前：公式ページ全文を貼り付け</h3>"),\n    prediction_text,\n    widgets.HBox([\n        btn_prediction_preview,\n        btn_prediction_save,\n        btn_prediction_apply,\n        btn_similar,\n    ]),\n    out_prediction,\n])\n\nplayer_box = widgets.VBox([\n    widgets.HTML("<h3>👤 選手履歴：選手ページを貼り付け</h3>"),\n    player_name_input,\n    player_text,\n    widgets.HBox([btn_player_preview, btn_player_save]),\n    out_player,\n])\n\nresult_box = widgets.VBox([\n    widgets.HTML("<h3>🏁 レース後：公式結果ページ全文を貼り付け</h3>"),\n    result_text,\n    widgets.HBox([btn_result_preview, btn_result_save]),\n    out_result,\n])\n\nv15_tabs = widgets.Tab(children=[prediction_box, player_box, result_box])\nv15_tabs.set_title(0, "予測入力")\nv15_tabs.set_title(1, "選手履歴")\nv15_tabs.set_title(2, "結果登録")\n\n_hidden_display(HTML("""\n<div style="padding:12px;border:1px solid #bbb;border-radius:10px;margin-bottom:10px">\n<b>AutoRaceAI Ver15.0</b><br>\nスマホでは公式ページを長押しコピーし、各タブの欄へ貼り付けます。\n必ずプレビューで解析内容を確認してから保存してください。\n</div>\n"""))\n_hidden_display(v15_tabs)\n\n# ============================================================\n# Ver15.1 縦型選手履歴コピペ対応\n# 「前走」「前々走」「3走前」...ごとに改行された形式を解析\n# ============================================================\n\ndef v151_parse_date(line):\n    m = re.search(r"(20\\d{2})年\\s*(\\d{1,2})月\\s*(\\d{1,2})日", line)\n    if not m:\n        return None\n    return f"{int(m.group(1)):04d}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"\n\n\ndef v151_is_history_header(line):\n    line = line.strip()\n    return bool(re.fullmatch(r"(?:前走|前々走|\\d+走前)", line))\n\n\ndef v151_split_history_blocks(text):\n    lines = [x.strip() for x in v15_clean_text(text).splitlines() if x.strip()]\n    blocks = []\n    current = None\n\n    for line in lines:\n        if v151_is_history_header(line):\n            if current:\n                blocks.append(current)\n            current = {"見出し": line, "lines": []}\n        elif current is not None:\n            current["lines"].append(line)\n\n    if current:\n        blocks.append(current)\n\n    return blocks\n\n\ndef v151_parse_history_block(block, player_name=""):\n    lines = block.get("lines", [])\n    if not lines:\n        return None\n\n    row = {\n        "選手名": v15_normalize_name(player_name),\n        "開催日": None,\n        "開催場": None,\n        "レース": None,\n        "着順": None,\n        "出走": None,\n        "走路": None,\n        "ハンデ": None,\n        "試走T": np.nan,\n        "競走T": np.nan,\n        "ST": np.nan,\n        "天候": None,\n        "走路温度": np.nan,\n        "気温": np.nan,\n        "湿度": np.nan,\n        "レース種別": None,\n        "距離": None,\n        "周回数": None,\n        "人気": None,\n        "車番": None,\n        "_raw": "\\n".join([block.get("見出し", "")] + lines),\n    }\n\n    # 先頭の単独数字は着順として扱う\n    for i, line in enumerate(lines):\n        if re.fullmatch(r"[1-8]", line):\n            row["着順"] = int(line)\n            break\n\n    for line in lines:\n        # 開催日\n        if row["開催日"] is None:\n            d = v151_parse_date(line)\n            if d:\n                row["開催日"] = d\n                continue\n\n        # 開催場\n        if line in ["川口", "伊勢崎", "浜松", "山陽", "飯塚"]:\n            row["開催場"] = line\n            continue\n\n        # レース種別\n        if any(x in line for x in [\n            "一般戦", "予選", "準決勝", "準決勝戦", "優勝戦",\n            "選抜戦", "特別選抜戦", "最終予選", "二次予選", "一次予選"\n        ]):\n            row["レース種別"] = line\n            continue\n\n        # 天候\n        if line in ["晴", "曇", "雨", "雪", "小雨"]:\n            row["天候"] = line\n            continue\n\n        # 走路\n        if line in ["良", "湿", "斑", "良走路", "湿走路", "斑走路"]:\n            row["走路"] = line\n            continue\n\n        # 走路温度\n        m = re.fullmatch(r"走\\s*(-?\\d+(?:\\.\\d+)?)", line)\n        if m:\n            row["走路温度"] = float(m.group(1))\n            continue\n\n        # 気温\n        m = re.fullmatch(r"気\\s*(-?\\d+(?:\\.\\d+)?)", line)\n        if m:\n            row["気温"] = float(m.group(1))\n            continue\n\n        # 湿度\n        m = re.fullmatch(r"湿\\s*(\\d+(?:\\.\\d+)?)", line)\n        if m:\n            row["湿度"] = float(m.group(1))\n            continue\n\n        # 車番・ハンデ\n        # 例: 1番-m / 3番10m / 8番20m\n        m = re.fullmatch(r"([1-8])番\\s*([+-]?\\d+|-)?m?", line)\n        if m:\n            row["車番"] = int(m.group(1))\n            if m.group(2) in (None, "-"):\n                row["ハンデ"] = 0\n            else:\n                row["ハンデ"] = int(m.group(2))\n            continue\n\n        # 距離・周回\n        m = re.fullmatch(r"(\\d{4})m\\((\\d+)周\\)", line)\n        if m:\n            row["距離"] = int(m.group(1))\n            row["周回数"] = int(m.group(2))\n            continue\n\n        # 人気\n        m = re.fullmatch(r"(\\d+)人気", line)\n        if m:\n            row["人気"] = int(m.group(1))\n            continue\n\n        # 競走タイム\n        m = re.fullmatch(r"([3-9]\\.\\d{3})", line)\n        if m:\n            row["競走T"] = float(m.group(1))\n            continue\n\n        # 試走\n        m = re.fullmatch(r"試\\s*([3-9]\\.\\d{2,3})", line)\n        if m:\n            row["試走T"] = float(m.group(1))\n            continue\n\n        # ST\n        m = re.fullmatch(r"ST\\s*([+-]?\\d?\\.\\d{2,3})", line, re.I)\n        if m:\n            row["ST"] = float(m.group(1))\n            continue\n\n    # 出走数は貼り付け情報にないため、通常8車として補完しない\n    # 必要なら結果ページ登録時に上書き\n    return row\n\n\ndef v151_parse_vertical_player_history(text, player_name=None):\n    blocks = v151_split_history_blocks(text)\n    if not blocks:\n        return pd.DataFrame()\n\n    rows = []\n    for block in blocks:\n        row = v151_parse_history_block(block, player_name=player_name or "")\n        if row:\n            rows.append(row)\n\n    return pd.DataFrame(rows)\n\n\n# 既存関数を上書きし、縦型と表形式の両方に対応\ndef v15_parse_player_history(text, player_name=None):\n    vertical = v151_parse_vertical_player_history(text, player_name=player_name)\n\n    # 縦型が2件以上取れたらこちらを採用\n    if not vertical.empty and vertical["開催日"].notna().sum() >= 1:\n        expected = [\n            "選手名", "開催日", "開催場", "レース", "着順", "出走", "走路",\n            "ハンデ", "試走T", "競走T", "ST",\n            "天候", "走路温度", "気温", "湿度",\n            "レース種別", "距離", "周回数", "人気", "車番", "_raw"\n        ]\n        for col in expected:\n            if col not in vertical.columns:\n                vertical[col] = np.nan\n        return vertical[expected]\n\n    # 従来の1行1レース形式へフォールバック\n    rows = []\n    for line in v15_clean_text(text).splitlines():\n        row = v15_guess_history_row(line)\n        if row:\n            rows.append(row)\n\n    df = pd.DataFrame(rows)\n    if df.empty:\n        return pd.DataFrame(columns=["選手名"] + V15_HISTORY_COLUMNS)\n\n    if player_name:\n        df.insert(0, "選手名", v15_normalize_name(player_name))\n    else:\n        guessed = v15_first_match([\n            r"選手名\\s*[:：]\\s*([^\\n]+)",\n            r"プロフィール\\s+([^\\n]+)",\n        ], v15_clean_text(text))\n        df.insert(0, "選手名", v15_normalize_name(guessed) if guessed else "")\n\n    return df[["選手名"] + V15_HISTORY_COLUMNS + ["_raw"]]\n\n\n# DBに追加列を安全に追加\ndef v151_ensure_player_import_columns(db_path=DB_PATH):\n    v15_init_tables(db_path)\n    extra_columns = {\n        "weather": "TEXT",\n        "track_temp": "REAL",\n        "air_temp": "REAL",\n        "humidity": "REAL",\n        "race_type": "TEXT",\n        "distance": "INTEGER",\n        "laps": "INTEGER",\n        "popularity": "INTEGER",\n        "car_no": "INTEGER",\n    }\n\n    with sqlite3.connect(db_path) as con:\n        cols = set(v15_columns(con, "v15_player_history_imports"))\n        for name, sql_type in extra_columns.items():\n            if name not in cols:\n                con.execute(\n                    f"ALTER TABLE v15_player_history_imports ADD COLUMN {name} {sql_type}"\n                )\n        con.commit()\n\n\ndef v15_save_player_history(df, db_path=DB_PATH):\n    if df is None or df.empty:\n        return 0, 0\n\n    v151_ensure_player_import_columns(db_path)\n    inserted = 0\n    skipped = 0\n    now = datetime.now().isoformat(timespec="seconds")\n\n    with sqlite3.connect(db_path) as con:\n        for _, row in df.iterrows():\n            key = v15_hash(\n                row.get("選手名"), row.get("開催日"), row.get("開催場"),\n                row.get("レース"), row.get("着順"), row.get("試走T"),\n                row.get("競走T"), row.get("車番")\n            )\n\n            exists = con.execute(\n                "SELECT 1 FROM v15_player_history_imports WHERE history_key=?",\n                (key,)\n            ).fetchone()\n            if exists:\n                skipped += 1\n                continue\n\n            con.execute("""\n                INSERT INTO v15_player_history_imports (\n                    history_key, player_name, race_date, venue, race_no,\n                    rank, starters, surface, handicap, trial_time,\n                    race_time, st, raw_line, created_at,\n                    weather, track_temp, air_temp, humidity,\n                    race_type, distance, laps, popularity, car_no\n                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)\n            """, (\n                key,\n                row.get("選手名"),\n                row.get("開催日"),\n                row.get("開催場"),\n                None if pd.isna(row.get("レース")) else int(row.get("レース")),\n                None if pd.isna(row.get("着順")) else int(row.get("着順")),\n                None if pd.isna(row.get("出走")) else int(row.get("出走")),\n                row.get("走路"),\n                None if pd.isna(row.get("ハンデ")) else int(row.get("ハンデ")),\n                None if pd.isna(row.get("試走T")) else float(row.get("試走T")),\n                None if pd.isna(row.get("競走T")) else float(row.get("競走T")),\n                None if pd.isna(row.get("ST")) else float(row.get("ST")),\n                row.get("_raw"),\n                now,\n                row.get("天候"),\n                None if pd.isna(row.get("走路温度")) else float(row.get("走路温度")),\n                None if pd.isna(row.get("気温")) else float(row.get("気温")),\n                None if pd.isna(row.get("湿度")) else float(row.get("湿度")),\n                row.get("レース種別"),\n                None if pd.isna(row.get("距離")) else int(row.get("距離")),\n                None if pd.isna(row.get("周回数")) else int(row.get("周回数")),\n                None if pd.isna(row.get("人気")) else int(row.get("人気")),\n                None if pd.isna(row.get("車番")) else int(row.get("車番")),\n            ))\n            inserted += 1\n\n        con.commit()\n\n    return inserted, skipped\n\n\nprint("✅ Ver15.1 縦型選手履歴パーサーを読み込みました。")\n\n# ============================================================\n# Ver15.2 縦型出走表コピペ対応\n# 公式サイトの「車番→選手名→複数行データ」形式を解析\n# ============================================================\n\ndef v152_split_entry_blocks(text):\n    lines = [x.strip() for x in v15_clean_text(text).splitlines() if x.strip()]\n    blocks = []\n    current = None\n\n    # 車番単独行、または「1 宍戸幸」のような行を開始点にする\n    for i, line in enumerate(lines):\n        m = re.fullmatch(r"([1-8])", line)\n        m_inline = re.fullmatch(r"([1-8])\\s+(.+)", line)\n\n        if m:\n            # 次行が選手名らしい場合のみ開始\n            if i + 1 < len(lines):\n                nxt = lines[i + 1]\n                if not re.fullmatch(r"\\d+(?:\\.\\d+)?%?", nxt):\n                    if current:\n                        blocks.append(current)\n                    current = {"車番": int(m.group(1)), "lines": []}\n            elif current:\n                current["lines"].append(line)\n\n        elif m_inline and not re.match(r"^\\d+\\.\\d", line):\n            if current:\n                blocks.append(current)\n            current = {\n                "車番": int(m_inline.group(1)),\n                "lines": [m_inline.group(2).strip()]\n            }\n        elif current is not None:\n            current["lines"].append(line)\n\n    if current:\n        blocks.append(current)\n\n    # 同じ車番が紛れた場合は最初の有効ブロックのみ\n    unique = {}\n    for block in blocks:\n        no = block["車番"]\n        if no not in unique and block["lines"]:\n            unique[no] = block\n    return [unique[k] for k in sorted(unique)]\n\n\ndef v152_parse_entry_block(block):\n    car_no = int(block["車番"])\n    lines = [x.strip() for x in block.get("lines", []) if x.strip()]\n    if not lines:\n        return None\n\n    # 先頭行を選手名として採用\n    player_name = v15_normalize_name(lines[0])\n\n    joined = "\\n".join(lines)\n    compact = " ".join(lines)\n\n    handicap = None\n    st = np.nan\n    hm = re.search(r"([+-]?\\d+|-)\\s*m\\s*/\\s*ST\\s*([+-]?\\d?\\.\\d{2,3})", compact, re.I)\n    if hm:\n        handicap = 0 if hm.group(1) == "-" else int(hm.group(1))\n        st = float(hm.group(2))\n    else:\n        hm2 = re.search(r"([+-]?\\d+|-)\\s*m", compact)\n        if hm2:\n            handicap = 0 if hm2.group(1) == "-" else int(hm2.group(1))\n        stm = re.search(r"ST\\s*([+-]?\\d?\\.\\d{2,3})", compact, re.I)\n        if stm:\n            st = float(stm.group(1))\n\n    # 当日試走\n    trial = np.nan\n    tm = re.search(r"試\\s*([3-9]\\.\\d{2,3}|-)", compact)\n    if tm and tm.group(1) != "-":\n        trial = float(tm.group(1))\n\n    # 試走偏差\n    trial_dev = np.nan\n    dm = re.search(r"(?:試\\s*[3-9]\\.\\d{2,3}|試-)\\s+(-|[0-9]\\.\\d{3})", compact)\n    if dm and dm.group(1) != "-":\n        trial_dev = float(dm.group(1))\n\n    # ランク\n    rank = v15_first_match([r"\\b([SAB]-?\\d+)\\b"], compact)\n    prev_rank = v15_first_match([r"\\(前\\s*([SAB]-?\\d+)\\)"], compact)\n\n    # 審査ポイント\n    review_point = np.nan\n    rpm = re.search(r"\\(前[SAB]-?\\d+\\)\\s*([0-9]{2,3}\\.\\d{3})", compact)\n    if rpm:\n        review_point = float(rpm.group(1))\n    else:\n        # ランク直後の3桁小数\n        rpm = re.search(r"\\b[SAB]-?\\d+\\b\\s*([0-9]{2,3}\\.\\d{3})", compact)\n        if rpm:\n            review_point = float(rpm.group(1))\n\n    current_year_v = v15_int(v15_first_match([r"\\bV(\\d+)\\b"], compact))\n    final_count = v15_int(v15_first_match([r"(\\d+)回"], compact))\n\n    all_v = None\n    all_vm = re.findall(r"\\bV(\\d+)\\b", compact)\n    if all_vm:\n        try:\n            all_v = int(all_vm[-1])\n        except:\n            pass\n\n    avg_trial = v15_float(v15_first_match([r"平均試走T\\s*([3-9]\\.\\d{2,3})"], compact))\n    avg_race = v15_float(v15_first_match([r"平均競走T\\s*([3-9]\\.\\d{3})"], compact))\n    best_race = v15_float(v15_first_match([r"最高競走T\\s*([3-9]\\.\\d{3})"], compact))\n\n    recent_finish = v15_first_match([r"着順\\s*(\\d+-\\d+-\\d+-\\d+)"], compact)\n    two_rate = v15_float(v15_first_match([r"2連\\s*([0-9.]+)%"], compact))\n    three_rate = v15_float(v15_first_match([r"3連\\s*([0-9.]+)%"], compact))\n\n    # 車名は「3連xx%」の後、単独行として現れることが多い\n    car_name = None\n    for idx, line in enumerate(lines):\n        if re.fullmatch(r"3連\\s*[0-9.]+%", line) and idx + 1 < len(lines):\n            candidate = lines[idx + 1]\n            if not re.fullmatch(r"\\d+", candidate):\n                car_name = candidate\n                break\n\n    # 近90/180日の末尾パーセント群\n    percent_values = [float(x) for x in re.findall(r"([0-9]+(?:\\.[0-9]+)?)%", compact)]\n    # 最初の2つは近10走の2連・3連なので、その後を着別成績用に使う\n    tail = percent_values[2:] if len(percent_values) >= 2 else []\n    metrics = {\n        "2連対率": tail[0] if len(tail) > 0 else np.nan,\n        "3連対率": tail[1] if len(tail) > 1 else np.nan,\n        "良2連対率": tail[2] if len(tail) > 2 else np.nan,\n        "良3連対率": tail[3] if len(tail) > 3 else np.nan,\n        "湿2連対率": tail[4] if len(tail) > 4 else np.nan,\n        "湿3連対率": tail[5] if len(tail) > 5 else np.nan,\n    }\n\n    row = {\n        "車番": car_no,\n        "選手名": player_name,\n        "ハンデ": handicap,\n        "試走T": trial,\n        "ST": st,\n        "年齢": None,\n        "級別": rank.split("-")[0] if rank else None,\n        "期別": None,\n        "現ランク": rank,\n        "前ランク": prev_rank,\n        "審査P": review_point,\n        "試走偏差": trial_dev,\n        "今年V": current_year_v,\n        "優出回数": final_count,\n        "通算V": all_v,\n        "平均試走T": avg_trial,\n        "平均競走T": avg_race,\n        "最高競走T": best_race,\n        "近10走着順": recent_finish,\n        "近10走2連": two_rate,\n        "近10走3連": three_rate,\n        "車名": car_name,\n        **metrics,\n        "_raw": joined,\n    }\n    return row\n\n\ndef v152_parse_vertical_entries(text):\n    blocks = v152_split_entry_blocks(text)\n    rows = []\n    for block in blocks:\n        row = v152_parse_entry_block(block)\n        if row:\n            rows.append(row)\n    return pd.DataFrame(rows)\n\n\n# 既存関数を上書きして縦型優先・従来形式フォールバック\ndef v15_parse_entries(text):\n    vertical = v152_parse_vertical_entries(text)\n\n    if not vertical.empty and vertical["車番"].nunique() >= 2:\n        return vertical.sort_values("車番").reset_index(drop=True)\n\n    rows = []\n    seen = set()\n    for raw in v15_clean_text(text).splitlines():\n        row = v15_parse_entry_line(raw)\n        if not row:\n            continue\n        key = row["車番"]\n        if key in seen:\n            continue\n        seen.add(key)\n        rows.append(row)\n\n    df = pd.DataFrame(rows)\n    if not df.empty:\n        df = df.sort_values("車番").reset_index(drop=True)\n    return df\n\n\n# レース情報パーサーも今回の形式へ強化\n_old_v15_parse_race_meta = v15_parse_race_meta\n\ndef v15_parse_race_meta(text):\n    meta = _old_v15_parse_race_meta(text)\n    compact = " ".join([x.strip() for x in v15_clean_text(text).splitlines() if x.strip()])\n\n    # 6R\n    m = re.search(r"(?:^|\\s)(\\d{1,2})R(?:\\s|$)", compact, re.I)\n    if m:\n        meta["レース"] = int(m.group(1))\n\n    # 13:13発走\n    m = re.search(r"(\\d{1,2}:\\d{2})\\s*発走", compact)\n    if m:\n        meta["発走時刻"] = m.group(1)\n\n    # 良走路 /60℃\n    m = re.search(r"(良走路|湿走路|斑走路|良|湿|斑)\\s*/\\s*(-?\\d+(?:\\.\\d+)?)℃", compact)\n    if m:\n        meta["走路状態"] = m.group(1)\n        meta["走路温度"] = float(m.group(2))\n\n    # 3100m 8車 6周\n    m = re.search(r"(\\d{4})m\\s+(\\d+)車\\s+(\\d+)周", compact)\n    if m:\n        meta["距離"] = int(m.group(1))\n        meta["出走数"] = int(m.group(2))\n        meta["周回数"] = int(m.group(3))\n\n    # 初日・最終日等\n    m = re.search(r"(初日|\\d+日目|最終日)", compact)\n    if m:\n        meta["開催日程"] = m.group(1)\n\n    # 締切\n    m = re.search(r"(\\d{1,2}:\\d{2})\\s*締切", compact)\n    if m:\n        meta["締切時刻"] = m.group(1)\n\n    return meta\n\n\nprint("✅ Ver15.2 縦型出走表パーサーを読み込みました。")'

# 旧画面を表示せず、予測エンジンだけ読み込む
with contextlib.redirect_stdout(_quiet_io.StringIO()):
    exec(compile(_ENGINE_SOURCE, "<AutoRaceAI_Ver15_2_engine>", "exec"), globals())

# DB保存先を確実に統一
DB_PATH = DB_DIR / "autorace_players.sqlite3"

# 履歴0件でも、元エンジンが必要とする列を必ず持たせる
_original_read_history = read_history
def read_history(ws):
    df = _original_read_history(ws)
    required = [
        "開催日","開催場","レース","着順","出走","走路","ハンデ",
        "ハンデ数値","試走T","競走T","ST","タイム差","有効"
    ]
    for col in required:
        if col not in df.columns:
            df[col] = pd.Series(dtype="bool" if col == "有効" else "float64")
    return df[required]


# Ver16.4: 履歴0件・同日履歴除外後0件でも本体が必要とする列を保持
_original_prepare_history = prepare_history

def prepare_history(df, current, race, settings):
    x = _original_prepare_history(df, current, race, settings)

    required_columns = {
        "開催日": "float64",
        "開催場": "object",
        "レース": "object",
        "着順": "float64",
        "出走": "float64",
        "走路": "object",
        "ハンデ": "object",
        "ハンデ数値": "float64",
        "試走T": "float64",
        "競走T": "float64",
        "ST": "float64",
        "タイム差": "float64",
        "有効": "bool",
        "経過日数": "float64",
        "最近重み": "float64",
        "場重み": "float64",
        "走路適合重み": "float64",
        "走路重み": "float64",
        "ハンデ重み": "float64",
        "総合重み": "float64",
    }

    for column, dtype in required_columns.items():
        if column not in x.columns:
            x[column] = pd.Series(index=x.index, dtype=dtype)

    return x



# Ver16.6: 同一レースは追加せず、既存行へ追加情報を統合
def _has_value(v):
    if v is None:
        return False
    if isinstance(v, float) and np.isnan(v):
        return False
    return str(v).strip() not in ("", "—", "-", "－", "nan", "None")

def _normalized_race_no(v):
    s = _s(v).upper().replace("Ｒ", "R").strip()
    if not s:
        return ""
    m = re.search(r"(\d+)", s)
    return f"{int(m.group(1))}R" if m else s

def _find_existing_history(con, player_id, row):
    race_date = _date_text(row.get("開催日"))
    venue = _s(row.get("開催場"))
    race_no = _normalized_race_no(row.get("レース"))

    if not race_date or not venue:
        return None

    # Rあり: 完全一致を優先し、なければ同日同場のR空欄を統合対象にする
    if race_no:
        exact = con.execute("""
            SELECT history_id FROM race_history
            WHERE player_id=? AND race_date=? AND venue=?
              AND UPPER(REPLACE(COALESCE(race_no,''),'Ｒ','R'))=?
            ORDER BY history_id LIMIT 1
        """, (player_id, race_date, venue, race_no)).fetchone()
        if exact:
            return exact[0]

        blank = con.execute("""
            SELECT history_id FROM race_history
            WHERE player_id=? AND race_date=? AND venue=?
              AND TRIM(COALESCE(race_no,''))=''
            ORDER BY history_id
        """, (player_id, race_date, venue)).fetchall()
        return blank[0][0] if len(blank) == 1 else None

    # Rなし: 同日同場の候補が1件だけなら統合。複数Rなら誤結合防止で新規扱い。
    candidates = con.execute("""
        SELECT history_id FROM race_history
        WHERE player_id=? AND race_date=? AND venue=?
        ORDER BY history_id
    """, (player_id, race_date, venue)).fetchall()
    return candidates[0][0] if len(candidates) == 1 else None

def add_history_rows(player, rows, source="履歴登録"):
    added = updated = 0
    canonical = _canonical_player_name(player)

    with sqlite3.connect(str(DB_PATH)) as con:
        pid = _ensure_player(con, canonical)

        for row in rows:
            normalized = {
                "開催日": _date_text(row.get("開催日")),
                "開催場": _s(row.get("開催場")),
                "レース": _normalized_race_no(row.get("レース")),
                "着順": _num(row.get("着順")),
                "出走": _num(row.get("出走")),
                "走路": _s(row.get("走路")),
                "ハンデ": _s(row.get("ハンデ")),
                "試走T": _num(row.get("試走T")),
                "競走T": _num(row.get("競走T")),
                "ST": _num(row.get("ST")),
                "結果区分": _s(row.get("結果区分", "通常")) or "通常",
            }

            if not normalized["開催日"] or not normalized["開催場"]:
                continue

            status = normalized["結果区分"]
            use = 0 if status in ["反妨","落車","故障","他落","反則","不成立"] else 1
            existing_id = _find_existing_history(con, pid, normalized)

            if existing_id is None:
                key_row = normalized.copy()
                key_row["レース"] = normalized["レース"]
                record_key = _record_key(canonical, key_row)
                # 同じ内容由来のキー衝突を避けるため、必要時だけ連番化
                suffix = 0
                base_key = record_key
                while con.execute(
                    "SELECT 1 FROM race_history WHERE record_key=?", (record_key,)
                ).fetchone():
                    suffix += 1
                    record_key = hashlib.sha256(
                        f"{base_key}|{suffix}".encode()
                    ).hexdigest()

                con.execute("""
                    INSERT INTO race_history(
                        player_id,race_date,venue,race_no,finish,starters,surface,
                        handicap,trial_time,race_time,start_time,result_status,
                        use_for_model,source,record_key
                    ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                """, (
                    pid, normalized["開催日"], normalized["開催場"],
                    normalized["レース"], normalized["着順"], normalized["出走"],
                    normalized["走路"], normalized["ハンデ"],
                    normalized["試走T"], normalized["競走T"], normalized["ST"],
                    status, use, source, record_key
                ))
                added += 1
                continue

            current = con.execute("""
                SELECT race_no,finish,starters,surface,handicap,
                       trial_time,race_time,start_time,result_status,source
                FROM race_history WHERE history_id=?
            """, (existing_id,)).fetchone()

            columns = [
                "race_no","finish","starters","surface","handicap",
                "trial_time","race_time","start_time","result_status"
            ]
            incoming = [
                normalized["レース"], normalized["着順"], normalized["出走"],
                normalized["走路"], normalized["ハンデ"],
                normalized["試走T"], normalized["競走T"], normalized["ST"], status
            ]

            updates = {}
            for col, old_value, new_value in zip(columns, current[:9], incoming):
                # 既存空欄を補完。結果系は新しい有効値があれば更新。
                if _has_value(new_value) and (
                    not _has_value(old_value)
                    or col in {"finish","trial_time","race_time","start_time","result_status"}
                ):
                    updates[col] = new_value

            updates["use_for_model"] = use
            updates["source"] = source

            if updates:
                set_sql = ", ".join(f"{k}=?" for k in updates)
                con.execute(
                    f"UPDATE race_history SET {set_sql} WHERE history_id=?",
                    (*updates.values(), existing_id)
                )
                updated += 1

    return added, updated


# Ver16.5: Web登録テーブルと予測用テーブルを統合
def _canonical_player_name(name):
    return re.sub(r"[\s　]+", "", str(name or "")).strip()

_original_ensure_player = _ensure_player
def _ensure_player(con, name):
    canonical = _canonical_player_name(name)
    if not canonical:
        raise ValueError("選手名が空です")
    con.execute(
        "INSERT OR IGNORE INTO players(player_name) VALUES (?)",
        (canonical,)
    )
    con.execute(
        "UPDATE players SET updated_at=CURRENT_TIMESTAMP WHERE player_name=?",
        (canonical,)
    )
    return con.execute(
        "SELECT player_id FROM players WHERE player_name=?",
        (canonical,)
    ).fetchone()[0]

def migrate_v15_web_history_to_main():
    """旧Web登録履歴を予測用race_historyへ自動移行する。"""
    migrated = skipped = 0
    try:
        with sqlite3.connect(str(DB_PATH)) as con:
            exists = con.execute("""
                SELECT 1 FROM sqlite_master
                WHERE type='table' AND name='v15_player_history_imports'
            """).fetchone()
            if not exists:
                return 0, 0

            old_rows = pd.read_sql_query("""
                SELECT
                    player_name, race_date, venue, race_no, rank, starters,
                    surface, handicap, trial_time, race_time, st
                FROM v15_player_history_imports
            """, con)

        if old_rows.empty:
            return 0, 0

        for player, group in old_rows.groupby("player_name", dropna=False):
            canonical = _canonical_player_name(player)
            if not canonical:
                continue
            rows = []
            for _, r in group.iterrows():
                rows.append({
                    "開催日": _date_text(r.get("race_date")),
                    "開催場": _s(r.get("venue")),
                    "レース": _s(r.get("race_no")),
                    "着順": _num(r.get("rank")),
                    "出走": _num(r.get("starters")),
                    "走路": _s(r.get("surface")),
                    "ハンデ": _s(r.get("handicap")),
                    "試走T": _num(r.get("trial_time")),
                    "競走T": _num(r.get("race_time")),
                    "ST": _num(r.get("st")),
                    "結果区分": "通常",
                })
            a, s = add_history_rows(canonical, rows, "旧Web登録自動移行")
            migrated += a
            skipped += s
    except Exception as e:
        print("旧Web履歴の移行警告:", e)
    return migrated, skipped

_migrated, _migration_skipped = migrate_v15_web_history_to_main()
if _migrated:
    print(f"旧Web登録履歴を予測DBへ移行: {_migrated}件")

print("準備完了：次のセルを実行してください")


In [ ]:
# ② アプリ起動：このセルを実行
import io
import re
import sqlite3
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from openpyxl import Workbook

def safe_float(v, default=None):
    try:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return default
        return float(v)
    except Exception:
        return default

def normalize_name(v):
    return re.sub(r"[\s　]+", "", str(v or ""))

def get_db_history(name):
    target = normalize_name(name)
    try:
        with sqlite3.connect(str(DB_PATH)) as con:
            df = pd.read_sql_query("""
                SELECT
                    h.race_date AS 開催日,
                    h.venue AS 開催場,
                    h.race_no AS レース,
                    h.finish AS 着順,
                    h.starters AS 出走,
                    h.surface AS 走路,
                    h.handicap AS ハンデ,
                    h.trial_time AS 試走T,
                    h.race_time AS 競走T,
                    h.start_time AS ST
                FROM race_history h
                JOIN players p ON p.player_id=h.player_id
                WHERE REPLACE(REPLACE(p.player_name,' ',''),'　','')=?
                  AND COALESCE(h.use_for_model,1)=1
                ORDER BY h.race_date DESC, h.history_id DESC
            """, con, params=(target,))
        return df
    except Exception:
        return pd.DataFrame(columns=[
            "開催日","開催場","レース","着順","出走",
            "走路","ハンデ","試走T","競走T","ST"
        ])

def surface_text(v):
    s = str(v or "")
    if "湿" in s: return "湿"
    if "斑" in s: return "斑"
    if "良" in s: return "良"
    return "良"

def make_settings(ws):
    defaults = {
        4:30,5:1.35,6:0.85,7:1.5,8:0.7,9:1.4,10:1.1,11:0.75,
        12:0.04,22:0.018,29:24,31:8,32:8,33:5,34:5,35:6,
        36:8,37:6,38:4,39:2,40:1,43:6,44:20,45:0.6,
        49:6,50:6,51:4,52:6
    }
    for row, value in defaults.items():
        ws.cell(row, 2, value)

def build_input_workbook(raw_text):
    meta = v15_parse_race_meta(raw_text)
    entries = v15_parse_entries(raw_text)

    if entries.empty:
        raise ValueError("出走表を解析できませんでした。公式出走表を全文貼り付けてください。")

    wb = Workbook()
    wb.remove(wb.active)

    race_ws = wb.create_sheet("レース予測")
    race_ws.append(["レース開催日", meta.get("開催日") or datetime.now().strftime("%Y-%m-%d")])
    race_ws.append(["今回の開催場", meta.get("開催場") or ""])
    race_ws.append(["今回の走路", surface_text(meta.get("走路状態") or meta.get("走路状況"))])

    settings_ws = wb.create_sheet("設定")
    make_settings(settings_ws)

    headers = ["開催日","開催場","レース","着順","出走","走路","ハンデ","試走T","競走T","ST"]
    by_car = {int(r["車番"]): r for _, r in entries.iterrows()}
    db_status = []

    for car in range(1, 9):
        ws = wb.create_sheet(f"選手{car}")
        row = by_car.get(car)

        if row is None:
            ws["B2"] = f"欠車{car}"
            ws["E2"] = 9.99
            ws["G2"] = 0
            ws["E3"] = surface_text(meta.get("走路状態"))
            ws["I2"] = 0
            ws["I3"] = "B-999"
            history = pd.DataFrame(columns=headers)
        else:
            name = str(row.get("選手名", "")).strip()
            history = get_db_history(name)

            ws["B2"] = name
            ws["E2"] = safe_float(row.get("試走T"), safe_float(row.get("平均試走T"), 3.50))
            ws["G2"] = safe_float(row.get("ハンデ"), 0)
            ws["E3"] = surface_text(meta.get("走路状態") or meta.get("走路状況"))
            ws["I2"] = safe_float(row.get("審査P"), 0)
            ws["I3"] = str(row.get("現ランク") or "B-999")

            db_status.append({
                "車番": car,
                "選手名": name,
                "DB履歴件数": len(history),
                "状態": "取得済み" if len(history) else "履歴なし"
            })

        for c, header in enumerate(headers, 1):
            ws.cell(6, c, header)

        for r_index, (_, hrow) in enumerate(history.head(100).iterrows(), 7):
            for c, header in enumerate(headers, 1):
                value = hrow.get(header)
                if header == "開催日" and value:
                    dt = pd.to_datetime(value, errors="coerce")
                    value = None if pd.isna(dt) else dt.to_pydatetime()
                elif not isinstance(value, str) and pd.isna(value):
                    value = None
                ws.cell(r_index, c, value)

    stream = io.BytesIO()
    wb.save(stream)
    return stream.getvalue(), entries, pd.DataFrame(db_status), meta



# --------------------
# Ver16.9 公式ページ環境条件・結果全文解析
# --------------------
def _extract_number(text, patterns):
    for pattern in patterns:
        m = re.search(pattern, text, re.I)
        if m:
            try:
                return float(m.group(1))
            except Exception:
                pass
    return None

def parse_environment_meta(raw_text):
    """予測ページ・結果ページ共通の環境条件を抽出。"""
    text = str(raw_text or "").replace("\u3000", " ")
    meta = {}

    # 開催期間から年を補完
    year_match = re.search(r"開催期間[:：]?\s*(20\d{2})年", text)
    year = int(year_match.group(1)) if year_match else datetime.now().year

    # 日付
    date_match = re.search(
        r"(20\d{2})[年/\-](\d{1,2})[月/\-](\d{1,2})日?", text
    )
    if date_match:
        meta["開催日"] = (
            f"{int(date_match.group(1)):04d}-"
            f"{int(date_match.group(2)):02d}-"
            f"{int(date_match.group(3)):02d}"
        )
    else:
        short_date = re.search(r"(?<!\d)(\d{1,2})月(\d{1,2})日", text)
        if short_date:
            meta["開催日"] = (
                f"{year:04d}-{int(short_date.group(1)):02d}-"
                f"{int(short_date.group(2)):02d}"
            )

    # レース番号
    race_patterns = [
        r"(?:予選|一般戦|準決勝戦?|優勝戦?|選抜戦?|特別一般戦?)?\s*(\d{1,2})\s*[RＲ]",
        r"レース\s*[:：]?\s*(\d{1,2})\s*[RＲ]?"
    ]
    for pattern in race_patterns:
        m = re.search(pattern, text, re.I)
        if m:
            meta["レース"] = f"{int(m.group(1))}R"
            break

    # 発走時刻
    m = re.search(r"発走予定\s*(\d{1,2}:\d{2})", text)
    if m:
        meta["発走時刻"] = m.group(1)

    # 環境
    meta["気温"] = _extract_number(
        text, [r"気温\s*[:：]?\s*([\-+]?\d+(?:\.\d+)?)\s*℃"]
    )
    meta["湿度"] = _extract_number(
        text, [r"湿度\s*[:：]?\s*(\d+(?:\.\d+)?)\s*%"]
    )
    meta["走路温度"] = _extract_number(
        text, [r"走路温度\s*[:：]?\s*([\-+]?\d+(?:\.\d+)?)\s*℃"]
    )
    meta["風速"] = _extract_number(
        text, [r"風速\s*[:：]?\s*(\d+(?:\.\d+)?)\s*m"]
    )

    weather = re.search(
        r"天候\s*(?:\r?\n|\t|[:：]\s*)?([晴曇雨雪霧]+)", text
    )
    if weather:
        meta["天候"] = weather.group(1)

    surface = re.search(
        r"走路状況\s*(?:\r?\n|\t|[:：]\s*)?"
        r"(良走路|湿走路|斑走路|良|湿|斑)", text
    )
    if surface:
        value = surface.group(1)
        meta["走路状況"] = {
            "良走路": "良", "湿走路": "湿", "斑走路": "斑"
        }.get(value, value)

    wind = re.search(
        r"風向\s*[:：]?\s*([^\s\t\r\n]+)", text
    )
    if wind:
        meta["風向"] = wind.group(1)

    return meta

def parse_full_result_text_v169(raw_text, venue=""):
    """
    ユーザー提示形式の公式結果全文を解析。
    既存パーサーを利用しつつ、環境条件と表形式結果を補強する。
    """
    text = str(raw_text or "")
    env = parse_environment_meta(text)

    # 既存パーサーを第一候補
    try:
        meta, results, lap_rows = parse_full_official_result(text)
    except Exception:
        meta, results, lap_rows = {}, [], []

    meta = dict(meta or {})
    for k, v in env.items():
        if v not in (None, ""):
            # 既存キーの英語名にも対応
            meta[k] = v

    if venue:
        meta["venue"] = venue
        meta["開催場"] = venue

    # 着順表が既存パーサーで取れない場合の補強
    if not results:
        lines = [x.strip() for x in text.splitlines()]
        start = next(
            (i for i, x in enumerate(lines) if x == "レース結果"),
            None
        )
        if start is not None:
            # 「着 事故 車 選手名」以降を探す
            head = next(
                (
                    i for i in range(start, min(len(lines), start + 30))
                    if lines[i].startswith("着") and "事故" in lines[i]
                ),
                None
            )
            if head is not None:
                i = head + 1
                parsed = []
                while i < len(lines):
                    if lines[i] in ("払戻金", "グランドノート"):
                        break

                    if re.fullmatch(r"\d+", lines[i]):
                        finish = int(lines[i])
                        block = lines[i:i+12]

                        # 車番
                        car = None
                        car_pos = None
                        for j, val in enumerate(block[1:], 1):
                            if re.fullmatch(r"[1-8]", val):
                                car = int(val)
                                car_pos = j
                                break

                        if car is not None and car_pos is not None:
                            # 車番直後の非数値を選手名として扱う
                            name = ""
                            machine = ""
                            for j in range(car_pos + 1, len(block)):
                                val = block[j]
                                if not val:
                                    continue
                                if re.fullmatch(r"\d+(?:\.\d+)?", val):
                                    break
                                if not name:
                                    name = val
                                elif not machine:
                                    machine = val
                                    break

                            nums = []
                            for val in block[car_pos + 1:]:
                                if re.fullmatch(r"\d+(?:\.\d+)?", val):
                                    nums.append(float(val))

                            if len(nums) >= 4:
                                handicap, trial, race_time, st = nums[-4:]
                                parsed.append({
                                    "着順": finish,
                                    "車番": car,
                                    "選手名": name,
                                    "競走車名": machine,
                                    "ハンデ": handicap,
                                    "試走T": trial,
                                    "競走T": race_time,
                                    "ST": st,
                                    "結果区分": "通常",
                                })
                    i += 1

                results = parsed

    # 周回順位の補強
    if not lap_rows and "グランドノート" in text:
        lap_rows = []
        for line in text.splitlines():
            cols = [c.strip() for c in re.split(r"\t+", line.strip())]
            if not cols:
                continue
            label = cols[0]
            if (
                label == "ゴール線通過"
                or re.fullmatch(r"\d+周回", label)
            ):
                order = []
                for c in cols[1:]:
                    if re.fullmatch(r"[1-8]", c):
                        order.append(int(c))
                if order:
                    lap_rows.append({
                        "周回": label,
                        "順位": order
                    })

    return meta, results, lap_rows

def environment_adjustments(meta):
    """環境条件から展開シミュレーション用の軽微な補正値を作る。"""
    surface = _s(meta.get("走路状況") or meta.get("走路"))
    track_temp = safe_float(meta.get("走路温度"), 30.0)
    humidity = safe_float(meta.get("湿度"), 60.0)
    wind_speed = safe_float(meta.get("風速"), 0.0)

    return {
        "track_temp": track_temp,
        "surface": surface,
        "humidity": humidity,
        "wind_speed": wind_speed,
        # 表示用。既存エンジンを壊さない範囲で利用
        "start_variance_factor": (
            1.10 if surface == "湿" else 1.04 if surface == "斑" else 1.0
        ),
        "overtake_factor": (
            0.92 if surface == "湿" else 0.97 if surface == "斑" else 1.0
        ),
    }


# --------------------
# Ver16.8 全結果学習を予測へ反映
# --------------------
def _current_learning_weights():
    init_learning_tables()
    with sqlite3.connect(str(DB_PATH)) as con:
        rows = con.execute(
            "SELECT setting_name,current_value FROM learning_settings"
        ).fetchall()
    values = dict(rows)
    return {k: float(values.get(k, v)) for k, v in INITIAL_WEIGHTS.items()}

def apply_all_result_learning(df):
    """
    過去に「予測保存＋結果登録」が揃った全レースから学習した重みを、
    今回の総合点へ補正として反映する。
    """
    result = df.copy()
    weights = _current_learning_weights()

    raw = []
    for _, row in result.iterrows():
        raw.append([
            _layer_value(row, LEARN_FEATURES[layer])
            for layer in INITIAL_WEIGHTS
        ])
    raw = np.asarray(raw, dtype=float)
    mu = np.nanmean(raw, axis=0)
    sd = np.nanstd(raw, axis=0)
    sd = np.where(sd < 1e-8, 1.0, sd)
    x = np.nan_to_num((raw - mu) / sd)

    w = np.asarray([weights[k] for k in INITIAL_WEIGHTS], dtype=float)
    learned = x @ w
    learned_sd = float(np.std(learned))
    learned_z = (learned - float(np.mean(learned))) / (
        learned_sd if learned_sd > 1e-8 else 1.0
    )

    result["全結果学習点"] = learned
    result["全結果学習補正"] = learned_z * 3.0
    result["学習前総合点"] = pd.to_numeric(
        result["改善後総合点"], errors="coerce"
    ).fillna(0.0)
    result["改善後総合点"] = (
        result["学習前総合点"] + result["全結果学習補正"]
    )
    result["改善後順位"] = (
        result["改善後総合点"]
        .rank(method="first", ascending=False)
        .astype(int)
    )
    return result, weights

def save_prediction_for_learning(df, meta, track_temp):
    global LATEST_PREDICTION_DF, LATEST_TRACK_TEMP
    LATEST_PREDICTION_DF = df.copy()
    LATEST_TRACK_TEMP = float(track_temp)

    # 環境条件を予測DFへ付加して学習保存
    for col in [
        "天候", "走路状況", "気温", "湿度",
        "走路温度", "風向", "風速", "発走時刻"
    ]:
        LATEST_PREDICTION_DF[col] = meta.get(col)

    date_text = _date_text(meta.get("開催日"))
    venue = _s(meta.get("開催場"))
    race_no = _normalized_race_no(meta.get("レース"))

    if not date_text or not venue or not race_no:
        return None, 0

    return save_latest_prediction(date_text, venue, race_no)


# --------------------
# 予測
# --------------------
prediction_text = widgets.Textarea(
    placeholder="公式出走表を全文コピーして貼り付け",
    layout=widgets.Layout(width="100%", height="400px")
)
prediction_button = widgets.Button(
    description="予測する",
    button_style="success",
    icon="calculator",
    layout=widgets.Layout(width="220px", height="46px")
)
prediction_output = widgets.Output()

def run_prediction(_):
    with prediction_output:
        clear_output()
        try:
            print("SQLite履歴を読み込み、展開シミュレーションを実行しています...")
            content, entries, db_status, meta = build_input_workbook(
                prediction_text.value
            )

            print("\nDB取得状況")
            display(db_status)

            # 貼り付け本文から環境条件を再確認
            parsed_env = parse_environment_meta(prediction_text.value)
            for k, v in parsed_env.items():
                if v not in (None, ""):
                    meta[k] = v

            env_adj = environment_adjustments(meta)
            track_temp = env_adj["track_temp"]
            trials = 10000

            print("\n読み取ったレース情報・環境条件")
            preview_meta = {
                "開催日": meta.get("開催日"),
                "開催場": meta.get("開催場"),
                "レース": meta.get("レース"),
                "発走時刻": meta.get("発走時刻"),
                "天候": meta.get("天候"),
                "走路状況": meta.get("走路状況"),
                "気温": meta.get("気温"),
                "湿度": meta.get("湿度"),
                "走路温度": meta.get("走路温度"),
                "風向": meta.get("風向"),
                "風速": meta.get("風速"),
            }
            display(
                pd.DataFrame(
                    [{"項目": k, "取得値": v} for k, v in preview_meta.items()]
                )
            )

            missing_core = [
                k for k in ("開催日", "開催場", "レース")
                if not meta.get(k)
            ]
            if missing_core:
                print(
                    "注意: 次の項目を読み取れませんでした: "
                    + "、".join(missing_core)
                )

            if meta.get("走路温度") in (None, ""):
                print("注意: 走路温度を取得できないため30℃を使用します。")

            df, bets, _ = run_model(
                content,
                "AutoRaceAI_Ver16_input.xlsx",
                trials,
                20260719,
                track_temp
            )

            # 全結果から学習済みの重みを今回予測へ反映
            df, learned_weights = apply_all_result_learning(df)

            # 補正後順位で展開シミュレーションを再計算
            finish_counts, bets = simulate(
                df, trials, 20260719, track_temp=track_temp
            )
            result_file = create_result_excel(
                content,
                "AutoRaceAI_Ver16_input.xlsx",
                df,
                finish_counts,
                bets,
                trials,
                track_temp=track_temp
            )

            # 後日の結果登録と照合できるよう今回予測を自動保存
            saved_key, saved_players = save_prediction_for_learning(
                df, meta, track_temp
            )

            result_cols = [
                c for c in [
                    "改善後順位","車","選手名","ハンデ",
                    "基礎スピード点","実戦能力点","勝負強さ点",
                    "展開適性点","学習前総合点",
                    "全結果学習補正","改善後総合点"
                ] if c in df.columns
            ]
            print("\n予測結果")
            display(
                df[result_cols]
                .sort_values("改善後順位")
                .reset_index(drop=True)
            )

            print("\n現在の全結果学習重み")
            display(pd.DataFrame({
                "項目": list(learned_weights.keys()),
                "重み": list(learned_weights.values())
            }))

            if saved_key:
                print(
                    f"\n学習用予測を保存しました: {saved_key} "
                    f"({saved_players}選手)"
                )
            else:
                print(
                    "\n注意: 開催日・開催場・Rのいずれかを取得できず、"
                    "学習用予測は保存されませんでした。"
                )

            print("\n三連単確率 上位10")
            for i, (combo, count) in enumerate(
                sorted(
                    bets["三連単"].items(),
                    key=lambda x: x[1],
                    reverse=True
                )[:10],
                1
            ):
                print(
                    f"{i}. {'-'.join(map(str, combo))}  "
                    f"{count/trials:.2%}"
                )

            files.download(result_file)

        except Exception as e:
            print("エラー:", e)
            traceback.print_exc(limit=3)

prediction_button.on_click(run_prediction)

prediction_tab = widgets.VBox([
    widgets.HTML("<b>公式出走表を貼り付けて予測します。</b>"),
    prediction_text,
    prediction_button,
    prediction_output
])


def parse_tabular_player_history(raw_text, player_name):
    """日付 場 R 着 車番 走路 ハンデ 試走T 競走T ST 形式を解析。"""
    name = normalize_name(player_name)
    if not name:
        raise ValueError("表形式では選手名欄の入力が必要です。")

    text = raw_text.replace("\u3000", " ").strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) < 2:
        return pd.DataFrame()

    # タブ優先。タブがない場合は2個以上の空白で分割。
    def split_line(line):
        if "\t" in line:
            return [x.strip() for x in line.split("\t")]
        return [x.strip() for x in re.split(r"\s{2,}", line)]

    headers = split_line(lines[0])
    aliases = {
        "日付": "開催日", "開催日": "開催日",
        "場": "開催場", "開催場": "開催場",
        "R": "レース", "Ｒ": "レース", "レース": "レース",
        "着": "着順", "着順": "着順",
        "車番": "車番",
        "走路": "走路",
        "ハンデ": "ハンデ",
        "試走T": "試走T", "試走": "試走T",
        "競走T": "競走T", "競走": "競走T",
        "ST": "ST", "ＳＴ": "ST",
    }
    mapped = [aliases.get(h, h) for h in headers]

    required = {"開催日","開催場","着順","ハンデ","試走T","競走T","ST"}
    if not required.issubset(set(mapped)):
        return pd.DataFrame()

    records = []
    for line in lines[1:]:
        values = split_line(line)
        if len(values) < len(mapped):
            # 単一空白区切りのフォールバック
            values = re.split(r"\s+", line.strip())
        if len(values) < len(mapped):
            continue

        rec = dict(zip(mapped, values))
        records.append({
            "選手名": name,
            "開催日": _date_text(rec.get("開催日")),
            "開催場": _s(rec.get("開催場")),
            "レース": _normalized_race_no(rec.get("レース")),
            "着順": _num(rec.get("着順")),
            "出走": None,
            "走路": "" if _s(rec.get("走路")) in ("—","-","－") else _s(rec.get("走路")),
            "ハンデ": _s(rec.get("ハンデ")),
            "試走T": _num(rec.get("試走T")),
            "競走T": _num(rec.get("競走T")),
            "ST": _num(rec.get("ST")),
            "車番": _num(rec.get("車番")),
        })

    return pd.DataFrame(records)


# --------------------
# Webページのコピペ登録
# --------------------
web_player_name = widgets.Text(
    description="選手名",
    placeholder="自動判定できない場合に入力",
    layout=widgets.Layout(width="100%")
)
web_history_text = widgets.Textarea(
    placeholder="公式ページ全文、または 日付・場・R・着・車番・走路・ハンデ・試走T・競走T・ST の表を貼り付け",
    layout=widgets.Layout(width="100%", height="400px")
)
web_register_button = widgets.Button(
    description="Web履歴を登録",
    button_style="primary",
    icon="save",
    layout=widgets.Layout(width="220px", height="46px")
)
web_register_output = widgets.Output()

def register_web_history(_):
    with web_register_output:
        clear_output()
        try:
            raw = web_history_text.value.strip()
            if not raw:
                raise ValueError("選手履歴ページを貼り付けてください。")

            name = web_player_name.value.strip() or None

            # まず表形式を判定し、該当しなければ公式Webページ形式を解析
            df = parse_tabular_player_history(raw, name)
            if df is None or df.empty:
                df = v15_parse_player_history(raw, player_name=name)

            if df is None or df.empty:
                raise ValueError(
                    "履歴を解析できませんでした。公式ページ全文、または"
                    "「日付・場・R・着・車番・走路・ハンデ・試走T・競走T・ST」"
                    "の表を貼り付けてください。"
                )

            preview_cols = [
                c for c in [
                    "選手名","開催日","開催場","着順","走路",
                    "ハンデ","試走T","競走T","ST"
                ] if c in df.columns
            ]
            print(f"解析件数: {len(df)}件")
            display(df[preview_cols].head(20))

            # 予測が実際に参照するrace_historyへ直接登録
            reports = []
            for player, group in df.groupby("選手名", dropna=False):
                canonical = normalize_name(player)
                if not canonical:
                    raise ValueError("選手名を判定できません。上の選手名欄へ入力してください。")

                rows = []
                for _, r in group.iterrows():
                    rows.append({
                        "開催日": _date_text(r.get("開催日")),
                        "開催場": _s(r.get("開催場")),
                        "レース": _s(r.get("レース")),
                        "着順": _num(r.get("着順")),
                        "出走": _num(r.get("出走")),
                        "走路": _s(r.get("走路")),
                        "ハンデ": _s(r.get("ハンデ")),
                        "試走T": _num(r.get("試走T")),
                        "競走T": _num(r.get("競走T")),
                        "ST": _num(r.get("ST")),
                        "結果区分": "通常",
                    })

                added, skipped = add_history_rows(
                    canonical, rows, "Webコピペ登録"
                )
                total = len(get_db_history(canonical))
                reports.append({
                    "選手名": canonical,
                    "新規追加": added,
                    "更新・統合": skipped,
                    "DB総履歴": total,
                    "予測連携": "連携済み" if total > 0 else "履歴なし"
                })

            print("\n登録完了・予測DBへ連携済み")
            display(pd.DataFrame(reports))

        except Exception as e:
            print("登録エラー:", e)
            traceback.print_exc(limit=3)

web_register_button.on_click(register_web_history)

web_tab = widgets.VBox([
    widgets.HTML("<b>公式選手履歴ページ、またはタブ区切りの履歴表を貼り付けて登録します。</b>"),
    web_player_name,
    web_history_text,
    web_register_button,
    web_register_output
])

# --------------------
# Excel登録
# --------------------
excel_upload = widgets.FileUpload(
    accept=".xlsx,.xlsm",
    multiple=False,
    description="Excel選択"
)
excel_register_button = widgets.Button(
    description="Excelを登録",
    button_style="primary",
    icon="upload",
    layout=widgets.Layout(width="220px", height="46px")
)
excel_register_output = widgets.Output()

def uploaded_file_data(upload_widget):
    value = upload_widget.value
    if not value:
        return None, None

    if isinstance(value, dict):
        item = next(iter(value.values()))
    else:
        item = value[0]

    name = item.get("name", "uploaded.xlsx")
    content = item.get("content")
    if hasattr(content, "tobytes"):
        content = content.tobytes()
    else:
        content = bytes(content)
    return name, content

def register_excel(_):
    with excel_register_output:
        clear_output()
        try:
            filename, content = uploaded_file_data(excel_upload)
            if not content:
                raise ValueError("Excelファイルを選択してください。")

            print(f"登録中: {filename}")
            report = import_workbook_to_db(content, filename)

            print("登録完了（既存レースは追加情報を統合）")
            if isinstance(report, pd.DataFrame):
                display(report)
            elif report is not None:
                print(report)

        except Exception as e:
            print("登録エラー:", e)
            traceback.print_exc(limit=3)

excel_register_button.on_click(register_excel)

excel_tab = widgets.VBox([
    widgets.HTML(
        "<b>従来の選手1〜選手8シートを含むExcelをSQLiteへ登録します。</b>"
    ),
    excel_upload,
    excel_register_button,
    excel_register_output
])



# ============================================================
# Ver17.0 着順ブロック解析・直接保存
# ============================================================
def _nonempty_result_lines(text):
    return [
        line.replace("\u3000", " ").strip()
        for line in str(text or "").replace("\r\n", "\n").replace("\r", "\n").split("\n")
        if line.replace("\u3000", " ").strip()
    ]

def _is_plain_int(value):
    return bool(re.fullmatch(r"\d+", str(value or "").strip()))

def _is_time_number(value):
    return bool(re.fullmatch(r"\d+(?:\.\d+)?", str(value or "").strip()))

def _normalize_result_meta(meta):
    """日本語キーと既存DB用キーを両方そろえる。"""
    meta = dict(meta or {})
    mappings = {
        "開催日": "race_date",
        "開催場": "venue",
        "レース": "race_no",
        "天候": "weather",
        "走路状況": "surface_condition",
        "走路温度": "track_temp",
        "気温": "air_temp",
        "湿度": "humidity",
        "発走時刻": "start_time_text",
    }
    for jp, en in mappings.items():
        if meta.get(jp) not in (None, ""):
            meta[en] = meta[jp]
        elif meta.get(en) not in (None, ""):
            meta[jp] = meta[en]

    if meta.get("race_no"):
        meta["race_no"] = _normalized_race_no(meta["race_no"])
        meta["レース"] = meta["race_no"]
    return meta

def parse_multiline_result_blocks(text):
    """
    公式結果の複数行形式を解析する。

    例:
      1 [事故空欄] 6
      横田 翔紀
      L・ダウ
      30 3.69 3.760 0.19
    """
    lines = _nonempty_result_lines(text)

    header_index = None
    for i, line in enumerate(lines):
        compact = re.sub(r"\s+", "", line)
        if "着" in compact and "事故" in compact and "車" in compact and "選手名" in compact:
            header_index = i
            break
    if header_index is None:
        raise ValueError("着・事故・車・選手名の見出しが見つかりません。")

    # 次の見出し行を飛ばす
    i = header_index + 1
    if i < len(lines) and all(k in lines[i] for k in ["ハンデ", "試走T", "競走T", "ST"]):
        i += 1

    results = []
    stop_words = {"払戻金", "グランドノート", "周回/順位", "周回・順位"}

    while i < len(lines):
        if any(lines[i].startswith(word) for word in stop_words):
            break

        # 着順行は「1  6」またはタブで「1  空欄  6」
        first_parts = [p.strip() for p in re.split(r"\t+|\s{2,}", lines[i]) if p.strip()]
        if len(first_parts) == 1:
            first_parts = re.split(r"\s+", lines[i])

        if not first_parts or not _is_plain_int(first_parts[0]):
            i += 1
            continue

        finish = int(first_parts[0])
        accident = ""
        car = None

        # 1行目から車番を取得
        for token in first_parts[1:]:
            if _is_plain_int(token) and 1 <= int(token) <= 8:
                car = int(token)
                break
            if token and not accident:
                accident = token

        # 車番が次行に分離している場合
        j = i + 1
        if car is None and j < len(lines) and _is_plain_int(lines[j]) and 1 <= int(lines[j]) <= 8:
            car = int(lines[j])
            j += 1

        if car is None:
            i += 1
            continue

        # 選手名、競走車名
        if j >= len(lines):
            break
        name = lines[j].strip()
        j += 1

        machine = ""
        if j < len(lines) and not _is_time_number(lines[j]):
            machine = lines[j].strip()
            j += 1

        # 数値4個: ハンデ、試走T、競走T、ST
        numeric_tokens = []
        abnormal = ""
        while j < len(lines):
            if any(lines[j].startswith(word) for word in stop_words):
                break

            # 次の着順開始を検知
            next_parts = [p for p in re.split(r"\t+|\s+", lines[j]) if p]
            if (
                numeric_tokens
                and next_parts
                and _is_plain_int(next_parts[0])
                and 1 <= int(next_parts[0]) <= 8
                and len(numeric_tokens) >= 4
            ):
                break

            for token in re.split(r"\t+|\s+", lines[j]):
                token = token.strip()
                if not token:
                    continue
                if _is_time_number(token):
                    numeric_tokens.append(float(token))
                elif token not in {"—", "-", "－"}:
                    abnormal = token
            j += 1
            if len(numeric_tokens) >= 4:
                break

        if len(numeric_tokens) < 4:
            raise ValueError(
                f"{finish}着・{car}号車のハンデ/試走T/競走T/STを読み取れませんでした。"
            )

        handicap, trial_time, race_time, start_time = numeric_tokens[:4]
        results.append({
            "着順": finish,
            "事故": accident,
            "車番": car,
            "選手名": _canonical_player_name(name),
            "競走車名": machine,
            "年齢期": "",
            "LG": "",
            "ハンデ": str(int(handicap)) if float(handicap).is_integer() else str(handicap),
            "試走T": float(trial_time),
            "競走T": float(race_time),
            "ST": float(start_time),
            "異常": abnormal,
            "人気": None,
            "結果区分": _normalize_status(accident, abnormal),
        })
        i = j

    if not results:
        raise ValueError("着順結果を1件も読み取れませんでした。")

    # 車番の重複や着順欠落を検査
    cars = [r["車番"] for r in results]
    finishes = [r["着順"] for r in results]
    if len(cars) != len(set(cars)):
        raise ValueError(f"車番が重複しています: {cars}")
    if len(finishes) != len(set(finishes)):
        raise ValueError(f"着順が重複しています: {finishes}")

    return sorted(results, key=lambda r: r["着順"])

def parse_grand_note_v170(text):
    rows = []
    raw_lines = str(text or "").replace("\r\n", "\n").replace("\r", "\n").split("\n")
    for raw in raw_lines:
        cols = [c.strip() for c in raw.split("\t")]
        if len(cols) < 2:
            cols = [c.strip() for c in re.split(r"\s{2,}", raw.strip()) if c.strip()]
        if not cols:
            continue

        label = cols[0]
        if not (label == "ゴール線通過" or re.fullmatch(r"\d+周回", label)):
            continue

        cars = [int(c) for c in cols[1:] if _is_plain_int(c) and 1 <= int(c) <= 8]
        if not cars:
            continue

        is_goal = label == "ゴール線通過"
        lap_no = 10**6 if is_goal else int(re.search(r"\d+", label).group())
        rows.append({
            "label": "ゴール線通過" if is_goal else label,
            "lap_no": lap_no,
            "is_goal": is_goal,
            "cars": cars,
        })

    # 古い方からゴールへ並べる
    unique = {r["label"]: r for r in rows}
    normal = sorted(
        [r for r in unique.values() if not r["is_goal"]],
        key=lambda r: r["lap_no"]
    )
    goals = [r for r in unique.values() if r["is_goal"]]
    ordered = normal + goals
    for idx, row in enumerate(ordered, 1):
        row["lap_index"] = idx
    return ordered

def parse_full_result_text_v170(raw_text, venue=""):
    text = str(raw_text or "")
    meta = _normalize_result_meta(parse_environment_meta(text))
    if venue:
        meta["venue"] = venue.strip()
        meta["開催場"] = venue.strip()

    # 距離
    m = re.search(r"(\d{4})m", text)
    if m:
        meta["distance_m"] = int(m.group(1))

    # レース名
    m = re.search(
        r"(予選|一般戦|準決勝戦?|優勝戦?|選抜戦?|特別一般戦?)\s*(\d{1,2})[RＲ]",
        text
    )
    if m:
        meta["race_name"] = m.group(1)
        meta["race_no"] = f"{int(m.group(2))}R"
        meta["レース"] = meta["race_no"]

    results = parse_multiline_result_blocks(text)
    lap_rows = parse_grand_note_v170(text)
    return meta, results, lap_rows

def save_parsed_result_v170(meta, results, lap_rows, source="Ver17.1公式結果登録"):
    """
    解析済みデータを直接保存する。
    旧パーサーを再実行しないため、着順と選手の対応が崩れない。
    """
    init_full_result_tables()
    meta = _normalize_result_meta(meta)

    if not meta.get("race_date"):
        raise ValueError("開催日を読み取れませんでした。")
    if not meta.get("venue"):
        raise ValueError("開催場を読み取れませんでした。開催場欄へ入力してください。")
    if not meta.get("race_no"):
        raise ValueError("レース番号を読み取れませんでした。")

    mapping = {int(r["車番"]): r["選手名"] for r in results}

    # 周回順位がない場合もレース本体を保存できるよう、ゴール着順を仮の1地点にする
    rows_for_save = list(lap_rows)
    if not rows_for_save:
        goal_cars = [r["車番"] for r in sorted(results, key=lambda x: x["着順"])]
        rows_for_save = [{
            "label": "ゴール線通過",
            "lap_no": 10**6,
            "is_goal": True,
            "cars": goal_cars,
            "lap_index": 1,
        }]

    race_id, features = save_lap_data(
        rows_for_save, meta, mapping, source=source
    )

    with sqlite3.connect(str(DB_PATH)) as con:
        for r in results:
            pid = _ensure_player(con, r["選手名"])
            con.execute("""
                INSERT INTO race_entries(
                    race_id,car_no,player_id,finish,accident,age_term,lg,
                    handicap,trial_time,race_time,start_time,abnormal,
                    popularity,result_status
                ) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                ON CONFLICT(race_id,car_no) DO UPDATE SET
                    player_id=excluded.player_id,
                    finish=excluded.finish,
                    accident=excluded.accident,
                    age_term=excluded.age_term,
                    lg=excluded.lg,
                    handicap=excluded.handicap,
                    trial_time=excluded.trial_time,
                    race_time=excluded.race_time,
                    start_time=excluded.start_time,
                    abnormal=excluded.abnormal,
                    popularity=excluded.popularity,
                    result_status=excluded.result_status
            """, (
                race_id, r["車番"], pid, r["着順"], r.get("事故", ""),
                r.get("年齢期", ""), r.get("LG", ""), r.get("ハンデ", ""),
                r.get("試走T"), r.get("競走T"), r.get("ST"),
                r.get("異常", ""), r.get("人気"), r.get("結果区分", "通常")
            ))

    added_history = 0
    updated_history = 0
    starters = len(results)
    surface = meta.get("surface_condition", "")

    for r in results:
        history_row = {
            "開催日": meta.get("race_date", ""),
            "開催場": meta.get("venue", ""),
            "レース": meta.get("race_no", ""),
            "着順": r["着順"],
            "出走": starters,
            "走路": surface,
            "ハンデ": r.get("ハンデ", ""),
            "試走T": r.get("試走T"),
            "競走T": r.get("競走T"),
            "ST": r.get("ST"),
            "結果区分": r.get("結果区分", "通常"),
        }
        a, u = add_history_rows(
            r["選手名"], [history_row], source
        )
        added_history += a
        updated_history += u

    return (
        race_id, meta, results, features,
        added_history, updated_history
    )



# ============================================================
# Ver17.1 _race_key互換修正
# ============================================================
def save_parsed_result_v171(meta, results, lap_rows, source="Ver17.1公式結果登録"):
    init_full_result_tables()
    meta = _normalize_result_meta(meta)

    race_date = meta.get("race_date") or meta.get("開催日")
    venue = meta.get("venue") or meta.get("開催場")
    race_no = meta.get("race_no") or meta.get("レース")

    if not race_date:
        raise ValueError("開催日を読み取れませんでした。")
    if not venue:
        raise ValueError("開催場を読み取れませんでした。開催場欄へ入力してください。")
    if not race_no:
        raise ValueError("レース番号を読み取れませんでした。")

    race_no = _normalized_race_no(race_no)
    race_key = _race_key(race_date, venue, race_no)

    meta["race_date"] = race_date
    meta["venue"] = venue
    meta["race_no"] = race_no
    meta["開催日"] = race_date
    meta["開催場"] = venue
    meta["レース"] = race_no
    meta["race_key"] = race_key

    mapping = {int(r["車番"]): r["選手名"] for r in results}

    rows_for_save = list(lap_rows)
    if not rows_for_save:
        goal_cars = [r["車番"] for r in sorted(results, key=lambda x: x["着順"])]
        rows_for_save = [{
            "label": "ゴール線通過",
            "lap_no": 10**6,
            "is_goal": True,
            "cars": goal_cars,
            "lap_index": 1,
        }]

    race_id, features = save_lap_data(rows_for_save, meta, mapping, source=source)

    with sqlite3.connect(str(DB_PATH)) as con:
        for r in results:
            pid = _ensure_player(con, r["選手名"])
            con.execute("""
                INSERT INTO race_entries(
                    race_id,car_no,player_id,finish,accident,age_term,lg,
                    handicap,trial_time,race_time,start_time,abnormal,
                    popularity,result_status
                ) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                ON CONFLICT(race_id,car_no) DO UPDATE SET
                    player_id=excluded.player_id,
                    finish=excluded.finish,
                    accident=excluded.accident,
                    age_term=excluded.age_term,
                    lg=excluded.lg,
                    handicap=excluded.handicap,
                    trial_time=excluded.trial_time,
                    race_time=excluded.race_time,
                    start_time=excluded.start_time,
                    abnormal=excluded.abnormal,
                    popularity=excluded.popularity,
                    result_status=excluded.result_status
            """, (
                race_id, r["車番"], pid, r["着順"], r.get("事故", ""),
                r.get("年齢期", ""), r.get("LG", ""), r.get("ハンデ", ""),
                r.get("試走T"), r.get("競走T"), r.get("ST"),
                r.get("異常", ""), r.get("人気"), r.get("結果区分", "通常")
            ))

    added_history = 0
    updated_history = 0
    starters = len(results)
    surface = meta.get("surface_condition") or meta.get("走路状況") or ""

    for r in results:
        history_row = {
            "開催日": race_date,
            "開催場": venue,
            "レース": race_no,
            "着順": r["着順"],
            "出走": starters,
            "走路": surface,
            "ハンデ": r.get("ハンデ", ""),
            "試走T": r.get("試走T"),
            "競走T": r.get("競走T"),
            "ST": r.get("ST"),
            "結果区分": r.get("結果区分", "通常"),
        }
        a, u = add_history_rows(r["選手名"], [history_row], source)
        added_history += a
        updated_history += u

    return race_id, meta, results, features, added_history, updated_history


# --------------------
# 公式結果登録＋全結果再学習
# --------------------
result_text = widgets.Textarea(
    placeholder=(
        "公式結果ページのレース情報、着順表、"
        "グランドノート、周回順位をまとめて貼り付け"
    ),
    layout=widgets.Layout(width="100%", height="430px")
)
result_venue = widgets.Text(
    description="開催場",
    placeholder="本文から取れない場合のみ入力",
    layout=widgets.Layout(width="100%")
)
result_race_no = widgets.Text(
    description="レース",
    placeholder="本文から取れない場合のみ 例: 12R",
    layout=widgets.Layout(width="100%")
)
result_preview_button = widgets.Button(
    description="結果を確認",
    button_style="info",
    icon="search",
    layout=widgets.Layout(width="200px", height="44px")
)
result_register_button = widgets.Button(
    description="結果登録＋再学習",
    button_style="success",
    icon="database",
    layout=widgets.Layout(width="240px", height="44px")
)
result_output = widgets.Output()
RESULT_STATE = {"parsed": None}

def preview_result(_):
    with result_output:
        clear_output()
        try:
            raw = result_text.value.strip()
            if not raw:
                raise ValueError("公式結果ページを貼り付けてください。")

            meta, results, lap_rows = parse_full_result_text_v170(
                raw,
                venue=result_venue.value.strip()
            )
            if result_race_no.value.strip():
                meta["race_no"] = _normalized_race_no(
                    result_race_no.value
                )
                meta["レース"] = meta["race_no"]

            RESULT_STATE["parsed"] = (meta, results, lap_rows)

            print("レース情報・環境条件")
            display(
                pd.DataFrame(
                    [{"項目": k, "値": v} for k, v in meta.items()]
                )
            )

            print(f"\n着順結果: {len(results)}選手")
            display(pd.DataFrame(results)[[
                c for c in [
                    "着順","車番","選手名","ハンデ",
                    "試走T","競走T","ST","結果区分"
                ] if c in pd.DataFrame(results).columns
            ]])

            if lap_rows:
                print(f"\n周回順位データ: {len(lap_rows)}行")
            else:
                print("\n周回順位データはありません。着順結果だけ登録できます。")

            print("\n全選手の着順・車番・選手名が正しければ、「結果登録＋再学習」を押してください。")

        except Exception as e:
            RESULT_STATE["parsed"] = None
            print("解析エラー:", e)
            traceback.print_exc(limit=2)

def register_result_and_learn(_):
    with result_output:
        clear_output()
        try:
            raw = result_text.value.strip()
            if not raw:
                raise ValueError("公式結果ページを貼り付けてください。")

            # Ver16.9形式で先に解析確認
            parsed_meta, parsed_results, parsed_laps = (
                parse_full_result_text_v170(
                    raw,
                    venue=result_venue.value.strip()
                )
            )
            if result_race_no.value.strip():
                parsed_meta["レース"] = _normalized_race_no(
                    result_race_no.value
                )
                parsed_meta["race_no"] = parsed_meta["レース"]

            if not parsed_results:
                raise ValueError("着順結果を解析できませんでした。")
            if len(parsed_results) < 2:
                raise ValueError(
                    "着順結果が1人分しか取得できていません。"
                    "「結果を確認」で全選手が表示されるか確認してください。"
                )

            # 解析済みの着順結果をそのまま保存
            saved = save_parsed_result_v171(
                parsed_meta,
                parsed_results,
                parsed_laps,
                source="Ver17.1公式結果登録"
            )
            race_id, meta, results, features, added, updated = saved

            # 環境条件テーブルが存在する場合は保存
            try:
                with sqlite3.connect(str(DB_PATH)) as con:
                    con.execute("""
                        CREATE TABLE IF NOT EXISTS race_environment (
                            race_id INTEGER PRIMARY KEY,
                            race_date TEXT,
                            venue TEXT,
                            race_no TEXT,
                            weather TEXT,
                            surface TEXT,
                            air_temp REAL,
                            humidity REAL,
                            track_temp REAL,
                            wind_direction TEXT,
                            wind_speed REAL,
                            start_time TEXT,
                            source TEXT,
                            updated_at TEXT DEFAULT CURRENT_TIMESTAMP
                        )
                    """)
                    con.execute("""
                        INSERT INTO race_environment(
                            race_id,race_date,venue,race_no,weather,surface,
                            air_temp,humidity,track_temp,wind_direction,
                            wind_speed,start_time,source
                        ) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)
                        ON CONFLICT(race_id) DO UPDATE SET
                            race_date=excluded.race_date,
                            venue=excluded.venue,
                            race_no=excluded.race_no,
                            weather=excluded.weather,
                            surface=excluded.surface,
                            air_temp=excluded.air_temp,
                            humidity=excluded.humidity,
                            track_temp=excluded.track_temp,
                            wind_direction=excluded.wind_direction,
                            wind_speed=excluded.wind_speed,
                            start_time=excluded.start_time,
                            source=excluded.source,
                            updated_at=CURRENT_TIMESTAMP
                    """, (
                        race_id,
                        parsed_meta.get("開催日"),
                        parsed_meta.get("開催場") or parsed_meta.get("venue"),
                        parsed_meta.get("レース") or parsed_meta.get("race_no"),
                        parsed_meta.get("天候"),
                        parsed_meta.get("走路状況"),
                        parsed_meta.get("気温"),
                        parsed_meta.get("湿度"),
                        parsed_meta.get("走路温度"),
                        parsed_meta.get("風向"),
                        parsed_meta.get("風速"),
                        parsed_meta.get("発走時刻"),
                        "Ver16.9公式結果登録"
                    ))
            except Exception as env_error:
                print("環境条件保存の警告:", env_error)

            print("結果登録完了")
            print(f"レースID: {race_id}")
            print(f"履歴新規追加: {added}件")
            print(f"既存履歴更新・統合: {updated}件")
            display(full_result_summary(results, features))

            # 保存済みの全予測＋全結果で毎回再学習
            try:
                settings_df, old_loss, new_loss, races_used = (
                    learn_best_weights(search_rounds=3500, seed=42)
                )
                print(
                    f"\n全結果から重みを再学習しました。"
                    f"対象: {races_used}レース"
                )
                print(f"順位誤差: {old_loss:.4f} → {new_loss:.4f}")
                display(settings_df)
            except RuntimeError as learn_error:
                print("\n結果は登録済みです。")
                print(
                    "重み再学習は、予測保存と結果登録が一致した"
                    "レースが3件以上になると開始します。"
                )
                print(str(learn_error))

        except Exception as e:
            print("登録エラー:", e)
            traceback.print_exc(limit=3)

result_preview_button.on_click(preview_result)
result_register_button.on_click(register_result_and_learn)

result_tab = widgets.VBox([
    widgets.HTML(
        "<b>レース後の公式結果ページを登録します。</b><br>"
        "登録後、保存済みの過去予測と全結果を使って重みを再学習します。"
    ),
    result_venue,
    result_race_no,
    result_text,
    widgets.HBox([
        result_preview_button,
        result_register_button
    ]),
    result_output
])


# --------------------
# DB登録確認
# --------------------
db_check_name = widgets.Text(
    description="選手名",
    placeholder="空欄なら全体を表示",
    layout=widgets.Layout(width="100%")
)
db_check_button = widgets.Button(
    description="登録状況を確認",
    button_style="info",
    icon="database",
    layout=widgets.Layout(width="220px", height="46px")
)
db_check_output = widgets.Output()

def show_db_status(_):
    with db_check_output:
        clear_output()
        try:
            with sqlite3.connect(str(DB_PATH)) as con:
                total_players = con.execute(
                    "SELECT COUNT(*) FROM players"
                ).fetchone()[0]
                total_history = con.execute(
                    "SELECT COUNT(*) FROM race_history"
                ).fetchone()[0]

                query_name = normalize_name(db_check_name.value)
                if query_name:
                    status = pd.read_sql_query("""
                        SELECT
                            p.player_name AS 選手名,
                            COUNT(h.history_id) AS 履歴件数,
                            SUM(CASE WHEN h.use_for_model=1 THEN 1 ELSE 0 END) AS 予測使用件数,
                            MAX(h.race_date) AS 最新開催日
                        FROM players p
                        LEFT JOIN race_history h ON h.player_id=p.player_id
                        WHERE REPLACE(REPLACE(p.player_name,' ',''),'　','')=?
                        GROUP BY p.player_id, p.player_name
                        ORDER BY p.player_name
                    """, con, params=(query_name,))
                else:
                    status = pd.read_sql_query("""
                        SELECT
                            p.player_name AS 選手名,
                            COUNT(h.history_id) AS 履歴件数,
                            SUM(CASE WHEN h.use_for_model=1 THEN 1 ELSE 0 END) AS 予測使用件数,
                            MAX(h.race_date) AS 最新開催日
                        FROM players p
                        LEFT JOIN race_history h ON h.player_id=p.player_id
                        GROUP BY p.player_id, p.player_name
                        ORDER BY 履歴件数 DESC, p.player_name
                    """, con)

            print(f"DB登録選手: {total_players}人")
            print(f"DB登録履歴: {total_history}件")
            if status.empty:
                print("該当する選手は登録されていません。")
            else:
                display(status)

        except Exception as e:
            print("確認エラー:", e)
            traceback.print_exc(limit=2)

db_check_button.on_click(show_db_status)

db_tab = widgets.VBox([
    widgets.HTML(
        "<b>実際に予測が参照するDBの登録状況を表示します。</b>"
    ),
    db_check_name,
    db_check_button,
    db_check_output
])


# --------------------
# DBリセット
# --------------------
reset_confirm = widgets.Checkbox(
    value=False,
    description="内容を確認し、リセットを実行する"
)
reset_full = widgets.Checkbox(
    value=False,
    description="選手マスターも削除する（完全初期化）"
)
reset_button = widgets.Button(
    description="DBをリセット",
    button_style="danger",
    icon="trash",
    layout=widgets.Layout(width="220px", height="46px")
)
reset_output = widgets.Output()

def reset_database(_):
    with reset_output:
        clear_output()
        try:
            if not reset_confirm.value:
                print("確認チェックを入れてください。")
                return

            db_path = Path(DB_PATH)
            if not db_path.exists():
                print("DBファイルが見つかりません。")
                return

            # リセット前に自動バックアップ
            backup_dir = db_path.parent / "backup"
            backup_dir.mkdir(parents=True, exist_ok=True)
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            backup_path = backup_dir / f"autorace_players_before_reset_{timestamp}.sqlite3"

            import shutil
            shutil.copy2(db_path, backup_path)

            with sqlite3.connect(str(DB_PATH)) as con:
                # 予測・履歴関連を削除
                for table in [
                    "race_history",
                    "v15_player_history_imports",
                    "player_condition_profiles",
                    "condition_learning_history",
                    "prediction_features",
                    "prediction_races",
                    "learning_history",
                ]:
                    exists = con.execute(
                        "SELECT 1 FROM sqlite_master WHERE type='table' AND name=?",
                        (table,)
                    ).fetchone()
                    if exists:
                        con.execute(f"DELETE FROM {table}")

                if reset_full.value:
                    exists = con.execute(
                        "SELECT 1 FROM sqlite_master WHERE type='table' AND name='players'"
                    ).fetchone()
                    if exists:
                        con.execute("DELETE FROM players")

                # AUTOINCREMENT番号も整理
                try:
                    if reset_full.value:
                        con.execute(
                            "DELETE FROM sqlite_sequence WHERE name IN ('race_history','players')"
                        )
                    else:
                        con.execute(
                            "DELETE FROM sqlite_sequence WHERE name='race_history'"
                        )
                except Exception:
                    pass

            print("リセット完了")
            print("バックアップ:", backup_path)
            if reset_full.value:
                print("履歴と選手マスターを削除しました。")
            else:
                print("履歴だけ削除しました。選手マスターは残しています。")

            reset_confirm.value = False
            reset_full.value = False

        except Exception as e:
            print("リセットエラー:", e)
            traceback.print_exc(limit=2)

reset_button.on_click(reset_database)

reset_tab = widgets.VBox([
    widgets.HTML(
        "<b>重複した登録履歴を削除して、登録し直すための画面です。</b><br>"
        "通常は「完全初期化」を選ばず、履歴だけ削除してください。"
    ),
    reset_confirm,
    reset_full,
    reset_button,
    reset_output
])


# --------------------
# 画面
# --------------------
tabs = widgets.Tab(children=[
    prediction_tab, result_tab, web_tab, excel_tab, db_tab, reset_tab
])
tabs.set_title(0, "予測")
tabs.set_title(1, "結果登録")
tabs.set_title(2, "選手履歴登録")
tabs.set_title(3, "Excel登録")
tabs.set_title(4, "DB登録確認")
tabs.set_title(5, "DBリセット")

display(widgets.HTML(
    "<h2>🏁 AutoRaceAI Ver16.3</h2>"
    "<p>実行セルは①と②だけです。操作は下の6タブにまとめています。</p>"
))
display(tabs)
